In [1]:
import h5py
import traceback
import numpy as np
import plotly.graph_objects as go
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
import scipy.signal as sc
from scipy.optimize import curve_fit
from scipy.ndimage import filters 
import tifffile as tiff
from scipy.signal import find_peaks
import plotly.express as px
import os
import csv
import pandas as pd
from roipoly import MultiRoi
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import ast
from AnalasysFunction import plot_spike_shapes_plotly_4states,BurstC,clean_and_parse,motorSp,plotFR,devideTr,plotVolCal,VolToCalIdx,CalSmooth,CorrWindow,ChooseSpk,CalInt,VolToCalIdx,CalAmp,calculate_firing_rate,ChooseCom,LongLIST,SingleSpk,linear_model,quadratic_model,exponential_model,MeanRes,lagOptimaizre
import sys
sys.path.insert(0, r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\code")
from TRY import LongLIST
from NewinternueronsAnalsys import analyze_block
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, linregress,ttest_ind
import pickle
from matplotlib.widgets import Slider, Button

from scipy.ndimage import median_filter
from qixin_spike_detection_4 import sst_spike_correction_gui,detect_bursts_from_vm,complex_bursts_detection, refine_single_spikes,spike_height_calculation2, detect_complex_spikes, refine_all_spikes


In [2]:
import copy

def select_params_interactive_extended(trace, spike_idx, frame_rate, 
                                       init_CS=0.5, init_SS=0.3, 
                                       init_threshold=0.5, init_cb_amp_threshold=0.3,
                                       init_cb_duration_threshold=20,init_threshold_CS=6,init_threshold_SS=6,
                                       inint_ss_cap=0.5, init_f_hp=20, CB_detection_method='simple',
                                       SS_detection_method='simple',CS_SP_TH=0.5, save_html_path=None,n=''):
    """
    Opens an interactive window to tune spike detection parameters.
    Press ENTER when done, or window auto-closes after 30 seconds.
    
    NOTE: ISI threshold is FIXED at 20 ms (10 samples at 500 Hz).
    """
    plt.close('all')
    
    if len(spike_idx) == 0:
        return init_CS, init_SS, init_threshold, init_cb_amp_threshold, init_cb_duration_threshold
    
    selected_params = {
        'pnorm_CS': init_CS,
        'pnorm_SS': init_SS,
        'ST_cs': init_threshold_CS,
        'ST_ss': init_threshold_SS,
        
        'ss_cap': inint_ss_cap,
        'threshold': float(init_threshold),
        'cb_amp_threshold': init_cb_amp_threshold,
        'cb_duration_threshold': init_cb_duration_threshold,
        'f_hp': float(init_f_hp),
        'isi_threshold_ms': 20  # HARDCODED
    }
    final_spikes = {
    "all_spikes_vm": np.array([], dtype=int),
    "complex_spikes_vm": np.array([], dtype=int),
    "simple_spikes_vm": np.array([], dtype=int),
    "complex_burst_vm":np.array([], dtype=int),
    "burst_metrics" : [],
    "vm"    : np.array([], dtype=float),
    "heights": np.array([], dtype=float),
    "SNR": np.array([], dtype=float),
        }

    
    # 65% trace area (top) + 35% controls area (bottom)
    fig = plt.figure(figsize=(15, 10))
    ax = fig.add_axes([0.06, 0.35, 0.92, 0.62])
    ax_vm = ax.twinx()
    ax_vm.set_ylabel('Vm (norm)', color='green')
    ax_vm.yaxis.set_label_position('right')
    ax_vm.yaxis.tick_right()
    ax_vm.spines['right'].set_visible(True)
    ax_vm.spines['right'].set_position(('outward', 6))
    ax_vm.spines['right'].set_color('green')
    ax_vm.spines['right'].set_linewidth(1.2)
    ax_vm.tick_params(axis='y', right=True, labelright=True, colors='green', width=1.0)
    
    t_axis = np.arange(len(trace)) / frame_rate
    trace_line, = ax.plot(t_axis, trace, 'k', lw=0.5, alpha=0.6, label='Trace (norm)')
    vm_line, = ax_vm.plot(t_axis, np.full(len(trace), np.nan, dtype=float), color='green', lw=0.9, alpha=0.85, label='Vm (norm)')
    
    # Scatter plots for spikes
    # Input spike candidates (for sanity-checking index alignment)
    scat_in = ax.scatter([], [], c='0.3', s=12, label='Input spike_idx', zorder=3)
    scat_ss = ax.scatter([], [], c='b', s=30, label='Single Spikes (Qixin)', zorder=5)
    scat_cs = ax.scatter([], [], c='r', s=50, marker='x', label='Complex Spikes (Qixin)', zorder=6)
    scat_vm_simple = ax.scatter([], [], c='cyan', s=20, marker='o', label='Simple Spikes (Vm)', zorder=4)
    scat_vm_complex = ax.scatter([], [], c='magenta', s=40, marker='x', label='Complex Spikes (Vm)', zorder=7)
    ss_cap_line = ax.axhline(np.nan, color='b', ls='--', lw=1, alpha=0.6, label='SS cap (raw)')
    
    count_text = ax.text(0.02, 0.95, "Initializing...", transform=ax.transAxes,
                         bbox=dict(facecolor='white', alpha=0.8), fontsize=10)
    burst_spans = []
    latest_state = {}

    def save_latest_state_to_html():
        if not save_html_path:
            return
        if not isinstance(latest_state, dict) or 'trace_norm' not in latest_state:
            print('No latest detection state available to save.')
            return
        try:
            out_dir = os.path.dirname(save_html_path)
            if out_dir:
                os.makedirs(out_dir, exist_ok=True)

            trace_norm = np.asarray(latest_state['trace_norm'], dtype=float).ravel()
            t = np.arange(len(trace_norm)) / frame_rate

            vm_norm = latest_state.get('vm_norm', None)
            if vm_norm is not None:
                vm_norm = np.asarray(vm_norm, dtype=float).ravel()

            fig_html = go.Figure()
            fig_html.add_trace(go.Scattergl(x=t, y=trace_norm, mode='lines', name='Trace (norm)', line=dict(color='black', width=1), opacity=0.6))

            if vm_norm is not None and vm_norm.size > 0:
                t_vm = np.arange(vm_norm.size) / frame_rate
                fig_html.add_trace(go.Scattergl(x=t_vm, y=vm_norm, mode='lines', name='Vm (norm)', line=dict(color='green', width=1), opacity=0.9))

            def _add_points(name, idx, color, symbol, size):
                if idx is None:
                    return
                idx = np.asarray(idx, dtype=int)
                idx = idx[(idx >= 0) & (idx < len(trace_norm))]
                if idx.size == 0:
                    return
                fig_html.add_trace(go.Scattergl(x=t[idx], y=trace_norm[idx], mode='markers', name=name, marker=dict(color=color, symbol=symbol, size=size)))

            _add_points('Input spike_idx', latest_state.get('idx_in'), 'rgba(80,80,80,0.8)', 'circle', 6)
            _add_points('Single Spikes (Qixin)', latest_state.get('idx_ss'), 'blue', 'circle', 8)
            _add_points('Complex Spikes (Qixin)', latest_state.get('idx_cs'), 'red', 'x', 10)
            _add_points('Simple Spikes (Vm)', latest_state.get('idx_vm_simple'), 'cyan', 'circle-open', 8)
            _add_points('Complex Spikes (Vm)', latest_state.get('idx_vm_complex'), 'magenta', 'x', 10)

            starts = latest_state.get('burst_starts', None)
            ends = latest_state.get('burst_ends', None)
            if starts is not None and ends is not None:
                starts = np.asarray(starts, dtype=float).ravel()
                ends = np.asarray(ends, dtype=float).ravel()
                for s, e in zip(starts, ends):
                    try:
                        fig_html.add_vrect(x0=float(s)/frame_rate, x1=float(e)/frame_rate, fillcolor='yellow', opacity=0.2, line_width=0, layer='below')
                    except Exception:
                        fig_html.add_shape(type='rect', x0=float(s)/frame_rate, x1=float(e)/frame_rate, y0=0, y1=1, yref='paper', fillcolor='yellow', opacity=0.2, line_width=0, layer='below')

            p_ss_cap = latest_state.get('p_ss_cap', None)
            if p_ss_cap is not None and np.isfinite(p_ss_cap):
                try:
                    fig_html.add_hline(y=float(p_ss_cap), line_dash='dash', line_color='blue', opacity=0.5)
                except Exception:
                    fig_html.add_shape(type='line', x0=float(t[0]), x1=float(t[-1]), y0=float(p_ss_cap), y1=float(p_ss_cap), line=dict(color='blue', dash='dash', width=1), opacity=0.5)

            latest_candidates = []
            for label, key in [('Complex Spikes (Vm)', 'idx_vm_complex'), ('Complex Spikes (Qixin)', 'idx_cs'), ('Simple Spikes (Vm)', 'idx_vm_simple'), ('Single Spikes (Qixin)', 'idx_ss')]:
                arr = latest_state.get(key, None)
                if arr is None:
                    continue
                arr = np.asarray(arr, dtype=int)
                arr = arr[(arr >= 0) & (arr < len(trace_norm))]
                if arr.size:
                    latest_candidates.append((int(arr.max()), label))
            if latest_candidates:
                latest_idx, latest_label = max(latest_candidates, key=lambda x: x[0])
                fig_html.add_trace(go.Scattergl(x=[t[latest_idx]], y=[trace_norm[latest_idx]], mode='markers+text', name='Latest spike', marker=dict(color='gold', size=14, symbol='star'), text=[f'Latest: {latest_label}'], textposition='top center'))

            params = latest_state.get('params', {}) if isinstance(latest_state.get('params', {}), dict) else {}
            title = 'Latest spike detected & classified (interactive)' + f" | CS={params.get('pnorm_CS', np.nan):.2f} SS={params.get('pnorm_SS', np.nan):.2f} T={params.get('threshold', np.nan):.2f}"
            fig_html.update_layout(title=title, xaxis_title='Time (s)', yaxis_title='Trace (norm)', template='plotly_white', height=650, legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0))
            fig_html.write_html(save_html_path, include_plotlyjs=True, full_html=True)
            print(f'Saved HTML figure: {save_html_path}')
        except Exception as e:
            print(f'Failed to save HTML figure: {e}')


    ax.set_title(f"Input Spikes: {len(spike_idx)} | Adjust Sliders & Press ENTER (or wait 30s)")
    ax.set_ylabel('Trace (norm)')
    ax.legend(loc='upper right', fontsize=8)
    
    # Create sliders for CS/SS detection (35% controls area)
    ax_cs = plt.axes([0.12, 0.315, 0.76, 0.022])
    ax_ss = plt.axes([0.12, 0.284, 0.76, 0.022])
    ax_th = plt.axes([0.12, 0.253, 0.76, 0.022])
    ax_cb_amp = plt.axes([0.12, 0.222, 0.76, 0.022])
    ax_cb_dur = plt.axes([0.12, 0.191, 0.76, 0.022])
    ax_cb_st = plt.axes([0.12, 0.160, 0.76, 0.022])
    ax_ss_st = plt.axes([0.12, 0.129, 0.76, 0.022])
    ax_cb_cap = plt.axes([0.12, 0.098, 0.76, 0.022])
    ax_fhp = plt.axes([0.12, 0.067, 0.76, 0.022])

    slider_cs = Slider(ax_cs, 'P-Norm CS', 0.0, 4, valinit=init_CS)
    slider_ss = Slider(ax_ss, 'P-Norm SS', 0.0, 2, valinit=init_SS)
    slider_th = Slider(ax_th, 'CS Height Thresh', 0.0, 4.5, valinit=init_threshold)
    slider_cb_amp = Slider(ax_cb_amp, 'CB Amp Thresh', 0.0, 4.5, valinit=init_cb_amp_threshold, valstep=0.05)
    slider_cb_dur = Slider(ax_cb_dur, 'CB Duration (ms)', 5, 300, valinit=init_cb_duration_threshold, valstep=5)
    slider_cs_st = Slider(ax_cb_st, 'simple threshol CS', 0.0, 10, valinit=init_threshold_CS)
    slider_ss_st = Slider(ax_ss_st, 'simple threshol SS', 0.0, 10, valinit= init_threshold_SS)
    slider_ss_cap = Slider(ax_cb_cap, 'SS min height (norm)', 0.0, 4, valinit= inint_ss_cap, valstep=0.05)
    slider_fhp = Slider(ax_fhp, 'HP Freq (Hz)', 5.0, 40.0, valinit=float(init_f_hp), valstep=1.0)
    # Add text label for ISI threshold (not adjustable)
    fig.text(0.12, 0.03, 'ISI Threshold (FIXED): 20 ms', fontsize=9, weight='bold', 
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    def run_detection(p_cs, p_ss, p_th, p_cb_amp, p_cb_dur, p_ss_cap, p_st_ss, p_st_cs, p_f_hp, CB_detection_method,
                      SS_detection_method):
        """Run full spike detection pipeline with Vm burst detection"""
        try:
            # Always apply simple thresholds during tuning
            CB_detection_method = "simple"
            SS_detection_method = "simple"

            # Step 1: Complex burst detection
            c_bursts, _ = complex_bursts_detection(
                trace, spike_idx, frame_rate,
                pnorm=p_cs, process_window=30, plotflag=False,CB_detection_method=CB_detection_method,
                simple_threshold=p_st_cs
            )
            
            # Step 2: Refine single spikes
            r_ss, t_noCS = refine_single_spikes(
                trace, spike_idx, c_bursts, frame_rate,
                process_window=30, pnorm=p_ss, f_hp=p_f_hp, min_spikes=5, plotflag=False
                ,SS_detection_method=SS_detection_method,simple_threshold_SS=p_st_ss,SS_height_cap=None
            )
            
            # Step 3: Calculate spike heights
            trace_mf = c_bursts.get('trace_mf', median_filter(trace, size=11))
            heights, snr = spike_height_calculation2(
                r_ss, trace, trace_mf, t_noCS, frame_rate
            )
            #height_scale = float(np.nanmedian(heights)) if np.size(heights) else 1.0
            #height_scale = max(height_scale, 1e-12)
            height_scale = float(np.nanmedian(heights)) if np.size(heights) else 1.0
            height_scale = max(height_scale, 1e-12)
            trace_norm = trace / height_scale
            # Apply SS cap on normalized trace (0..1 slider makes sense here)
            if p_ss_cap is not None:
                idx_ss = r_ss.astype(int)
                idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
                r_ss = idx_ss[trace_norm[idx_ss] >= float(p_ss_cap)]
            ss_cap_raw = float(p_ss_cap) * height_scale
            
            # Step 4: Detect complex spikes
            cs_spikes_raw = detect_complex_spikes(
                trace_norm, c_bursts, np.ones_like(trace_norm), threshold=p_th, plotflag=False
            )
            
            # Step 5: Refine all spikes
            c_bursts_final, r_ss_final, all_cs, all_s = refine_all_spikes(
                c_bursts, cs_spikes_raw, r_ss
            )
            
            # Step 6: Detect bursts from Vm (with ISI = 20 ms hardcoded)
            ret = detect_bursts_from_vm(
                trace_norm,
                np.ones_like(trace_norm),
                c_bursts_final,
                all_s,
                frame_rate,
                highpass=1.0,
                median_window=11,
                cb_amp_threshold=p_cb_amp,
                cb_duration_threshold=p_cb_dur,
                isi_threshold_ms=20,
                baseline_subtract=True,
                baseline_window_ms=20,
                baseline_percentile=10,
                merge_SS_ms = 20,
                merge_CB_ms = 5,
            )
            if not isinstance(ret, tuple) or len(ret) < 7:
                raise RuntimeError(f"detect_bursts_from_vm return mismatch: {type(ret)}, len={len(ret) if isinstance(ret, tuple) else 'NA'}")
            simple_spikes_vm, complex_spikes_vm, all_spikes_vm = ret[0], ret[1], ret[2]
            trace_snr, vm, burst_metrics, complex_bursts_vm = ret[3], ret[4], ret[5], ret[6]
            
            return c_bursts_final, r_ss_final, all_cs, complex_bursts_vm, simple_spikes_vm, complex_spikes_vm, ss_cap_raw, trace_norm,all_spikes_vm,burst_metrics,vm,heights,trace_snr
        
        except Exception as e:
            print(f"Error in run_detection: {e}")
            import traceback
            traceback.print_exc()
            trace_norm = np.asarray(trace, dtype=float)
            return {}, np.array([], dtype=int), np.array([], dtype=int), {}, np.array([], dtype=int), np.array([], dtype=int), float("nan"), trace_norm, np.array([], dtype=int), [], np.array([], dtype=float), np.array([], dtype=float), np.array([], dtype=float)

    def update(val):
        p_cs = slider_cs.val
        p_ss = slider_ss.val
        p_th = float(slider_th.val)
        p_cb_amp = slider_cb_amp.val
        p_cb_dur = slider_cb_dur.val
        p_ss_cap = slider_ss_cap.val
        p_st_ss = slider_ss_st.val
        p_st_cs = slider_cs_st.val
        p_f_hp = slider_fhp.val

                
        
        selected_params.update({
            'pnorm_CS': p_cs,
            'pnorm_SS': p_ss,
            'threshold': p_th,
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'ST_cs': p_st_cs,
            'ST_ss': p_st_ss,
            'ss_cap': p_ss_cap,
            'f_hp': p_f_hp
        })
        
        try:
            c_bursts, r_ss, all_cs, c_bursts_vm, simple_spikes_vm, complex_spikes_vm, ss_cap_raw, trace_norm, all_spikes_vm,burst_metrics,vm,heights,SNR = run_detection(
                p_cs, p_ss, p_th, p_cb_amp, p_cb_dur, p_ss_cap, p_st_ss, p_st_cs, p_f_hp, CB_detection_method,
                SS_detection_method
            )
            final_spikes["all_spikes_vm"] = np.asarray(all_spikes_vm, dtype=int).copy()
            final_spikes["complex_spikes_vm"] = np.asarray(complex_spikes_vm, dtype=int).copy()
            final_spikes["simple_spikes_vm"] = np.asarray(simple_spikes_vm, dtype=int).copy()
            final_spikes["complex_burst_vm"] = c_bursts_vm
            final_spikes["vm"] = np.asarray(vm, dtype=float).copy()
            final_spikes["burst_metrics"] = burst_metrics.copy()
            final_spikes["heights"] = np.asarray(heights, dtype=float).copy()
            final_spikes["SNR"] = np.asarray(SNR, dtype=float).copy()

            trace_norm = np.asarray(trace_norm, dtype=float).ravel()
            vm_norm = np.asarray(vm, dtype=float).ravel()
            if vm_norm.size != len(trace_norm):
                vm_aligned = np.full(len(trace_norm), np.nan, dtype=float)
                n_copy = min(len(trace_norm), vm_norm.size)
                if n_copy > 0:
                    vm_aligned[:n_copy] = vm_norm[:n_copy]
                vm_norm = vm_aligned
            idx_in = np.asarray(spike_idx, dtype=int)
            idx_in = idx_in[(idx_in >= 0) & (idx_in < len(trace_norm))]
            if idx_in.size > 0:
                scat_in.set_offsets(np.c_[t_axis[idx_in], trace_norm[idx_in]])
                scat_in.set_visible(True)
            else:
                scat_in.set_visible(False)
            
            idx_ss = np.array([], dtype=int)
            idx_cs = np.array([], dtype=int)
            idx_vm_simple = np.array([], dtype=int)
            idx_vm_complex = np.array([], dtype=int)
            starts = np.array([], dtype=int)
            ends = np.array([], dtype=int)

            # --- QIXIN SPIKE COUNTS ---
            n_ss = len(r_ss) if isinstance(r_ss, np.ndarray) else 0
            n_cs = len(all_cs) if isinstance(all_cs, np.ndarray) else 0
            
            # --- VM BURST SPIKE COUNTS ---
            n_vm_simple = len(simple_spikes_vm) if isinstance(simple_spikes_vm, np.ndarray) else 0
            n_vm_complex = len(complex_spikes_vm) if isinstance(complex_spikes_vm, np.ndarray) else 0
            n_bursts_vm = len(c_bursts_vm.get('starts', [])) if isinstance(c_bursts_vm, dict) else 0
            
            count_text.set_text(
                f"QIXIN: SS={n_ss} | CS={n_cs}\n"
                f"VM-BURST: Simple={n_vm_simple} | Complex={n_vm_complex} | Bursts={n_bursts_vm}\n"
                f"CS={p_cs:.2f} | SS={p_ss:.2f} | T={p_th:.2f} | HP={p_f_hp:.1f}Hz\n"
                f"CB_Amp={p_cb_amp:.2f} | CB_Dur={p_cb_dur:.0f}ms | ISI=20ms (fixed)\n"
                + (f"SS_cap(norm)={p_ss_cap:.2f} ~ raw {ss_cap_raw:.3g}" if ss_cap_raw is not None else f"SS_cap(norm)={p_ss_cap:.2f}")
            )

            # Update displayed trace to normalized trace
            trace_line.set_ydata(trace_norm)
            vm_line.set_ydata(vm_norm)
            ax.relim()
            ax.autoscale_view(scalex=False, scaley=True)
            ax_vm.relim()
            ax_vm.autoscale_view(scalex=False, scaley=True)

            # Update SS cap threshold line (normalized units)
            if ss_cap_raw is not None and np.isfinite(ss_cap_raw):
                ss_cap_line.set_ydata([float(p_ss_cap), float(p_ss_cap)])
                ss_cap_line.set_visible(True)
            else:
                ss_cap_line.set_visible(False)

            # --- PLOT QIXIN SINGLE SPIKES (BLUE) ---
            if n_ss > 0:
                idx_ss = r_ss.astype(int)
                idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
                if len(idx_ss) > 0:
                    scat_ss.set_offsets(np.c_[t_axis[idx_ss], trace_norm[idx_ss]])
                    scat_ss.set_visible(True)
                else:
                    scat_ss.set_visible(False)
            else:
                scat_ss.set_visible(False)

            # --- PLOT QIXIN COMPLEX SPIKES (RED X) ---
            if n_cs > 0:
                idx_cs = all_cs.astype(int)
                idx_cs = idx_cs[(idx_cs >= 0) & (idx_cs < len(trace_norm))]
                if len(idx_cs) > 0:
                    scat_cs.set_offsets(np.c_[t_axis[idx_cs], trace_norm[idx_cs]])
                    scat_cs.set_visible(True)
                else:
                    scat_cs.set_visible(False)
            else:
                scat_cs.set_visible(False)
            
            # --- PLOT VM SIMPLE SPIKES (CYAN CIRCLES) ---
            if n_vm_simple > 0:
                idx_vm_simple = simple_spikes_vm.astype(int)
                idx_vm_simple = idx_vm_simple[(idx_vm_simple >= 0) & (idx_vm_simple < len(trace_norm))]
                if len(idx_vm_simple) > 0:
                    scat_vm_simple.set_offsets(np.c_[t_axis[idx_vm_simple], trace_norm[idx_vm_simple]])
                    scat_vm_simple.set_visible(True)
                else:
                    scat_vm_simple.set_visible(False)
            else:
                scat_vm_simple.set_visible(False)
            
            # --- PLOT VM COMPLEX SPIKES (MAGENTA X) ---
            if n_vm_complex > 0:
                idx_vm_complex = complex_spikes_vm.astype(int)
                idx_vm_complex = idx_vm_complex[(idx_vm_complex >= 0) & (idx_vm_complex < len(trace_norm))]
                if len(idx_vm_complex) > 0:
                    scat_vm_complex.set_offsets(np.c_[t_axis[idx_vm_complex], trace_norm[idx_vm_complex]])
                    scat_vm_complex.set_visible(True)
                else:
                    scat_vm_complex.set_visible(False)
            else:
                scat_vm_complex.set_visible(False)
            
            # --- PLOT VM BURST SPANS (YELLOW BANDS) ---
            for span in burst_spans:
                span.remove()
            burst_spans.clear()
            
            if isinstance(c_bursts_vm, dict):
                starts = c_bursts_vm.get('starts', np.array([]))
                ends = c_bursts_vm.get('ends', np.array([]))
                for s, e in zip(starts, ends):
                    span = ax.axvspan(s/frame_rate, e/frame_rate, color='yellow', alpha=0.3)
                    burst_spans.append(span)
            
            latest_state.clear()
            latest_state.update({
                'trace_norm': trace_norm,
                'vm_norm': vm_norm,
                'idx_in': idx_in,
                'idx_ss': idx_ss,
                'idx_cs': idx_cs,
                'idx_vm_simple': idx_vm_simple,
                'idx_vm_complex': idx_vm_complex,
                'burst_starts': starts,
                'burst_ends': ends,
                'p_ss_cap': float(p_ss_cap) if p_ss_cap is not None else None,
                'params': selected_params.copy(),
            })

            fig.canvas.draw_idle()
        except Exception as e:
            count_text.set_text(f"ERROR: {str(e)[:80]}")
            fig.canvas.draw_idle()

    slider_cs.on_changed(update)
    slider_ss.on_changed(update)
    slider_th.on_changed(update)
    slider_cb_amp.on_changed(update)
    slider_cb_dur.on_changed(update)
    slider_ss_cap.on_changed(update)
    slider_ss_st.on_changed(update)
    slider_cs_st.on_changed(update) 
    slider_fhp.on_changed(update)
    
    update(None)
    
    def on_key(event):
        """Capture ENTER key press"""
        if event.key == 'enter':
            plt.close(fig)

    fig.canvas.mpl_connect('key_press_event', on_key)
    
    # --- SAFE WAIT LOOP (non-blocking with timeout) ---
    plt.show(block=False)
    
    for _ in range(40000):  # 30 seconds max
        if not plt.fignum_exists(fig.number):
            break
        plt.pause(0.015)
    else:
        print("  ⚠ Interactive timeout – using current parameter values")
    
    plt.close('all')
    save_latest_state_to_html()

    return (
        selected_params['pnorm_CS'],
        selected_params['pnorm_SS'],
        selected_params['threshold'],
        selected_params['cb_amp_threshold'],
        selected_params['cb_duration_threshold'],
        selected_params['ss_cap'],
        selected_params['ST_ss'],
        selected_params['ST_cs'],
        final_spikes["all_spikes_vm"],
        final_spikes["complex_spikes_vm"],
        final_spikes["simple_spikes_vm"],
        final_spikes["complex_burst_vm"],
        final_spikes["vm"],
        final_spikes["burst_metrics"],
        final_spikes["heights"],
        final_spikes["SNR"]    
    )




In [3]:
def get_Vol_Cal_SS(ss_list, Vol_trace, Cal_trace, volAx, calAx, preW=100, postW=500, save_path=None):
    """
    Extract spike-aligned voltage and calcium traces.
    Handles edge spikes with variable window lengths.
    Returns actual lengths for proper time-axis alignment per spike.
    """
    calTl = []
    volTl = []
    calIdx = []
    actual_lens = []  # Track actual length of each trace
    
    for i, l in enumerate(ss_list):
        l = int(l)
        
        # Clamp to valid range (allows variable window)
        volStart = max(0, l - preW)
        volEnd = min(len(Vol_trace) - 1, l + postW)
        
        # Calculate actual windows used (may be shorter at edges)
        actual_pre = l - volStart  # Actual pre-spike window
        actual_post = volEnd - l   # Actual post-spike window
        actual_total = volEnd - volStart + 1
        
        # Convert to calcium indices
        calS = VolToCalIdx(volStart, volAx, calAx)
        calE = VolToCalIdx(volEnd, volAx, calAx)
        calE = min(calE, len(Cal_trace) - 1)
        
        # Extract traces WITHOUT padding
        vol_segment = Vol_trace[volStart:volEnd + 1]
        cal_segment = Cal_trace[calS:calE + 1]
        
        volTl.append(vol_segment)
        calTl.append(cal_segment)
        calIdx.append(VolToCalIdx(l, volAx, calAx))
        actual_lens.append({
            'idx': i,
            'vol_len': len(vol_segment),
            'cal_len': len(cal_segment),
            'pre': actual_pre,
            'post': actual_post
        })
    
    if save_path is not None:
        with open(save_path, "w") as f:
            f.write(f"preW = {preW}\n")
            f.write(f"postW = {postW}\n")
            f.write(f"actual_lengths = {actual_lens}\n")
    
    return calTl, volTl, calIdx, actual_lens  # ← Return actual lengths!

def get_Vol_Cal_Burst(Burst_list,Vol_trae,Cal_trace,volAx,calAx, preW = 100, postW = 500,save_path = None):
    calTl = []
    volTl = []
    for i,l in enumerate(Burst_list):
        volStart = l[0] - preW
        volend = l[-1]+ postW
        calS = VolToCalIdx(volStart,volAx,calAx)
        calE = VolToCalIdx(volend,volAx,calAx)
        volTl.append(Vol_trae[volStart:volend])
        calTl.append(Cal_trace[calS:calE])
        if save_path is not None:
            with open(save_path, "w") as f:
                f.write(f"preW = {preW}\n")
                f.write(f"postW = {postW}\n")
    return calTl,volTl
def flatten_traces(traces):
    """Flattens nested lists into a simple list of 1D numpy arrays."""
    out = []
    for x in traces:
        if x is None:
            continue
        if isinstance(x, (list, tuple)):
            out.extend(flatten_traces(x))
        else:
            arr = np.asarray(x).ravel()
            if arr.size > 0:
                out.append(arr.astype(float))
    return out

def auc_window(v, t_ms, win, baseline_win=(-50,-20), positive_only=True):
    v = np.asarray(v, float)
    t = np.asarray(t_ms, float)

    bi = np.where((t >= baseline_win[0]) & (t <= baseline_win[1]))[0]
    wi = np.where((t >= win[0]) & (t <= win[1]))[0]
    if bi.size < 3 or wi.size < 3:
        return np.nan

    baseline = np.nanmedian(v[bi])
    y = v - baseline
    y = y[wi]
    if positive_only:
        y = np.maximum(y, 0)
    return np.trapezoid(y, t[wi])
def remove_spike_linear_interp(v, t_ms, peak_search_win=(-5, 5), cut_win=(-2, 6)):
    v = np.asarray(v, dtype=float).copy()
    t_ms = np.asarray(t_ms, dtype=float)

    # --- tolerate mismatch by truncating to the shorter length ---
    L = min(len(v), len(t_ms))
    v = v[:L]
    t = t_ms[:L]

    # ---- find spike peak near 0 ms ----
    ps_idx = np.where((t >= peak_search_win[0]) & (t <= peak_search_win[1]))[0]
    if ps_idx.size == 0:
        pk = int(np.nanargmax(v))
    else:
        pk = ps_idx[int(np.nanargmax(v[ps_idx]))]

    t_pk = t[pk]

    # ---- cut region around peak ----
    cut_start = t_pk + cut_win[0]
    cut_end   = t_pk + cut_win[1]
    cut_idx = np.where((t >= cut_start) & (t <= cut_end))[0]
    if cut_idx.size == 0:
        return v

    i0, i1 = int(cut_idx[0]), int(cut_idx[-1])

    # ---- edge safety ----
    if i0 <= 0 or i1 >= (L - 1):
        v[i0:i1+1] = np.nan
        return v

    # ---- linear interpolation ----
    y0 = v[i0 - 1]
    y1 = v[i1 + 1]
    v[i0:i1+1] = np.linspace(y0, y1, i1 - i0 + 1)

    return v
def classify_subthreshold_with_spike_removed_samewin(
    V_traces, t_ms,
    signal_win=(-20, 100),
    noise_win=(-140, -20),
    cut_win=(-2, 4),
    k=3.0,
    positive_only=True):
    """
    Classify spikes based on AUC, handling variable-length traces.
    """
    V_list = flatten_traces(V_traces)
    
    if len(V_list) == 0:
        return np.array([], dtype=bool), np.array([]), np.array([]), np.nan
    
    # Get common length for all traces
    common_len = max(len(v) for v in V_list)
    t_ms = np.asarray(t_ms, dtype=float)
    
    # If t_ms is shorter than traces, pad it
    if len(t_ms) < common_len:
        dt = t_ms[1] - t_ms[0] if len(t_ms) > 1 else 1.0
        t_ms_extended = np.arange(common_len) * dt + t_ms[0]
        t_ms = t_ms_extended
    
    # Truncate to common length and align
    auc_sig = []
    auc_nse = []
    
    for v in V_list:
        v_padded = np.asarray(v, dtype=float)
        
        # Pad if shorter than common length
        if len(v_padded) < common_len:
            v_padded = np.pad(v_padded, (0, common_len - len(v_padded)), 
                             mode='constant', constant_values=np.nan)
        else:
            v_padded = v_padded[:common_len]
        
        # Clean spike
        v_clean = remove_spike_linear_interp(v_padded, t_ms[:common_len], cut_win=cut_win)
        
        # Calculate AUC
        sig_auc = auc_window(v_clean, t_ms[:common_len], signal_win, 
                            baseline_win=(-50, -20), positive_only=positive_only)
        nse_auc = auc_window(v_clean, t_ms[:common_len], noise_win, 
                            baseline_win=(-50, -20), positive_only=positive_only)
        
        auc_sig.append(sig_auc)
        auc_nse.append(nse_auc)
    
    auc_sig = np.array(auc_sig, dtype=float)
    auc_nse = np.array(auc_nse, dtype=float)
    
    # Threshold from noise distribution
    mu = np.nanmean(auc_nse)
    sd = np.nanstd(auc_nse)
    thr = mu + k * sd
    
    mask = auc_sig > thr
    
    return mask, auc_sig, auc_nse, thr


In [4]:
# def load_and_refine_pyr_spikes(
#     path,trace_vol,trace_cal,
#     frame_rate=500,
#     default_pn_CS=0.5,
#     default_pn_SS=0.3,
#     default_thresh=0.5,
#     default_cb_amp_threshold=0.3,
#     default_cb_duration_threshold=20,
#     CB_detection_method = 'simple', # 'simple' or 'volpy_based'
#     SS_detection_method = 'simple',
#     simple_threshold_cs = 6,
#     simple_threshold_SS = 6,
#     ss_height_cap = 0.5,
#     f_hp = 20,
#     preW=75,
#     postW=100,
#     burst_window_ms=15,
#     save_dir=None,
#     interactive=True,
#     spikeID = None,params_pr = None,
#     save_only=False,
#     final_spikes=None,
#     selected_params=None,
#     name =''

# ):
#     """
#     Complete pipeline for pyramidal cell spike refinement using qixin_spike_detection:
#     1. Load voltage & calcium traces
#     2. Load or detect initial spikes
#     3. Run interactive spike detection (if interactive=True)
#     4. Refine spikes using qixin functions
#     5. Detect bursts from Vm (FINAL spike detection)
#     6. Save refined spikes as PKL
#     7. Extract spike-aligned voltage & calcium traces
#     8. Classify spikes (single vs burst vs complex)
#     9. Choose sparse single spikes (no ADP)
    
#     Parameters
#     ----------
#     path : str
#         Path to cell directory (contains calTraceDF.csv, volTraceDF.csv, etc.)
#     frame_rate : int
#         Sampling rate in Hz (default 500 for voltage)
#     default_pn_CS : float
#         Initial complex spike p-norm parameter
#     default_pn_SS : float
#         Initial single spike p-norm parameter
#     default_thresh : float
#         Initial CS detection threshold
#     default_cb_amp_threshold : float
#         Initial complex burst amplitude threshold (default 0.3)
#     default_cb_duration_threshold : float
#         Initial complex burst duration threshold (default 20 ms)
#     preW, postW : int
#         Pre/post window samples for spike-aligned extraction
#     burst_window_ms : int
#         Window for burst detection (ms)
#     save_dir : str or None
#         Directory to save outputs (defaults to path)
#     interactive : bool
#         If True, open interactive tuning window; else use defaults
    
#     Returns
#     -------
#     results_dict : dict with comprehensive spike analysis results using Vm burst detection
#     """
    
#     if save_dir is None:
#         save_dir = path
#     os.makedirs(save_dir, exist_ok=True)

#     # ===== SAVE-ONLY: WRITE PICKLE WITHOUT RE-RUNNING =====
#     # Does NOT load anything from disk. Keeps the same saved pickle structure.
#     # Provide `final_spikes` (outputs of select_params_interactive_extended OR selected from your loaded cell__CS_detection.pkl).
#     if save_only:
#         if final_spikes is None or not isinstance(final_spikes, dict):
#             raise ValueError('save_only=True requires final_spikes dict')

#         p = selected_params if isinstance(selected_params, dict) else (params_pr if isinstance(params_pr, dict) else {})
#         p_CS = p.get('pnorm_CS', default_pn_CS)
#         p_SS = p.get('pnorm_SS', default_pn_SS)
#         p_Th = p.get('threshold', default_thresh)
#         p_cb_amp = p.get('cb_amp_threshold', default_cb_amp_threshold)
#         p_cb_dur = p.get('cb_duration_threshold', default_cb_duration_threshold)
#         p_ss_cap = p.get('ss_cap', ss_height_cap)
#         p_ST_ss = p.get('simple_threshold_ss', simple_threshold_SS)
#         p_ST_cs = p.get('simple_threshold_cs', simple_threshold_cs)

#         # Accept either vm_* keys (new) or non-vm keys (older/select_params output).
#         all_spikes = final_spikes.get('vm_all_spikes', final_spikes.get('all_spikes', []))
#         complex_spikes = final_spikes.get('vm_complex_spikes', final_spikes.get('complex_spikes', []))
#         simple_spikes = final_spikes.get('vm_simple_spikes', final_spikes.get('simple_spikes', []))
#         complex_bursts = final_spikes.get('vm_burst_dict', final_spikes.get('complex_bursts', final_spikes.get('complex_bursts_dict', {})))
#         burst_metrics = final_spikes.get('burst_metrics', final_spikes.get('vm_burst_metrics', []))
#         heights = final_spikes.get('heights', final_spikes.get('spike_heights', final_spikes.get('spike_heights_interpolated', [])))
#         snr = final_spikes.get('snr', final_spikes.get('SNR', final_spikes.get('spike_SNR', final_spikes.get('SNR_interpolated', []))))
#         input_spikes = final_spikes.get('input_spikes', spikeID if spikeID is not None else [])

#         save_data = {
#             'trace_vol': trace_vol,
#             'trace_cal': trace_cal,
#             'input_spikes': np.asarray(input_spikes, dtype=int),
#             'vm_burst_dict': complex_bursts if isinstance(complex_bursts, dict) else {},
#             'vm_simple_spikes': np.asarray(simple_spikes, dtype=int),
#             'vm_complex_spikes': np.asarray(complex_spikes, dtype=int),
#             'vm_all_spikes': np.asarray(all_spikes, dtype=int),
#             'vm_burst_metrics': burst_metrics if burst_metrics is not None else [],
#             'spike_heights': np.asarray(heights, dtype=float) if heights is not None else np.array([], dtype=float),
#             'spike_SNR': np.asarray(snr, dtype=float) if snr is not None else np.array([], dtype=float),
#             'detection_params': {
#                 'pnorm_CS': p_CS,
#                 'pnorm_SS': p_SS,
#                 'threshold': p_Th,
#                 'cb_amp_threshold': p_cb_amp,
#                 'cb_duration_threshold': p_cb_dur,
#                 'ss_cap': p_ss_cap,
#                 'simple_threshold_ss': p_ST_ss,
#                 'simple_threshold_cs': p_ST_cs,
#                 'isi_threshold_ms': 20,
#                 'frame_rate': frame_rate,
#                 'preW': preW,
#                 'postW': postW,
#                 'detection_method': p.get('detection_method', 'Vm_burst_detection')
#             }
#         }

#         pkl_path = os.path.join(save_dir, f'spike_detection_refined_new{name}.pkl')
#         with open(pkl_path, 'wb') as f:
#             pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
#         return save_data


#     # ===== 2. LOAD OR DETECT INITIAL SPIKES =====
#     spike_pkl_path = os.path.join(path, 'cell__CS_detection.pkl')
#     initial_spikes = spikeID
#     prev_params = params_pr
    
#     if initial_spikes is None or len(initial_spikes) == 0:
#         threshold = np.percentile(np.abs(np.diff(trace_vol)), 95)
#         initial_spikes = np.where(np.abs(np.diff(trace_vol)) > threshold)[0].tolist()
    
#     initial_spikes = np.asarray(initial_spikes, dtype=int)
#     init_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
#     init_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
#     init_thresh = prev_params['threshold'] if prev_params else default_thresh
#     init_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
#     init_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
#     init_simple_thresh_CS = prev_params.get('simple_threshold_cs', simple_threshold_cs) if prev_params else simple_threshold_cs
#     init_simple_thresh_SS =  prev_params.get('simple_threshold_ss', simple_threshold_SS)if prev_params else simple_threshold_SS
#     init_ss_cap = prev_params.get('ss_cap', ss_height_cap)if prev_params else ss_height_cap
   

#     #2.5 FIRS ROUND RUN
#     complex_bursts_dict, segment_bounds = complex_bursts_detection(
#                             trace_vol, initial_spikes, frame_rate,
#                             pnorm=init_CS,
#                             process_window=30,
#                             plotflag=False,
#                             CB_detection_method=CB_detection_method,
#                             simple_threshold=init_simple_thresh_CS
#                         )

#     refined_SS, trace_noCS = refine_single_spikes(trace_vol, initial_spikes, complex_bursts_dict, frame_rate, process_window=30, pnorm=init_CS, f_hp=f_hp, min_spikes=5, plotflag=False,
#                                                    SS_detection_method=SS_detection_method, simple_threshold_SS=init_simple_thresh_SS )
#     initial_spikes= refined_SS
#     spike_heights_interpolated, SNR_interpolated = spike_height_calculation2(refined_SS, trace_vol, complex_bursts_dict['trace_mf'], trace_noCS, frame_rate, plotflag=False )

#     # ===== 3. INTERACTIVE REFINEMENT =====
#     if interactive and len(initial_spikes) > 0:
#         init_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
#         init_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
#         init_thresh = prev_params['threshold'] if prev_params else default_thresh
#         init_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
#         init_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
        
#         p_CS, p_SS, p_Th, p_cb_amp, p_cb_dur,p_ss_cap,p_ST_ss,p_ST_cs,all_spikes,complex_spikes,simple_spikes,complex_bursts,vm,burst_metrics,heights,snr = select_params_interactive_extended(
#             trace= trace_vol,spike_idx= initial_spikes, frame_rate=frame_rate,
#             init_CS=init_CS,
#             init_SS=init_SS,
#             init_threshold=init_thresh,
#             init_cb_amp_threshold=init_cb_amp,
#             init_cb_duration_threshold=init_cb_dur,
#             init_threshold_CS=init_simple_thresh_CS,
#             init_threshold_SS=init_simple_thresh_SS,
#             inint_ss_cap=init_ss_cap,
#             CB_detection_method=CB_detection_method,
#             SS_detection_method=SS_detection_method, 
#             save_html_path=os.path.join(save_dir, f'latest_spike_detected_and_classified{name}.html')
#         )
#     # else:
#     #     p_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
#     #     p_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
#     #     p_Th = prev_params['threshold'] if prev_params else default_thresh
#     #     p_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
#     #     p_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
    
#     # # # ===== 4. REFINE SPIKES USING QIXIN FUNCTIONS =====
#     # # try:
#     # #     # Step 1: Detect complex bursts
#     # #     complex_bursts_dict, _ = complex_bursts_detection(
#     # #         trace_vol, initial_spikes, frame_rate,CB_detection_method=CB_detection_method,  
#     # #         simple_threshold=p_ST_cs,
#     # #         pnorm=p_CS, process_window=30, plotflag=False
#     # #     )
        
#     # #     # Step 2: Refine single spikes
#     # #     refined_SS, trace_noCS = refine_single_spikes(
#     # #         trace_vol, initial_spikes, complex_bursts_dict, frame_rate,
#     # #         process_window=30, pnorm=p_SS, min_spikes=5,SS_detection_method=SS_detection_method,
#     # #         simple_threshold=p_ST_ss,SS_height_cap=p_ss_cap, plotflag=False
#     # #     )
        
#     # #     # Step 3: Calculate spike heights
#     # #     heights, snr = spike_height_calculation2(
#     # #         refined_SS, trace_vol, complex_bursts_dict.get('trace_mf'), 
#     # #         trace_noCS, frame_rate
#     # #     )
#     # #     trace_vol = trace_vol/heights
#     # #     # Step 4: Detect complex spikes
#     # #     CS_spikes = detect_complex_spikes(
#     # #         trace_vol, complex_bursts_dict, np.ones_like(trace_vol),
#     # #         threshold=p_Th, plotflag=False
#     # #     )
        
#     # #     # Step 5: Refine all spikes
#     # #     complex_bursts_dict_final, refined_SS_final, all_CS_spikes, all_spikes = refine_all_spikes(
#     # #         complex_bursts_dict, CS_spikes, refined_SS
#     # #     )
        
#     # # except Exception as e:
#     # #     print(f"✗ Error during Qixin refinement: {e}")
#     # #     refined_SS_final = initial_spikes
#     # #     all_CS_spikes = np.array([], dtype=int)
#     # #     all_spikes = initial_spikes
#     # #     complex_bursts_dict_final = {}
#     # #     heights = None
#     # #     snr = None
    
#     # # # ===== 5. FINAL: DETECT BURSTS FROM VM (PRIMARY SPIKE DETECTION) =====
#     # # try:
#     # #     simple_spikes_vm, complex_spikes_vm, all_spikes_vm, trace_snr, vm, burst_metrics, complex_bursts_vm = detect_bursts_from_vm(
#     # #         trace_vol,
#     # #         np.ones_like(trace_vol),
#     # #         complex_bursts_dict_final,
#     # #         all_spikes,
#     # #         frame_rate,
#     # #         highpass=1.0,
#     # #         median_window=11,
#     # #         cb_amp_threshold=p_cb_amp,
#     # #         cb_duration_threshold=p_cb_dur,
#     # #         isi_threshold_ms=20,  # FIXED
#     # #         baseline_subtract=True,
#     # #         baseline_window_ms=20,
#     # #         baseline_percentile=10,
#     # #         merge_SS_ms = 20,
#     # #         merge_CB_ms = 5,
#     # #     )
        
#     #     # ✅ USE VM BURST DETECTION AS FINAL SPIKES
#     #     final_all_spikes =  all_spikes_vm
#     #     final_simple_spikes = simple_spikes_vm
#     #     final_complex_spikes = complex_spikes_vm
#     #     final_burst_dict = final_complex_bursts
        
#     # except Exception as e:
#     #     print(f"✗ Error during Vm burst detection: {e}")
#     #     # Fallback to Qixin detection
#     #     final_all_spikes = all_spikes
#     #     final_simple_spikes = refined_SS_final
#     #     final_complex_spikes = all_CS_spikes
#     #     final_burst_dict = complex_bursts_dict_final
#     #     burst_metrics = []
    
#     # ===== 6. SAVE REFINED SPIKE DETECTION PKL =====
#     save_data = {
#         'trace_vol': trace_vol,
#         'trace_cal': trace_cal,
#         'input_spikes': initial_spikes,
        
#         'vm_burst_dict': complex_bursts,  # ← PRIMARY
#         'vm_simple_spikes': simple_spikes,  # ← PRIMARY
#         'vm_complex_spikes': complex_spikes,  # ← PRIMARY
#         'vm_all_spikes': all_spikes,  # ← PRIMARY (used for analysis)
#         'vm_burst_metrics': burst_metrics,
#         'spike_heights': heights,
#         'spike_SNR': snr,
#         'detection_params': {
#             'pnorm_CS': p_CS,
#             'pnorm_SS': p_SS,
#             'threshold': p_Th,
#             'cb_amp_threshold': p_cb_amp,
#             'cb_duration_threshold': p_cb_dur,
#             'ss_cap': p_ss_cap,
#             'simple_threshold_ss': p_ST_ss,
#             'simple_threshold_cs': p_ST_cs,
#             'isi_threshold_ms': 20,  # FIXED
#             'frame_rate': frame_rate,
#             'preW': preW,
#             'postW': postW,
#             'detection_method': 'Vm_burst_detection'  # ← Document final method
#         }
#     }
    
#     pkl_path = os.path.join(save_dir, f'spike_detection_refined_new{name}.pkl')
#     with open(pkl_path, 'wb') as f:
#         pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
#     # ===== 7. CLASSIFY BURST SPIKES (using final Vm spikes) =====
#     burst_window_samples = int(burst_window_ms * frame_rate / 1000)
#     MyBurstSpike, numberOFspike = BurstC(all_spikes.tolist(), burst_window_samples)
    
#     # Extract single vs burst indices (from VM detection)
#     single_spike_idx = np.array([all_spikes[i] for i, n in enumerate(numberOFspike) if n == 1], dtype=int)
#     burst_spike_idx = np.array([all_spikes[i] for i, n in enumerate(numberOFspike) if n > 1], dtype=int)
    
#     # ===== 8. EXTRACT SPIKE-ALIGNED TRACES =====
#     vol_ax = np.linspace(0, len(trace_vol)/frame_rate, len(trace_vol))
#     cal_ax = np.linspace(0, len(trace_cal)/30, len(trace_cal))

#     cal_vol_SS, cal_vol_V, cal_indices, actual_lens = get_Vol_Cal_SS(
#         single_spike_idx, trace_vol, trace_cal,
#         vol_ax, cal_ax,
#         preW=preW, postW=postW
#     )

#     # ===== 9. CLASSIFY SPIKES (SPARSE vs ADP) - HANDLE VARIABLE LENGTHS =====
#     if len(cal_vol_V) > 0:
#         max_len = max(len(v) for v in cal_vol_V)
#         first_pre_ms = actual_lens[0]['pre'] * 1000 / frame_rate
#         t_ms_base = np.arange(max_len) * 1000 / frame_rate - first_pre_ms
        
#         sparse_mask = []
#         auc_sig_all = []
#         auc_nse_all = []
#         auc_thr = np.nan
        
#         #print(f"  Processing {len(cal_vol_V)} traces with variable lengths...")
        
#         for spike_idx_i, v_trace in enumerate(cal_vol_V):
#             if len(v_trace) == 0:
#                 sparse_mask.append(False)
#                 auc_sig_all.append(np.nan)
#                 auc_nse_all.append(np.nan)
#                 continue
            
#             actual_pre_ms = actual_lens[spike_idx_i]['pre'] * 1000 / frame_rate
#             t_ms_spike = np.arange(len(v_trace)) * 1000 / frame_rate - actual_pre_ms
            
#             try:
#                 mask_i, auc_sig_i, auc_nse_i, thr_i = classify_subthreshold_with_spike_removed_samewin(
#                     [v_trace], t_ms_spike,
#                     signal_win=(-20, 100),
#                     noise_win=(-140, -20),
#                     cut_win=(-2, 4),
#                     k=3.0,
#                     positive_only=True
#                 )
                
#                 sparse_mask.append(bool(mask_i[0]))
#                 auc_sig_all.append(auc_sig_i[0])
#                 auc_nse_all.append(auc_nse_i[0])
#                 auc_thr = thr_i
                
#             except Exception as e:
#                 print(f"    ⚠ Warning spike {spike_idx_i}: {e}")
#                 sparse_mask.append(False)
#                 auc_sig_all.append(np.nan)
#                 auc_nse_all.append(np.nan)
        
#         sparse_mask = np.array(sparse_mask, dtype=bool)
#         auc_sig = np.array(auc_sig_all, dtype=float)
#         auc_noise = np.array(auc_nse_all, dtype=float)
        
#         valid_noise = auc_noise[~np.isnan(auc_noise)]
#         if len(valid_noise) > 0:
#             mu = np.nanmean(valid_noise)
#             sd = np.nanstd(valid_noise)
#             auc_thr = mu + 3.0 * sd
        
#         sparse_single_spikes = single_spike_idx[sparse_mask] if len(sparse_mask) > 0 else np.array([])
#         adp_single_spikes = single_spike_idx[~sparse_mask] if len(sparse_mask) > 0 else np.array([])
#     else:
#         sparse_mask = np.array([], dtype=bool)
#         auc_sig = np.array([])
#         auc_noise = np.array([])
#         auc_thr = np.nan
#         sparse_single_spikes = np.array([])
#         adp_single_spikes = np.array([])

#     pct_sparse = 100 * sparse_mask.sum() / len(sparse_mask) if len(sparse_mask) > 0 else 0
    
#     # ===== 10. COMPILE RESULTS (Using VM Burst Detection) =====
#     results_dict = {
#         'trace_vol': trace_vol,
#         'trace_cal': trace_cal,
#         'spike_data': save_data,
#         'refined_spikes': all_spikes,  # ✅ VM BURST DETECTION (FINAL)
#         'single_spikes': simple_spikes,  # ✅ VM BURST DETECTION (FINAL)
#         'complex_spikes': complex_spikes,  # ✅ VM BURST DETECTION (FINAL)
#         'burst_spikes': burst_spike_idx,
#         'sparse_single_spikes': sparse_single_spikes,
#         'adp_single_spikes': adp_single_spikes,
#         'vol_traces': cal_vol_V,
#         'cal_traces': cal_vol_SS,
#         'cal_indices': cal_indices,
#         'auc_sig': auc_sig,
#         'auc_noise': auc_noise,
#         'auc_threshold': auc_thr,
#         'sparse_mask': sparse_mask,
#         'burst_metrics': burst_metrics,
#         'params': {
#             'frame_rate': frame_rate,
#             'preW': preW,
#             'postW': postW,
#             'burst_window_ms': burst_window_ms,
#             'pnorm_CS': p_CS,
#             'pnorm_SS': p_SS,
#             'threshold': p_Th,
            
#             'cb_amp_threshold': p_cb_amp,
#             'cb_duration_threshold': p_cb_dur,
#             'isi_threshold_ms': 20,
#             'detection_method': 'Vm_burst_detection'  # ← FINAL METHOD
#         }
#     }
    
#     # Save results summary
#     summary_path = os.path.join(save_dir, f'spike_classification_summary_new{name}.pkl')
#     with open(summary_path, 'wb') as f:
#         pickle.dump(results_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
    
#     # print(f"\n✓ SPIKE DETECTION COMPLETE (Vm Burst Method)")
#     # print(f"  Total spikes: {len(final_all_spikes)}")
#     # print(f"  Simple spikes: {len(final_simple_spikes)}")
#     # print(f"  Complex spikes: {len(final_complex_spikes)}")
#     # print(f"  Sparse (no ADP): {len(sparse_single_spikes)} ({pct_sparse:.1f}%)")
    
#     return results_dict


In [4]:
def Split_cal(chane_p,VolT,CalT,volax,calax,mot,spikeIdx):
    calMot = []
    volMot = []
    volRes = []
    calRes = []
    spikeMot = []
    spikeRes = []
    if chane_p[0]<5:
        chane_p = chane_p[1:]
    for i in range(len(chane_p)+1):
        if i == 0:
            sIDX = 0
            eIdx = chane_p[i]
        elif i > 0 and i < len(chane_p):
            sIDX = chane_p[i-1]
            eIdx = chane_p[i]
        elif i ==len(chane_p):
            sIDX = chane_p[-1]
            eIdx = len(VolT)-1
        #print(s)
        calS = VolToCalIdx(sIDX,volax,calax)
        calE = VolToCalIdx(eIdx,volax,calax)
        if mot[sIDX+7] == 1:
            #print(f'motor{sIDX}')
            spikeId = [i for i in spikeIdx if i >= sIDX and i < eIdx]
            spikeMot.append(np.array(spikeId) - sIDX)
            calMot.append(CalT[calS:calE+1])
            volMot.append(VolT[sIDX:eIdx+1])
        if mot[sIDX+7] == 0:
            #print(f'rest{sIDX}')
            
            spikeId = [i for i in spikeIdx if i >= sIDX and i < eIdx]
            spikeRes.append(np.array(spikeId) - sIDX)
            calRes.append(CalT[calS:calE+1])
            #print(len(calRes))
            volRes.append(VolT[sIDX:eIdx+1])
            #print(len(volRes))
    return calMot,calRes,spikeMot,spikeRes,volMot,volRes



In [5]:
import copy

def load_and_refine_pyr_spikes(
    path, trace_vol, trace_cal,
    frame_rate=500,
    default_pn_CS=0.5,
    default_pn_SS=0.3,
    default_thresh=0.5,
    default_cb_amp_threshold=0.3,
    default_cb_duration_threshold=20,
    CB_detection_method='simple',   # 'simple' or 'volpy_based'
    SS_detection_method='simple',
    simple_threshold_cs=6,
    simple_threshold_SS=6,
    ss_height_cap=0.5,
    f_hp=20,
    preW=75,
    postW=100,
    burst_window_ms=15,
    save_dir=None,
    interactive=True,
    spikeID=None,
    params_pr=None,
    save_only=False,
    final_spikes=None,
    selected_params=None,
    plateau_amp_threshold=0.8,
    plateau_duration_threshold=100,
    plateau_kernel_ms=100,
    plateau_score_min_ms=80,
    name=''
):
    """
    Complete pipeline for pyramidal cell spike refinement using qixin_spike_detection.
    Now supports interactive tuning of high-pass frequency (f_hp).
    """

    if save_dir is None:
        save_dir = path
    os.makedirs(save_dir, exist_ok=True)

    # ===== SAVE-ONLY: WRITE PICKLE WITHOUT RE-RUNNING =====
    if save_only:
        if final_spikes is None or not isinstance(final_spikes, dict):
            raise ValueError('save_only=True requires final_spikes dict')

        p = selected_params if isinstance(selected_params, dict) else (
            params_pr if isinstance(params_pr, dict) else {}
        )

        p_CS = p.get('pnorm_CS', default_pn_CS)
        p_SS = p.get('pnorm_SS', default_pn_SS)
        p_Th = p.get('threshold', default_thresh)
        p_cb_amp = p.get('cb_amp_threshold', default_cb_amp_threshold)
        p_cb_dur = p.get('cb_duration_threshold', default_cb_duration_threshold)
        p_ss_cap = p.get('ss_cap', ss_height_cap)
        p_ST_ss = p.get('simple_threshold_ss', simple_threshold_SS)
        p_ST_cs = p.get('simple_threshold_cs', simple_threshold_cs)
        p_f_hp = p.get('f_hp', f_hp)
        p_plateau_amp = p.get('plateau_amp_threshold', plateau_amp_threshold)
        p_plateau_dur = p.get('plateau_duration_threshold', plateau_duration_threshold)
        p_plateau_kernel = p.get('plateau_kernel_ms', plateau_kernel_ms)
        p_plateau_score = p.get('plateau_score_min_ms', plateau_score_min_ms)

        all_spikes = final_spikes.get('vm_all_spikes', final_spikes.get('all_spikes', []))
        complex_spikes = final_spikes.get('vm_complex_spikes', final_spikes.get('complex_spikes', []))
        simple_spikes = final_spikes.get('vm_simple_spikes', final_spikes.get('simple_spikes', []))
        complex_bursts = final_spikes.get(
            'vm_burst_dict',
            final_spikes.get('complex_bursts', final_spikes.get('complex_bursts_dict', {}))
        )
        burst_metrics = final_spikes.get('burst_metrics', final_spikes.get('vm_burst_metrics', []))
        heights = final_spikes.get(
            'heights',
            final_spikes.get('spike_heights', final_spikes.get('spike_heights_interpolated', []))
        )
        snr = final_spikes.get(
            'snr',
            final_spikes.get('SNR', final_spikes.get('spike_SNR', final_spikes.get('SNR_interpolated', [])))
        )
        input_spikes = final_spikes.get('input_spikes', spikeID if spikeID is not None else [])

        save_data = {
            'trace_vol': trace_vol,
            'trace_cal': trace_cal,
            'input_spikes': np.asarray(input_spikes, dtype=int),
            'vm': np.asarray(final_spikes.get('vm', []), dtype=float),
            'vm_burst_dict': complex_bursts if isinstance(complex_bursts, dict) else {},
            'vm_simple_spikes': np.asarray(simple_spikes, dtype=int),
            'vm_complex_spikes': np.asarray(complex_spikes, dtype=int),
            'vm_all_spikes': np.asarray(all_spikes, dtype=int),
            'vm_burst_metrics': burst_metrics if burst_metrics is not None else [],
            'vm_plateaus_dict': copy.deepcopy(final_spikes.get('vm_plateaus_dict', {})),
            'spike_heights': np.asarray(heights, dtype=float) if heights is not None else np.array([], dtype=float),
            'spike_SNR': np.asarray(snr, dtype=float) if snr is not None else np.array([], dtype=float),
            'detection_params': {
                'pnorm_CS': p_CS,
                'pnorm_SS': p_SS,
                'threshold': p_Th,
                'cb_amp_threshold': p_cb_amp,
                'cb_duration_threshold': p_cb_dur,
                'ss_cap': p_ss_cap,
                'simple_threshold_ss': p_ST_ss,
                'simple_threshold_cs': p_ST_cs,
                'f_hp': p_f_hp,
                'isi_threshold_ms': 20,
                'frame_rate': frame_rate,
                'preW': preW,
                'postW': postW,
                'detection_method': p.get('detection_method', 'Vm_burst_detection'),
                'plateau_amp_threshold': p_plateau_amp,
                'plateau_duration_threshold': p_plateau_dur,
                'plateau_kernel_ms': p_plateau_kernel,
                'plateau_score_min_ms': p_plateau_score
            }
        }

        pkl_path = os.path.join(save_dir, f'spike_detection_refined_new{name}.pkl')
        with open(pkl_path, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
        return save_data

    # ===== 2. LOAD OR DETECT INITIAL SPIKES =====
    spike_pkl_path = os.path.join(path, 'cell__CS_detection.pkl')  # kept for compatibility
    initial_spikes = spikeID
    prev_params = params_pr

    if initial_spikes is None or len(initial_spikes) == 0:
        threshold = np.percentile(np.abs(np.diff(trace_vol)), 95)
        initial_spikes = np.where(np.abs(np.diff(trace_vol)) > threshold)[0].tolist()

    initial_spikes = np.asarray(initial_spikes, dtype=int)

    init_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
    init_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
    init_thresh = prev_params['threshold'] if prev_params else default_thresh
    init_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
    #init_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
    init_cb_dur = default_cb_duration_threshold  # FIXED to default for consistency, can be made interactive later
    init_simple_thresh_CS = prev_params.get('simple_threshold_cs', simple_threshold_cs) if prev_params else simple_threshold_cs
    init_simple_thresh_SS = prev_params.get('simple_threshold_ss', simple_threshold_SS) if prev_params else simple_threshold_SS
    init_ss_cap = prev_params.get('ss_cap', ss_height_cap) if prev_params else ss_height_cap
    init_f_hp = prev_params.get('f_hp', f_hp) if prev_params else f_hp

    def _call_detect_bursts_vm(trace_norm, bursts_final, all_s, p_cb_amp, p_cb_dur):
        ret = detect_bursts_from_vm(
            trace_norm,
            np.ones_like(trace_norm),
            bursts_final,
            all_s,
            frame_rate,
            highpass=1.0,
            median_window=11,
            cb_amp_threshold=p_cb_amp,
            cb_duration_threshold=p_cb_dur,
            isi_threshold_ms=20,
            baseline_subtract=True,
            baseline_window_ms=20,
            baseline_percentile=10,
            merge_SS_ms=20,
            merge_CB_ms=5,
            plateau_duration_threshold=plateau_duration_threshold,
            plateau_amp_threshold=plateau_amp_threshold,
            plateau_kernel_ms=plateau_kernel_ms,
            plateau_score_min_ms=plateau_score_min_ms,
            plotflag=False,
        )

        if not isinstance(ret, tuple):
            raise RuntimeError('detect_bursts_from_vm returned non-tuple output')
        if len(ret) < 7:
            raise RuntimeError(f'detect_bursts_from_vm returned {len(ret)} values, expected >=7')

        simple_spikes_vm = ret[0]
        complex_spikes_vm = ret[1]
        all_spikes_vm = ret[2]
        trace_snr = ret[3]
        vm = ret[4]
        burst_metrics = ret[5]
        complex_bursts_vm = ret[6]
        vm_plateaus_dict = ret[7] if len(ret) >= 8 else {}

        return (
            simple_spikes_vm,
            complex_spikes_vm,
            all_spikes_vm,
            trace_snr,
            vm,
            burst_metrics,
            complex_bursts_vm,
            vm_plateaus_dict,
        )

    # ===== 2.5 FIRST ROUND RUN =====
    complex_bursts_dict, segment_bounds = complex_bursts_detection(
        trace_vol, initial_spikes, frame_rate,
        pnorm=init_CS,
        process_window=30,
        plotflag=False,
        CB_detection_method=CB_detection_method,
        simple_threshold=init_simple_thresh_CS
    )

    refined_SS, trace_noCS = refine_single_spikes(
        trace_vol, initial_spikes, complex_bursts_dict, frame_rate,
        process_window=30, pnorm=init_CS, f_hp=init_f_hp, min_spikes=5, plotflag=False,
        SS_detection_method=SS_detection_method, simple_threshold_SS=init_simple_thresh_SS
    )
    initial_spikes = refined_SS
    spike_heights_interpolated, SNR_interpolated = spike_height_calculation2(
        refined_SS, trace_vol, complex_bursts_dict['trace_mf'], trace_noCS, frame_rate, plotflag=False
    )

    # ===== 3. INTERACTIVE REFINEMENT =====
    if interactive and len(initial_spikes) > 0:
        (
            p_CS, p_SS, p_Th, p_cb_amp, p_cb_dur,
            p_ss_cap, p_ST_ss, p_ST_cs, p_f_hp,
            all_spikes, complex_spikes, simple_spikes, complex_bursts,
            vm, burst_metrics, heights, snr, vm_plateaus_dict
        ) = select_params_interactive_extended(
            trace=trace_vol,
            spike_idx=initial_spikes,
            frame_rate=frame_rate,
            init_CS=init_CS,
            init_SS=init_SS,
            init_threshold=init_thresh,
            init_cb_amp_threshold=init_cb_amp,
            init_cb_duration_threshold=init_cb_dur,
            init_threshold_CS=init_simple_thresh_CS,
            init_threshold_SS=init_simple_thresh_SS,
            inint_ss_cap=init_ss_cap,
            init_f_hp=init_f_hp,
            CB_detection_method=CB_detection_method,
            SS_detection_method=SS_detection_method,
            save_html_path=os.path.join(save_dir, f'latest_spike_detected_and_classified{name}.html'),
            plateau_amp_threshold=plateau_amp_threshold,
            plateau_duration_threshold=plateau_duration_threshold,
            plateau_kernel_ms=plateau_kernel_ms,
            plateau_score_min_ms=plateau_score_min_ms
        )
    else:
        p_CS = init_CS
        p_SS = init_SS
        p_Th = init_thresh
        p_cb_amp = init_cb_amp
        p_cb_dur = init_cb_dur
        p_ss_cap = init_ss_cap
        p_ST_ss = init_simple_thresh_SS
        p_ST_cs = init_simple_thresh_CS
        p_f_hp = init_f_hp
        vm_plateaus_dict = {}

        # non-interactive fallback
        complex_bursts, _ = complex_bursts_detection(
            trace_vol, initial_spikes, frame_rate,
            pnorm=p_CS, process_window=30, plotflag=False,
            CB_detection_method=CB_detection_method,
            simple_threshold=p_ST_cs
        )

        refined_SS, trace_noCS = refine_single_spikes(
            trace_vol, initial_spikes, complex_bursts, frame_rate,
            process_window=30, pnorm=p_SS, f_hp=p_f_hp, min_spikes=5, plotflag=False,
            SS_detection_method=SS_detection_method, simple_threshold_SS=p_ST_ss
        )

        trace_mf = complex_bursts.get('trace_mf', median_filter(trace_vol, size=11))
        heights, snr = spike_height_calculation2(refined_SS, trace_vol, trace_mf, trace_noCS, frame_rate)

        height_scale = float(np.nanmedian(heights)) if np.size(heights) else 1.0
        height_scale = max(height_scale, 1e-12)
        trace_norm = trace_vol / height_scale

        idx_ss = refined_SS.astype(int)
        idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
        if p_ss_cap is not None:
            refined_SS = idx_ss[trace_norm[idx_ss] >= float(p_ss_cap)]

        cs_spikes_raw = detect_complex_spikes(
            trace_norm, complex_bursts, np.ones_like(trace_norm), threshold=p_Th, plotflag=False
        )

        complex_bursts_final, refined_SS_final, all_cs, all_s = refine_all_spikes(
            complex_bursts, cs_spikes_raw, refined_SS
        )

        (
            simple_spikes, complex_spikes, all_spikes, trace_snr, vm, burst_metrics, complex_bursts, vm_plateaus_dict
        ) = _call_detect_bursts_vm_local(
            trace_norm, complex_bursts_final, all_s, p_cb_amp, p_cb_dur
        )

    # ===== 6. SAVE REFINED SPIKE DETECTION PKL =====
    save_data = {
        'trace_vol': trace_vol,
        'trace_cal': trace_cal,
        'input_spikes': initial_spikes,
        'vm': np.asarray(vm, dtype=float),
        'vm_burst_dict': complex_bursts,
        'vm_simple_spikes': simple_spikes,
        'vm_complex_spikes': complex_spikes,
        'vm_all_spikes': all_spikes,
        'vm_burst_metrics': burst_metrics,
        'vm_plateaus_dict': copy.deepcopy(vm_plateaus_dict),
        'spike_heights': heights,
        'spike_SNR': snr,
        'detection_params': {
            'pnorm_CS': p_CS,
            'pnorm_SS': p_SS,
            'threshold': p_Th,
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'ss_cap': p_ss_cap,
            'simple_threshold_ss': p_ST_ss,
            'simple_threshold_cs': p_ST_cs,
            'f_hp': p_f_hp,
            'isi_threshold_ms': 20,
            'frame_rate': frame_rate,
            'preW': preW,
            'postW': postW,
            'detection_method': 'Vm_burst_detection',
            'plateau_amp_threshold': float(plateau_amp_threshold),
            'plateau_duration_threshold': float(plateau_duration_threshold),
            'plateau_kernel_ms': float(plateau_kernel_ms),
            'plateau_score_min_ms': float(plateau_score_min_ms)
        }
    }

    pkl_path = os.path.join(save_dir, f'spike_detection_refined_new{name}.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)

    # ===== 7. CLASSIFY BURST SPIKES =====
    burst_window_samples = int(burst_window_ms * frame_rate / 1000)
    MyBurstSpike, numberOFspike = BurstC(all_spikes.tolist(), burst_window_samples)

    single_spike_idx = np.array([all_spikes[i] for i, n in enumerate(numberOFspike) if n == 1], dtype=int)
    burst_spike_idx = np.array([all_spikes[i] for i, n in enumerate(numberOFspike) if n > 1], dtype=int)

    # ===== 8. EXTRACT SPIKE-ALIGNED TRACES =====
    vol_ax = np.linspace(0, len(trace_vol) / frame_rate, len(trace_vol))
    cal_ax = np.linspace(0, len(trace_cal) / 30, len(trace_cal))

    cal_vol_SS, cal_vol_V, cal_indices, actual_lens = get_Vol_Cal_SS(
        single_spike_idx, trace_vol, trace_cal, vol_ax, cal_ax, preW=preW, postW=postW
    )

    # ===== 9. CLASSIFY SPIKES (SPARSE vs ADP) =====
    if len(cal_vol_V) > 0:
        max_len = max(len(v) for v in cal_vol_V)
        first_pre_ms = actual_lens[0]['pre'] * 1000 / frame_rate
        t_ms_base = np.arange(max_len) * 1000 / frame_rate - first_pre_ms

        sparse_mask = []
        auc_sig_all = []
        auc_nse_all = []
        auc_thr = np.nan

        for spike_idx_i, v_trace in enumerate(cal_vol_V):
            if len(v_trace) == 0:
                sparse_mask.append(False)
                auc_sig_all.append(np.nan)
                auc_nse_all.append(np.nan)
                continue

            actual_pre_ms = actual_lens[spike_idx_i]['pre'] * 1000 / frame_rate
            t_ms_spike = np.arange(len(v_trace)) * 1000 / frame_rate - actual_pre_ms

            try:
                mask_i, auc_sig_i, auc_nse_i, thr_i = classify_subthreshold_with_spike_removed_samewin(
                    [v_trace], t_ms_spike,
                    signal_win=(-20, 100),
                    noise_win=(-140, -20),
                    cut_win=(-2, 4),
                    k=3.0,
                    positive_only=True
                )

                sparse_mask.append(bool(mask_i[0]))
                auc_sig_all.append(auc_sig_i[0])
                auc_nse_all.append(auc_nse_i[0])
                auc_thr = thr_i

            except Exception as e:
                print(f"    ⚠ Warning spike {spike_idx_i}: {e}")
                sparse_mask.append(False)
                auc_sig_all.append(np.nan)
                auc_nse_all.append(np.nan)

        sparse_mask = np.array(sparse_mask, dtype=bool)
        auc_sig = np.array(auc_sig_all, dtype=float)
        auc_noise = np.array(auc_nse_all, dtype=float)

        valid_noise = auc_noise[~np.isnan(auc_noise)]
        if len(valid_noise) > 0:
            mu = np.nanmean(valid_noise)
            sd = np.nanstd(valid_noise)
            auc_thr = mu + 3.0 * sd

        sparse_single_spikes = single_spike_idx[sparse_mask] if len(sparse_mask) > 0 else np.array([])
        adp_single_spikes = single_spike_idx[~sparse_mask] if len(sparse_mask) > 0 else np.array([])
    else:
        sparse_mask = np.array([], dtype=bool)
        auc_sig = np.array([])
        auc_noise = np.array([])
        auc_thr = np.nan
        sparse_single_spikes = np.array([])
        adp_single_spikes = np.array([])

    pct_sparse = 100 * sparse_mask.sum() / len(sparse_mask) if len(sparse_mask) > 0 else 0

    # ===== 10. COMPILE RESULTS =====
    results_dict = {
        'trace_vol': trace_vol,
        'trace_cal': trace_cal,
        'spike_data': save_data,
        'refined_spikes': all_spikes,
        'single_spikes': simple_spikes,
        'complex_spikes': complex_spikes,
        'burst_spikes': burst_spike_idx,
        'sparse_single_spikes': sparse_single_spikes,
        'adp_single_spikes': adp_single_spikes,
        'vol_traces': cal_vol_V,
        'cal_traces': cal_vol_SS,
        'cal_indices': cal_indices,
        'auc_sig': auc_sig,
        'auc_noise': auc_noise,
        'auc_threshold': auc_thr,
        'sparse_mask': sparse_mask,
        'burst_metrics': burst_metrics,
        'params': {
            'frame_rate': frame_rate,
            'preW': preW,
            'postW': postW,
            'burst_window_ms': burst_window_ms,
            'pnorm_CS': p_CS,
            'pnorm_SS': p_SS,
            'threshold': p_Th,
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'f_hp': p_f_hp,
            'isi_threshold_ms': 20,
            'detection_method': 'Vm_burst_detection',
            'plateau_amp_threshold': float(plateau_amp_threshold),
            'plateau_duration_threshold': float(plateau_duration_threshold),
            'plateau_kernel_ms': float(plateau_kernel_ms),
            'plateau_score_min_ms': float(plateau_score_min_ms)
        }
    }

    summary_path = os.path.join(save_dir, f'spike_classification_summary_new{name}.pkl')
    with open(summary_path, 'wb') as f:
        pickle.dump(results_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    return results_dict

def select_params_interactive_extended(
    trace, spike_idx, frame_rate,
    init_CS=0.5, init_SS=0.3,
    init_threshold=0.5, init_cb_amp_threshold=0.3,
    init_cb_duration_threshold=20, init_threshold_CS=6, init_threshold_SS=6,
    inint_ss_cap=0.5, init_f_hp=20,
    CB_detection_method='simple',
    SS_detection_method='simple',
    CS_SP_TH=0.5, save_html_path=None, n='',
    plateau_amp_threshold=0.8,
    plateau_duration_threshold=100,
    plateau_kernel_ms=100,
    plateau_score_min_ms=80
):
    """
    Opens an interactive window to tune spike detection parameters.
    Now also supports interactive tuning of high-pass frequency (f_hp).
    Press ENTER when done, or window auto-closes after timeout.
    """

    plt.close('all')

    if len(spike_idx) == 0:
        return (
            init_CS, init_SS, init_threshold, init_cb_amp_threshold,
            init_cb_duration_threshold, inint_ss_cap, init_threshold_SS,
            init_threshold_CS, init_f_hp,
            np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int),
            {}, np.array([], dtype=float), [], np.array([], dtype=float), np.array([], dtype=float), {}
        )

    selected_params = {
        'pnorm_CS': init_CS,
        'pnorm_SS': init_SS,
        'ST_cs': init_threshold_CS,
        'ST_ss': init_threshold_SS,
        'ss_cap': inint_ss_cap,
        'threshold': float(init_threshold),
        'cb_amp_threshold': init_cb_amp_threshold,
        'cb_duration_threshold': init_cb_duration_threshold,
        'f_hp': float(init_f_hp),
        'isi_threshold_ms': 20
    }

    final_spikes = {
        "all_spikes_vm": np.array([], dtype=int),
        "complex_spikes_vm": np.array([], dtype=int),
        "simple_spikes_vm": np.array([], dtype=int),
        "complex_burst_vm": {},
        "burst_metrics": [],
        "vm": np.array([], dtype=float),
        "heights": np.array([], dtype=float),
        "SNR": np.array([], dtype=float),
        "vm_plateaus_dict": {},
    }

    # 65% trace area (top) + 35% controls area (bottom)
    fig = plt.figure(figsize=(15, 11))
    ax = fig.add_axes([0.06, 0.35, 0.92, 0.62])
    ax_vm = ax.twinx()
    ax_vm.set_ylabel('Vm (norm)', color='green')
    ax_vm.yaxis.set_label_position('right')
    ax_vm.yaxis.tick_right()
    ax_vm.spines['right'].set_visible(True)
    ax_vm.spines['right'].set_position(('outward', 6))
    ax_vm.spines['right'].set_color('green')
    ax_vm.spines['right'].set_linewidth(1.2)
    ax_vm.tick_params(axis='y', right=True, labelright=True, colors='green', width=1.0)

    t_axis = np.arange(len(trace)) / frame_rate
    trace_line, = ax.plot(t_axis, trace, 'k', lw=0.5, alpha=0.6, label='Trace (norm)')
    vm_line, = ax_vm.plot(t_axis, np.full(len(trace), np.nan, dtype=float), color='green', lw=0.9, alpha=0.85, label='Vm (norm)')

    scat_in = ax.scatter([], [], c='0.3', s=12, label='Input spike_idx', zorder=3)
    scat_ss = ax.scatter([], [], c='b', s=30, label='Single Spikes (Qixin)', zorder=5)
    scat_cs = ax.scatter([], [], c='r', s=50, marker='x', label='Complex Spikes (Qixin)', zorder=6)
    scat_vm_simple = ax.scatter([], [], c='cyan', s=20, marker='o', label='Simple Spikes (Vm)', zorder=4)
    scat_vm_complex = ax.scatter([], [], c='magenta', s=40, marker='x', label='Complex Spikes (Vm)', zorder=7)
    ss_cap_line = ax.axhline(np.nan, color='b', ls='--', lw=1, alpha=0.6, label='SS cap (norm)')

    count_text = ax.text(
        0.02, 0.95, "Initializing...", transform=ax.transAxes,
        bbox=dict(facecolor='white', alpha=0.8), fontsize=10
    )
    burst_spans = []
    latest_state = {}

    def save_latest_state_to_html():
        if not save_html_path:
            return
        if not isinstance(latest_state, dict) or 'trace_norm' not in latest_state:
            print('No latest detection state available to save.')
            return
        try:
            out_dir = os.path.dirname(save_html_path)
            if out_dir:
                os.makedirs(out_dir, exist_ok=True)

            trace_norm = np.asarray(latest_state['trace_norm'], dtype=float).ravel()
            t = np.arange(len(trace_norm)) / frame_rate

            vm_norm = latest_state.get('vm_norm', None)
            if vm_norm is not None:
                vm_norm = np.asarray(vm_norm, dtype=float).ravel()

            fig_html = go.Figure()
            fig_html.add_trace(go.Scattergl(
                x=t, y=trace_norm, mode='lines', name='Trace (norm)',
                line=dict(color='black', width=1), opacity=0.6
            ))

            if vm_norm is not None and vm_norm.size > 0:
                t_vm = np.arange(vm_norm.size) / frame_rate
                fig_html.add_trace(go.Scattergl(
                    x=t_vm, y=vm_norm, mode='lines', name='Vm (norm)',
                    line=dict(color='green', width=1), opacity=0.9
                ))

            def _add_points(name, idx, color, symbol, size):
                if idx is None:
                    return
                idx = np.asarray(idx, dtype=int)
                idx = idx[(idx >= 0) & (idx < len(trace_norm))]
                if idx.size == 0:
                    return
                fig_html.add_trace(go.Scattergl(
                    x=t[idx], y=trace_norm[idx], mode='markers', name=name,
                    marker=dict(color=color, symbol=symbol, size=size)
                ))

            _add_points('Input spike_idx', latest_state.get('idx_in'), 'rgba(80,80,80,0.8)', 'circle', 6)
            _add_points('Single Spikes (Qixin)', latest_state.get('idx_ss'), 'blue', 'circle', 8)
            _add_points('Complex Spikes (Qixin)', latest_state.get('idx_cs'), 'red', 'x', 10)
            _add_points('Simple Spikes (Vm)', latest_state.get('idx_vm_simple'), 'cyan', 'circle-open', 8)
            _add_points('Complex Spikes (Vm)', latest_state.get('idx_vm_complex'), 'magenta', 'x', 10)

            starts = latest_state.get('burst_starts', None)
            ends = latest_state.get('burst_ends', None)
            if starts is not None and ends is not None:
                starts = np.asarray(starts, dtype=float).ravel()
                ends = np.asarray(ends, dtype=float).ravel()
                for s, e in zip(starts, ends):
                    try:
                        fig_html.add_vrect(
                            x0=float(s) / frame_rate, x1=float(e) / frame_rate,
                            fillcolor='yellow', opacity=0.2, line_width=0, layer='below'
                        )
                    except Exception:
                        fig_html.add_shape(
                            type='rect', x0=float(s) / frame_rate, x1=float(e) / frame_rate,
                            y0=0, y1=1, yref='paper',
                            fillcolor='yellow', opacity=0.2, line_width=0, layer='below'
                        )

            plateaus = latest_state.get('vm_plateaus_dict', {}) if isinstance(latest_state.get('vm_plateaus_dict', {}), dict) else {}
            p_starts = np.asarray(plateaus.get('starts', []), dtype=float).ravel()
            p_ends = np.asarray(plateaus.get('ends', []), dtype=float).ravel()
            p_locs = np.asarray(plateaus.get('locs', []), dtype=int).ravel()
            for s, e in zip(p_starts, p_ends):
                if not (np.isfinite(s) and np.isfinite(e)):
                    continue
                if e < s:
                    s, e = e, s
                try:
                    fig_html.add_vrect(
                        x0=float(s) / frame_rate, x1=float(e) / frame_rate,
                        fillcolor='rgba(255,140,0,0.20)', opacity=1.0, line_width=0, layer='below'
                    )
                except Exception:
                    fig_html.add_shape(
                        type='rect', x0=float(s) / frame_rate, x1=float(e) / frame_rate,
                        y0=0, y1=1, yref='paper',
                        fillcolor='rgba(255,140,0,0.20)', opacity=1.0, line_width=0, layer='below'
                    )
            if p_locs.size > 0:
                valid = p_locs[(p_locs >= 0) & (p_locs < len(trace_norm))]
                if valid.size > 0:
                    fig_html.add_trace(go.Scattergl(
                        x=t[valid], y=trace_norm[valid], mode='markers', name='Plateau loci',
                        marker=dict(color='darkorange', symbol='diamond', size=8)
                    ))

            p_ss_cap = latest_state.get('p_ss_cap', None)
            if p_ss_cap is not None and np.isfinite(p_ss_cap):
                try:
                    fig_html.add_hline(
                        y=float(p_ss_cap), line_dash='dash', line_color='blue', opacity=0.5
                    )
                except Exception:
                    fig_html.add_shape(
                        type='line',
                        x0=float(t[0]), x1=float(t[-1]),
                        y0=float(p_ss_cap), y1=float(p_ss_cap),
                        line=dict(color='blue', dash='dash', width=1), opacity=0.5
                    )

            latest_candidates = []
            for label, key in [
                ('Complex Spikes (Vm)', 'idx_vm_complex'),
                ('Complex Spikes (Qixin)', 'idx_cs'),
                ('Simple Spikes (Vm)', 'idx_vm_simple'),
                ('Single Spikes (Qixin)', 'idx_ss')
            ]:
                arr = latest_state.get(key, None)
                if arr is None:
                    continue
                arr = np.asarray(arr, dtype=int)
                arr = arr[(arr >= 0) & (arr < len(trace_norm))]
                if arr.size:
                    latest_candidates.append((int(arr.max()), label))

            if latest_candidates:
                latest_idx, latest_label = max(latest_candidates, key=lambda x: x[0])
                fig_html.add_trace(go.Scattergl(
                    x=[t[latest_idx]], y=[trace_norm[latest_idx]],
                    mode='markers+text', name='Latest spike',
                    marker=dict(color='gold', size=14, symbol='star'),
                    text=[f'Latest: {latest_label}'],
                    textposition='top center'
                ))

            params = latest_state.get('params', {}) if isinstance(latest_state.get('params', {}), dict) else {}
            title = (
                'Latest spike detected & classified (interactive)'
                f" | CS={params.get('pnorm_CS', np.nan):.2f}"
                f" SS={params.get('pnorm_SS', np.nan):.2f}"
                f" T={params.get('threshold', np.nan):.2f}"
                f" HP={params.get('f_hp', np.nan):.1f}Hz"
            )
            fig_html.update_layout(
                title=title,
                xaxis_title='Time (s)',
                yaxis_title='Trace (norm)',
                template='plotly_white',
                height=650,
                legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
            )
            fig_html.write_html(save_html_path, include_plotlyjs=True, full_html=True)
            print(f'Saved HTML figure: {save_html_path}')
        except Exception as e:
            print(f'Failed to save HTML figure: {e}')

    ax.set_title(f"Input Spikes: {len(spike_idx)} | Adjust Sliders & Press ENTER")
    ax.set_ylabel('Trace (norm)')
    ax.legend(loc='upper right', fontsize=8)

    ax_cs = plt.axes([0.12, 0.315, 0.76, 0.022])
    ax_ss = plt.axes([0.12, 0.284, 0.76, 0.022])
    ax_th = plt.axes([0.12, 0.253, 0.76, 0.022])
    ax_cb_amp = plt.axes([0.12, 0.222, 0.76, 0.022])
    ax_cb_dur = plt.axes([0.12, 0.191, 0.76, 0.022])
    ax_cb_st = plt.axes([0.12, 0.160, 0.76, 0.022])
    ax_ss_st = plt.axes([0.12, 0.129, 0.76, 0.022])
    ax_cb_cap = plt.axes([0.12, 0.098, 0.76, 0.022])
    ax_fhp = plt.axes([0.12, 0.067, 0.76, 0.022])

    slider_cs = Slider(ax_cs, 'P-Norm CS', 0.0, 4.0, valinit=init_CS)
    slider_ss = Slider(ax_ss, 'P-Norm SS', 0.0, 2.0, valinit=init_SS)
    slider_th = Slider(ax_th, 'CS Height Thresh', 0.0, 4.5, valinit=init_threshold)
    slider_cb_amp = Slider(ax_cb_amp, 'CB Amp Thresh', 0.0, 4.5, valinit=init_cb_amp_threshold, valstep=0.05)
    slider_cb_dur = Slider(ax_cb_dur, 'CB Duration (ms)', 5, 300, valinit=init_cb_duration_threshold, valstep=5)
    slider_cs_st = Slider(ax_cb_st, 'Simple thresh CS', 0.0, 10.0, valinit=init_threshold_CS)
    slider_ss_st = Slider(ax_ss_st, 'Simple thresh SS', 0.0, 10.0, valinit=init_threshold_SS)
    slider_ss_cap = Slider(ax_cb_cap, 'SS min height (norm)', 0.0, 4.0, valinit=inint_ss_cap, valstep=0.05)
    slider_fhp = Slider(ax_fhp, 'HP Freq (Hz)', 5.0, 40.0, valinit=init_f_hp, valstep=1.0)

    fig.text(
        0.12, 0.03, 'ISI Threshold (FIXED): 20 ms',
        fontsize=9, weight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    )

    def _call_detect_bursts_vm_local(trace_norm, bursts_final, all_s, p_cb_amp, p_cb_dur):
        import inspect
        kwargs = dict(
            highpass=1.0,
            median_window=11,
            cb_amp_threshold=p_cb_amp,
            cb_duration_threshold=p_cb_dur,
            isi_threshold_ms=20,
            baseline_subtract=True,
            baseline_window_ms=20,
            baseline_percentile=10,
            merge_SS_ms=20,
            merge_CB_ms=5,
            plateau_duration_threshold=plateau_duration_threshold,
            plateau_amp_threshold=plateau_amp_threshold,
            plateau_kernel_ms=plateau_kernel_ms,
            plateau_score_min_ms=plateau_score_min_ms,
            plotflag=False,
        )
        try:
            sig = inspect.signature(detect_bursts_from_vm)
            kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
        except Exception:
            pass

        ret = detect_bursts_from_vm(
            trace_norm,
            np.ones_like(trace_norm),
            bursts_final,
            all_s,
            frame_rate,
            **kwargs,
        )
        if not isinstance(ret, tuple):
            raise RuntimeError('detect_bursts_from_vm returned non-tuple output')
        if len(ret) < 7:
            raise RuntimeError(f'detect_bursts_from_vm returned {len(ret)} values, expected >=7')

        simple_spikes_vm = ret[0]
        complex_spikes_vm = ret[1]
        all_spikes_vm = ret[2]
        trace_snr = ret[3]
        vm = ret[4]
        burst_metrics = ret[5]
        complex_bursts_vm = ret[6]
        vm_plateaus_dict = ret[7] if len(ret) >= 8 else {}

        return (
            simple_spikes_vm,
            complex_spikes_vm,
            all_spikes_vm,
            trace_snr,
            vm,
            burst_metrics,
            complex_bursts_vm,
            vm_plateaus_dict,
        )

    def run_detection(
        p_cs, p_ss, p_th, p_cb_amp, p_cb_dur,
        p_ss_cap, p_st_ss, p_st_cs, p_f_hp,
        CB_detection_method, SS_detection_method
    ):
        try:
            # During tuning, keep simple mode
            CB_detection_method = "simple"
            SS_detection_method = "simple"

            c_bursts, _ = complex_bursts_detection(
                trace, spike_idx, frame_rate,
                pnorm=p_cs, process_window=30, plotflag=False,
                CB_detection_method=CB_detection_method,
                simple_threshold=p_st_cs
            )

            r_ss, t_noCS = refine_single_spikes(
                trace, spike_idx, c_bursts, frame_rate,
                process_window=30, pnorm=p_ss, f_hp=p_f_hp, min_spikes=5, plotflag=False,
                SS_detection_method=SS_detection_method,
                simple_threshold_SS=p_st_ss,
                SS_height_cap=None
            )

            trace_mf = c_bursts.get('trace_mf', median_filter(trace, size=11))
            heights, snr = spike_height_calculation2(r_ss, trace, trace_mf, t_noCS, frame_rate)

            height_scale = float(np.nanmedian(heights)) if np.size(heights) else 1.0
            height_scale = max(height_scale, 1e-12)
            trace_norm = trace / height_scale

            if p_ss_cap is not None:
                idx_ss = r_ss.astype(int)
                idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
                r_ss = idx_ss[trace_norm[idx_ss] >= float(p_ss_cap)]

            ss_cap_norm = float(p_ss_cap) if p_ss_cap is not None else np.nan

            cs_spikes_raw = detect_complex_spikes(
                trace_norm, c_bursts, np.ones_like(trace_norm),
                threshold=p_th, plotflag=False
            )

            c_bursts_final, r_ss_final, all_cs, all_s = refine_all_spikes(
                c_bursts, cs_spikes_raw, r_ss
            )

            (
                simple_spikes_vm, complex_spikes_vm, all_spikes_vm, trace_snr, vm, burst_metrics, complex_bursts_vm, vm_plateaus_dict
            ) = _call_detect_bursts_vm_local(
                trace_norm, c_bursts_final, all_s, p_cb_amp, p_cb_dur
            )

            return (
                c_bursts_final, r_ss_final, all_cs, complex_bursts_vm,
                simple_spikes_vm, complex_spikes_vm, ss_cap_norm, trace_norm,
                all_spikes_vm, burst_metrics, vm, heights, trace_snr, vm_plateaus_dict
            )

        except Exception as e:
            print(f"Error in run_detection: {e}")
            import traceback
            traceback.print_exc()
            trace_norm = np.asarray(trace, dtype=float)
            return (
                {}, np.array([], dtype=int), np.array([], dtype=int), {},
                np.array([], dtype=int), np.array([], dtype=int), float("nan"), trace_norm,
                np.array([], dtype=int), [], np.array([], dtype=float),
                np.array([], dtype=float), np.array([], dtype=float), {}
            )

    def update(val):
        p_cs = slider_cs.val
        p_ss = slider_ss.val
        p_th = float(slider_th.val)
        p_cb_amp = slider_cb_amp.val
        p_cb_dur = slider_cb_dur.val
        p_ss_cap = slider_ss_cap.val
        p_st_ss = slider_ss_st.val
        p_st_cs = slider_cs_st.val
        p_f_hp = slider_fhp.val

        selected_params.update({
            'pnorm_CS': p_cs,
            'pnorm_SS': p_ss,
            'threshold': p_th,
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'ST_cs': p_st_cs,
            'ST_ss': p_st_ss,
            'ss_cap': p_ss_cap,
            'f_hp': p_f_hp
        })

        try:
            (
                c_bursts, r_ss, all_cs, c_bursts_vm,
                simple_spikes_vm, complex_spikes_vm, ss_cap_norm,
                trace_norm, all_spikes_vm, burst_metrics, vm, heights, SNR, vm_plateaus_dict
            ) = run_detection(
                p_cs, p_ss, p_th, p_cb_amp, p_cb_dur,
                p_ss_cap, p_st_ss, p_st_cs, p_f_hp,
                CB_detection_method, SS_detection_method
            )

            final_spikes["all_spikes_vm"] = np.asarray(all_spikes_vm, dtype=int).copy()
            final_spikes["complex_spikes_vm"] = np.asarray(complex_spikes_vm, dtype=int).copy()
            final_spikes["simple_spikes_vm"] = np.asarray(simple_spikes_vm, dtype=int).copy()
            final_spikes["complex_burst_vm"] = c_bursts_vm
            final_spikes["vm"] = np.asarray(vm, dtype=float).copy()
            final_spikes["burst_metrics"] = burst_metrics.copy()
            final_spikes["heights"] = np.asarray(heights, dtype=float).copy()
            final_spikes["SNR"] = np.asarray(SNR, dtype=float).copy()
            final_spikes["vm_plateaus_dict"] = copy.deepcopy(vm_plateaus_dict)

            trace_norm = np.asarray(trace_norm, dtype=float).ravel()
            vm_norm = np.asarray(vm, dtype=float).ravel()
            if vm_norm.size != len(trace_norm):
                vm_aligned = np.full(len(trace_norm), np.nan, dtype=float)
                n_copy = min(len(trace_norm), vm_norm.size)
                if n_copy > 0:
                    vm_aligned[:n_copy] = vm_norm[:n_copy]
                vm_norm = vm_aligned
            idx_in = np.asarray(spike_idx, dtype=int)
            idx_in = idx_in[(idx_in >= 0) & (idx_in < len(trace_norm))]
            if idx_in.size > 0:
                scat_in.set_offsets(np.c_[t_axis[idx_in], trace_norm[idx_in]])
                scat_in.set_visible(True)
            else:
                scat_in.set_visible(False)

            idx_ss = np.array([], dtype=int)
            idx_cs = np.array([], dtype=int)
            idx_vm_simple = np.array([], dtype=int)
            idx_vm_complex = np.array([], dtype=int)
            starts = np.array([], dtype=int)
            ends = np.array([], dtype=int)

            n_ss = len(r_ss) if isinstance(r_ss, np.ndarray) else 0
            n_cs = len(all_cs) if isinstance(all_cs, np.ndarray) else 0
            n_vm_simple = len(simple_spikes_vm) if isinstance(simple_spikes_vm, np.ndarray) else 0
            n_vm_complex = len(complex_spikes_vm) if isinstance(complex_spikes_vm, np.ndarray) else 0
            n_bursts_vm = len(c_bursts_vm.get('starts', [])) if isinstance(c_bursts_vm, dict) else 0
            n_plateaus_vm = len(vm_plateaus_dict.get('starts', [])) if isinstance(vm_plateaus_dict, dict) else 0

            count_text.set_text(
                f"QIXIN: SS={n_ss} | CS={n_cs}\n"
                f"VM-BURST: Simple={n_vm_simple} | Complex={n_vm_complex} | Bursts={n_bursts_vm} | Plateaus={n_plateaus_vm}\n"
                f"CS={p_cs:.2f} | SS={p_ss:.2f} | T={p_th:.2f} | HP={p_f_hp:.1f}Hz\n"
                f"CB_Amp={p_cb_amp:.2f} | CB_Dur={p_cb_dur:.0f}ms | ISI=20ms (fixed)\n"
                f"SS_cap(norm)={p_ss_cap:.2f}"
            )

            trace_line.set_ydata(trace_norm)
            vm_line.set_ydata(vm_norm)
            ax.relim()
            ax.autoscale_view(scalex=False, scaley=True)
            ax_vm.relim()
            ax_vm.autoscale_view(scalex=False, scaley=True)

            if np.isfinite(ss_cap_norm):
                ss_cap_line.set_ydata([float(ss_cap_norm), float(ss_cap_norm)])
                ss_cap_line.set_visible(True)
            else:
                ss_cap_line.set_visible(False)

            if n_ss > 0:
                idx_ss = r_ss.astype(int)
                idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
                if len(idx_ss) > 0:
                    scat_ss.set_offsets(np.c_[t_axis[idx_ss], trace_norm[idx_ss]])
                    scat_ss.set_visible(True)
                else:
                    scat_ss.set_visible(False)
            else:
                scat_ss.set_visible(False)

            if n_cs > 0:
                idx_cs = all_cs.astype(int)
                idx_cs = idx_cs[(idx_cs >= 0) & (idx_cs < len(trace_norm))]
                if len(idx_cs) > 0:
                    scat_cs.set_offsets(np.c_[t_axis[idx_cs], trace_norm[idx_cs]])
                    scat_cs.set_visible(True)
                else:
                    scat_cs.set_visible(False)
            else:
                scat_cs.set_visible(False)

            if n_vm_simple > 0:
                idx_vm_simple = simple_spikes_vm.astype(int)
                idx_vm_simple = idx_vm_simple[(idx_vm_simple >= 0) & (idx_vm_simple < len(trace_norm))]
                if len(idx_vm_simple) > 0:
                    scat_vm_simple.set_offsets(np.c_[t_axis[idx_vm_simple], trace_norm[idx_vm_simple]])
                    scat_vm_simple.set_visible(True)
                else:
                    scat_vm_simple.set_visible(False)
            else:
                scat_vm_simple.set_visible(False)

            if n_vm_complex > 0:
                idx_vm_complex = complex_spikes_vm.astype(int)
                idx_vm_complex = idx_vm_complex[(idx_vm_complex >= 0) & (idx_vm_complex < len(trace_norm))]
                if len(idx_vm_complex) > 0:
                    scat_vm_complex.set_offsets(np.c_[t_axis[idx_vm_complex], trace_norm[idx_vm_complex]])
                    scat_vm_complex.set_visible(True)
                else:
                    scat_vm_complex.set_visible(False)
            else:
                scat_vm_complex.set_visible(False)

            for span in burst_spans:
                span.remove()
            burst_spans.clear()

            if isinstance(c_bursts_vm, dict):
                starts = c_bursts_vm.get('starts', np.array([]))
                ends = c_bursts_vm.get('ends', np.array([]))
                for s, e in zip(starts, ends):
                    span = ax.axvspan(s / frame_rate, e / frame_rate, color='yellow', alpha=0.3)
                    burst_spans.append(span)

            latest_state.clear()
            latest_state.update({
                'trace_norm': trace_norm,
                'vm_norm': vm_norm,
                'idx_in': idx_in,
                'idx_ss': idx_ss,
                'idx_cs': idx_cs,
                'idx_vm_simple': idx_vm_simple,
                'idx_vm_complex': idx_vm_complex,
                'burst_starts': starts,
                'burst_ends': ends,
                'p_ss_cap': float(p_ss_cap) if p_ss_cap is not None else None,
                'params': selected_params.copy(),
                'vm_plateaus_dict': copy.deepcopy(vm_plateaus_dict),
            })

            fig.canvas.draw_idle()

        except Exception as e:
            count_text.set_text(f"ERROR: {str(e)[:80]}")
            fig.canvas.draw_idle()

    slider_cs.on_changed(update)
    slider_ss.on_changed(update)
    slider_th.on_changed(update)
    slider_cb_amp.on_changed(update)
    slider_cb_dur.on_changed(update)
    slider_ss_cap.on_changed(update)
    slider_ss_st.on_changed(update)
    slider_cs_st.on_changed(update)
    slider_fhp.on_changed(update)

    update(None)

    def on_key(event):
        if event.key == 'enter':
            plt.close(fig)

    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.show(block=False)
    for _ in range(40000):
        if not plt.fignum_exists(fig.number):
            break
        plt.pause(0.015)
    else:
        print("  ⚠ Interactive timeout – using current parameter values")

    plt.close('all')
    save_latest_state_to_html()

    return (
        selected_params['pnorm_CS'],
        selected_params['pnorm_SS'],
        selected_params['threshold'],
        selected_params['cb_amp_threshold'],
        selected_params['cb_duration_threshold'],
        selected_params['ss_cap'],
        selected_params['ST_ss'],
        selected_params['ST_cs'],
        selected_params['f_hp'],
        final_spikes["all_spikes_vm"],
        final_spikes["complex_spikes_vm"],
        final_spikes["simple_spikes_vm"],
        final_spikes["complex_burst_vm"],
        final_spikes["vm"],
        final_spikes["burst_metrics"],
        final_spikes["heights"],
        final_spikes["SNR"],
        final_spikes["vm_plateaus_dict"]
    )






In [8]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
AllCalSR = list(r['CALsr'])
MotorSSVol = []
MotorSSCal = []
RestSSVol = []
RestSSCal = []
AwakeSSVol = []
AwakeSSCal = []
AnstSSVol = []
AnstSSCal = []
Rest_result = []
Motor_result = []
Anst_result = []
Awake_result = []
AllCellAllData= []



In [7]:
#remove frame
for i in range(len(pathPyr) -2,len(pathPyr)):
    
    l = pathPyr[i]
    print(l)
    parentP = os.path.dirname(l)
    calSR= AllCalSR[i]
    #print(l)
    TracePathSPIKE = os.path.join(l,'SpikeIdx.csv')
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    
    spikeId = pd.read_csv(TracePathSPIKE)
    spikeId = np.array(spikeId)
    spikeId = spikeId.flatten()
    spikeId = spikeId.tolist()
    motor_active = motor.any()
    motor = motor[0:np.size(trace_vol_full,0)]
    
    
    VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
    CalAX = np.linspace(0, (len(trace_cal_full)/calSR), len(trace_cal_full))
    N_Trace,N_TraceC,N_spikeId,N_VolAX,N_CalAX,N_motor,mask_Voltage,mask_calcium=remove_Frame_Multi(trace_vol_full,trace_cal_full,spikeId,VolAX,CalAX,motor)
    TracePathCal = os.path.join(l,'calMask.csv')
    TracePathVol = os.path.join(l,'volMask.csv')
    df = pd.DataFrame(mask_calcium, columns=['mask'])  # create df with column name
    df.to_csv(TracePathCal, index=False)
    df = pd.DataFrame(mask_Voltage, columns=['mask'])  # create df with column name
    df.to_csv(TracePathVol, index=False)


Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\RL2\07-01-2025-ans\fov2\cell0


NameError: name 'remove_Frame_Multi' is not defined

In [9]:
#run correction and split
for i in range(len(pathPyr)):
    
    
    l = pathPyr[i]
    print(l)
    calSR= AllCalSR[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    TracePathSpikeID = os.path.join(l,'SpikeIdx.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = trace_vol_full.copy().astype(float)
    TraceC = trace_cal_full.copy().astype(float)
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor.to_numpy().astype(float)
    motor = motor[:len(VolMask)]
    TraceV[~VolMask] = np.nan
    TraceC[~CalMask] = np.nan
    motor[~VolMask] = np.nan

    spikes = pd.read_csv(TracePathSpikeID)
    spikes = pd.to_numeric(spikes.iloc[:, 0], errors='coerce').dropna().astype(int).values
    valid_spikes = spikes[VolMask[spikes]]
    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))

    # path =os.path.join(l,r'cell__CS_detection.pkl')
    # # 2. Open the file
    # with open(path, 'rb') as f:
    #     data = pickle.load(f)
    
    # # 3. Retrieve the specific key
    # # This is currently a list of lists (e.g., [ [1, 5, 10], [2, 8, 20] ])
    # raw_spikes_structure = data['all_spikes']
    # long_all_spikes = []
    # if isinstance(raw_spikes_structure, list):
    #     for sublist in raw_spikes_structure:
    #         if isinstance(sublist, (list, np.ndarray)):
    #             long_all_spikes.extend(sublist)
    #         else:
    #             long_all_spikes.append(sublist)
    # else:
    #     long_all_spikes = list(raw_spikes_structure)
    # params_list = data['CS_detection_params']['selected_params_per_cell']
    # if len(params_list) > 0:
    #     prev_params = params_list[-1]   # last used params
    #     init_CS = prev_params['pnorm_CS']
    #     init_SS = prev_params['pnorm_SS']
    #     init_thresh = prev_params['threshold']
    # # Sort to ensure chronological order
    # long_all_spikes = sorted(list(set(long_all_spikes)))
    # long_all_spikes = sorted(long_all_spikes)
    if bsPyr[i].lower()=='motor':
        N = ''
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        # results = load_and_refine_pyr_spikes(
        #             path=l,trace_vol=TraceV,trace_cal=TraceC,
        #             frame_rate=500,
        #             interactive=True,  # Set to True for interactive tuning
        #             preW=75,
        #             postW=100,spikeID=valid_spikes,name=N

        #         )
        calMot,calRes, spikeMot,spikeRes,vmot,volR = Split_cal(changeP,TraceV,TraceC,VolAX,CalAX,motor,valid_spikes)
        
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        g = 0
        r = 0
        for k in range(len(MotorFR)):
            
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'm{k}'
            
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
                    INIT_CB_AMP = prev_params['cb_amp_threshold']
                    INIT_CB_DUR = prev_params['cb_duration_threshold']
                    init_thresh_SS = prev_params['simple_threshold_ss']
                    init_thresh_CS = prev_params['simple_threshold_cs']
                    init_thresh_ISI = prev_params['isi_threshold_ms']
            

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeMot[k].tolist()
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Motor_result.append(results)
            MotorSSVol.extend(results['vol_traces'])
            MotorSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
        for j in range(len(RestFR)):
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'r{j}'
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeRes[j].tolist()
            if len(trace_vol_full) < 10 * 500:
                print(f"SKIP {l} r{j}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            currSp = np.asarray(currSp, dtype=np.int64).ravel()
            currSp = currSp[np.isfinite(currSp)]
            currSp = currSp[(currSp >= 0) & (currSp < len(trace_vol_full))]

            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Rest_result.append(results)
            RestSSVol.extend(results['vol_traces'])
            RestSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
    elif bsPyr[i].lower()=='awake':
        N=''
        plt.close('all')
        spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
        spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
        if os.path.exists(spikePath) or os.path.exists(spikePath2):
            if os.path.exists(spikePath):
                with open(spikePath, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = spike_data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
            else:
                with open(spikePath2, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = spike_data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=TraceV,trace_cal=TraceC,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,params_pr=prev_params,name=N

                )
        Awake_result.append(results)
        AwakeSSVol.extend(results['vol_traces'])   
        AwakeSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)
    elif bsPyr[i].lower()=='anst':
        plt.close('all')
        spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
        spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
        if os.path.exists(spikePath) or os.path.exists(spikePath2):
            if os.path.exists(spikePath):
                with open(spikePath, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = spike_data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
            else:
                with open(spikePath2, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=TraceV,trace_cal=TraceC,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,params_pr=prev_params

                )
        Anst_result.append(results)
        AnstSSVol.extend(results['vol_traces'])   
        AnstSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)


Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0
Trace length: 59992 frames
Processing trace in 4 segments for CS detection
Segment 0: 0 to 15000 (15000 frames)
  Found 231 complex spikes in segment 0
Segment 1: 15000 to 30000 (15000 frames)
  Found 234 complex spikes in segment 1
Segment 2: 30000 to 45000 (15000 frames)
  Found 254 complex spikes in segment 2
Segment 3: 45000 to 59992 (14992 frames)
  Found 252 complex spikes in segment 3
Total complex spikes detected: 971
Trace length: 59992 frames
Processing trace in 4 segments
Segment 0: 0 to 15000 (15000 frames)
Segment 1: 15000 to 30000 (15000 frames)
Segment 2: 30000 to 45000 (15000 frames)
Segment 3: 45000 to 59992 (14992 frames)
Window size: 10000, Step size: 5000
Session 1 - Simple Spikes Linear slope: -0.001578
Session 1 - Simple Spikes Exp decay (b): 0.023051
Session 1 - Simple Spikes Tau (1/b): 43.38 seconds
Session 1 - Baseline Noise Linear slope: -0.000306
Session 1 - Baseline Noise Exp decay (b): 

KeyboardInterrupt: 

In [ ]:
#direct from basic
for i in range(82,len(pathPyr)):
    l = pathPyr[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    vol_mask = np.array(pd.read_csv(TracePathVolM)).flatten().astype(bool)
    cal_mask = np.array(pd.read_csv(TracePathCalM)).flatten().astype(bool)
    # Apply masks
    trace_vol = trace_vol_full[vol_mask]
    trace_cal = trace_cal_full[cal_mask]
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(trace_vol,0)]
    VolAX = np.linspace(0, (len(trace_vol)/500), len(trace_vol)) 
    CalAX = np.linspace(0, (len(trace_cal)/30), len(trace_cal))
    path =os.path.join(l,r'SpikeIdx.csv')
    IspikeId = pd.read_csv(path)
    IspikeId = np.array(IspikeId)
    IspikeId = IspikeId.flatten()
    IspikeId = IspikeId.tolist()
    VolMask = vol_mask.astype(bool)

    # 2. Create a mapping from Old Index -> New Index
    # This creates an array where every index contains the count of "True" frames before it
    new_mapping = np.cumsum(vol_mask) - 1 

    # 3. Filter and Map
    # We keep the spike IF the mask was True at that spot...
    # ...AND we convert it to the new index using our mapping.
    spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(vol_mask) and vol_mask[int(i)]]
    long_all_spikes = sorted(list(set(spikeId)))
    long_all_spikes = sorted(spikeId)
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        calMot,calRes, spikeMot,spikeRes,vmot,volR = Split_cal(changeP,trace_vol,trace_cal,VolAX,CalAX,motor,long_all_spikes)
        
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        g = 0
        r = 0
        for k in range(len(MotorFR)):
            
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'm{k}'
            r = r + 1
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
                    INIT_CB_AMP = prev_params['cb_amp_threshold']
                    INIT_CB_DUR = prev_params['cb_duration_threshold']
                    init_thresh_SS = prev_params['simple_threshold_ss']
                    init_thresh_CS = prev_params['simple_threshold_cs']
                    init_thresh_ISI = prev_params['isi_threshold_ms']
            

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeMot[k].tolist()
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Motor_result.append(results)
            MotorSSVol.extend(results['vol_traces'])
            MotorSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
        for j in range(len(RestFR)):
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'r{j}'
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeRes[j].tolist()
            if len(trace_vol_full) < 10 * 500:
                print(f"SKIP {l} r{j}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Rest_result.append(results)
            RestSSVol.extend(results['vol_traces'])
            RestSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
    elif bsPyr[i].lower()=='awake':
        N=''
        plt.close('all')
        # spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
        # spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
        # if os.path.exists(spikePath) or os.path.exists(spikePath2):
        #     if os.path.exists(spikePath):
        #         with open(spikePath, 'rb') as f:
        #             spike_data = pickle.load(f)
        #         long_all_spikes = spike_data['vm_all_spikes']
        #         params_list = spike_data['detection_params']
        #         if len(params_list) > 0:
        #             prev_params = params_list
        #             init_CS = prev_params['pnorm_CS']
        #             init_SS = prev_params['pnorm_SS']
        #             init_thresh = prev_params['threshold']
        #     else:
        #         with open(spikePath2, 'rb') as f:
        #             spike_data = pickle.load(f)
        #         long_all_spikes = spike_data['vm_all_spikes']
        #         params_list = spike_data['detection_params']
        #         if len(params_list) > 0:
        #             prev_params = params_list
        #             init_CS = prev_params['pnorm_CS']
        #             init_SS = prev_params['pnorm_SS']
        #             init_thresh = prev_params['threshold']
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol,trace_cal=trace_cal,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,name=N

                )
        Awake_result.append(results)
        AwakeSSVol.extend(results['vol_traces'])   
        AwakeSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)
    elif bsPyr[i].lower()=='anst':
        N=''
        plt.close('all')
        
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol,trace_cal=trace_cal,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,name=N

                )
        Anst_result.append(results)
        AnstSSVol.extend(results['vol_traces'])   
        AnstSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)


In [6]:
#sst

DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\SST_F.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
AllCalSR = r['CALsr']
MotorSSVol = []
MotorSSCal = []
RestSSVol = []
RestSSCal = []
AwakeSSVol = []
AwakeSSCal = []
AnstSSVol = []
AnstSSCal = []
Rest_result = []
Motor_result = []
Anst_result = []
Awake_result = []
AllCellAllData= []


In [7]:
import numpy as np
import pickle
from pathlib import Path

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def robust_z(x):
    x = np.asarray(x, float).ravel()
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    return (x - med) / (1.4826 * mad)

def compute_z_proxy_from_suite2p_ops(
    ops,
    use_vcorr=True,
    upsample_vcorr_if_needed=True,
    w_xy=0.2,
    w_corrxy=0.4,
    w_vcorr=0.4,
):
    xoff = np.asarray(ops.get("xoff", ops.get("xoff1", [])), float).ravel()
    yoff = np.asarray(ops.get("yoff", ops.get("yoff1", [])), float).ravel()
    if xoff.size == 0 or yoff.size == 0:
        raise KeyError("ops missing xoff/yoff.")

    n = min(xoff.size, yoff.size)
    motion_xy = np.sqrt(xoff[:n]**2 + yoff[:n]**2)
    xy_motion_z = robust_z(motion_xy)

    corrXY = ops.get("corrXY", ops.get("corrXY1", None))
    if corrXY is None:
        raise KeyError("ops missing corrXY.")
    corrXY_bad_z = robust_z(-np.asarray(corrXY, float).ravel()[:n])

    z_proxy = w_xy * xy_motion_z + w_corrxy * corrXY_bad_z
    weight_sum = w_xy + w_corrxy

    comps = {
        "xy_motion_z": xy_motion_z,
        "corrXY_bad_z": corrXY_bad_z,
    }

    if use_vcorr and "Vcorr" in ops:
        Vcorr = np.asarray(ops["Vcorr"], float).ravel()
        if Vcorr.size == n:
            vc = robust_z(-Vcorr)
        elif upsample_vcorr_if_needed and Vcorr.size > 2:
            x_old = np.linspace(0, n - 1, Vcorr.size)
            x_new = np.arange(n)
            vc = robust_z(-np.interp(x_new, x_old, Vcorr))
        else:
            vc = None

        if vc is not None:
            z_proxy += w_vcorr * vc
            weight_sum += w_vcorr
            comps["Vcorr_bad_z"] = vc

    z_proxy /= (weight_sum + 1e-12)
    return z_proxy, comps

def pick_bad_frames(z_proxy, method="mad", k=6.0, q=99.0):
    if method == "mad":
        med = np.nanmedian(z_proxy)
        mad = np.nanmedian(np.abs(z_proxy - med)) + 1e-12
        thr = med + k * mad
    else:
        thr = np.nanpercentile(z_proxy, q)

    bad = np.isfinite(z_proxy) & (z_proxy > thr)
    return bad, thr

def cal_bad_to_vol_bad_mask(bad_cal_mask, VolXaX, CalXax):
    bad_v = np.zeros(len(VolXaX), dtype=bool)
    dt_ca = np.median(np.diff(CalXax)) if len(CalXax) > 1 else 1.0

    bad_idx = np.where(bad_cal_mask)[0]
    for ci in bad_idx:
        t0 = CalXax[ci]
        t1 = CalXax[ci + 1] if ci < len(CalXax) - 1 else t0 + dt_ca
        bad_v |= (VolXaX >= t0) & (VolXaX < t1)

    return bad_v

# ---------------------------------------------------------
# Main Pipeline
# ---------------------------------------------------------
def remove_high_z_motion_both_modalities(
    cal_trace,
    vol_trace,
    *,
    ops,
    VolXaX,
    CalXax,
    mode="mask",
    thr_method="mad",
    thr_k=6.0,
    thr_q=99.0,
    pad_ca_frames=0,
    save_pickle_path=None,
):

    cal = np.asarray(cal_trace, float).ravel()
    vol = np.asarray(vol_trace, float).ravel()

    z_proxy, comps = compute_z_proxy_from_suite2p_ops(ops)

    n = min(len(cal), len(z_proxy))
    z_use = z_proxy[:n]

    bad_ca, thr = pick_bad_frames(
        z_use, method=thr_method, k=thr_k, q=thr_q
    )

    # Expand bad frames
    if pad_ca_frames > 0:
        expanded = bad_ca.copy()
        idx = np.where(bad_ca)[0]
        for i in idx:
            a = max(0, i - pad_ca_frames)
            b = min(n, i + pad_ca_frames + 1)
            expanded[a:b] = True
        bad_ca = expanded

    bad_v = cal_bad_to_vol_bad_mask(
        bad_ca,
        VolXaX=np.asarray(VolXaX),
        CalXax=np.asarray(CalXax),
    )

    if mode == "mask":
        cal_out = cal.copy()
        vol_out = vol.copy()
        cal_out[:n][bad_ca] = np.nan
        vol_out[bad_v] = np.nan
    else:
        keep_ca = np.ones_like(cal, dtype=bool)
        keep_ca[:n] = ~bad_ca
        cal_out = cal[keep_ca]
        vol_out = vol[~bad_v]

    out = {
        "cal_out": cal_out,
        "vol_out": vol_out,
        "bad_ca_mask": bad_ca,
        "bad_v_mask": bad_v,
        "bad_ca_idx": np.where(bad_ca)[0],
        "bad_v_idx": np.where(bad_v)[0],
        "z_proxy": z_use,
        "z_components": comps,
        "threshold": thr,
        "params": {
            "mode": mode,
            "thr_method": thr_method,
            "thr_k": thr_k,
            "thr_q": thr_q,
            "pad_ca_frames": pad_ca_frames,
        }
    }

    # -------------------------
    # Save pickle automatically
    # -------------------------
    if save_pickle_path is not None:
        save_path = Path(save_pickle_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, "wb") as f:
            pickle.dump(out, f)
        print("Saved z-cleaning output to:", save_path)

    return out



In [9]:
import plotly.io as pio
pio.renderers.default = "browser"



In [8]:
import numpy as np
import plotly.graph_objects as go

def plot_z_proxy_threshold(out, *, CalXax=None, title="Z-shift proxy per frame (Suite2p)"):
    """
    out: dict returned by remove_high_z_motion_both_modalities(...)
    CalXax: optional time axis in seconds (len >= n_used). If None, uses frame index.
    """
    z = np.asarray(out["z_proxy"], float).ravel()
    thr = float(out["threshold"])
    bad = np.asarray(out["bad_ca_mask"], bool).ravel()

    # Align lengths (out["bad_ca_mask"] may be length n_used or full length)
    n = z.size
    bad = bad[:n] if bad.size >= n else np.pad(bad, (0, n - bad.size), constant_values=False)

    if CalXax is not None:
        x = np.asarray(CalXax, float).ravel()[:n]
        xlab = "Time (s)"
    else:
        x = np.arange(n)
        xlab = "Frame"

    fig = go.Figure()

    # z_proxy line
    fig.add_trace(go.Scatter(
        x=x, y=z, mode="lines",
        name="z_proxy (combined)"
    ))

    # threshold line
    fig.add_trace(go.Scatter(
        x=x, y=np.full(n, thr),
        mode="lines",
        name=f"threshold = {thr:.3g}",
        line=dict(dash="dash")
    ))

    # bad frames markers
    if bad.any():
        fig.add_trace(go.Scatter(
            x=x[bad], y=z[bad],
            mode="markers",
            name=f"bad frames ({bad.sum()})",
            marker=dict(size=6, symbol="x")
        ))

    # Optional component traces
    comps = out.get("z_components", {})
    for k, v in comps.items():
        v = np.asarray(v, float).ravel()
        if v.size == n:
            fig.add_trace(go.Scatter(
                x=x, y=v,
                mode="lines",
                name=k,
                opacity=0.45
            ))

    fig.update_layout(
        title=title,
        xaxis_title=xlab,
        yaxis_title="Proxy / robust z-score",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )

    fig.show()
    return fig



In [17]:
#SST
MotorCalTran = []
MotorFrTran = []
RestCalTran = []
RestFrTran = []
AwakeCalTran = []
AwakeFrTran = []
AnstCalTran = []
AnstFrTran = []
MotorPears = []
MotorP_val = []
RestPears = []
RestP_val = []
AwakePears = []
AwakeP_val = []
AnstPears = []
AnstP_val = []
MotorCalDiff = []
MotorFRDiff = []
RestCalDiff = []
RestFRDiff = []
AwakeCalDiff = []
AwakeFRDiff = []
AnstCalDiff = []
AnstFRDiff = []
MotorOpt_r = []
MotorOpt_lag = []
RestOpt_r = []
RestOpt_lag = []
AwakeOpt_r = []
AwakeOpt_lag = []
AnstOpt_r = []
AnstOpt_lag = []
MotorWindow_r = []
MotorWindow_p = []
RestWindow_r = []
RestWindow_p = []
AwakeWindow_r = []
AwakeWindow_p = []
AnstWindow_r = []
AnstWindow_p = []
MotorFRMean = []
RestFRMean=[]
AwakeFRMean = []
AnstFRMean = []
MotorFRdiffMean = []
RestFRdiffMean=[]
AwakeFRdiffMean = []
AnstFRdiffMean = []
AllPath = []

calSR = 30
volSR =500
corr_summary_rows = []
for i,l in enumerate(pathPyr):
    calSR= AllCalSR[i]
    print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace[VolMask]
    TraceC = CalTrace[CalMask]
    motor= motor[VolMask]
    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))
    fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    if os.path.exists(fpath):
        pathSpike = fpath
        spikeId = pd.read_csv(pathSpike)
        spikeId = np.array(spikeId)
        spikeId = spikeId.flatten()
        spikeId = spikeId.tolist()
    else:
        pathSpike = os.path.join(l,r'SpikeIdx.csv')
        print('LOL')
        IspikeId = pd.read_csv(pathSpike)
        IspikeId = np.array(IspikeId)
        IspikeId = IspikeId.flatten()
        IspikeId = IspikeId.tolist()
        VolMask = VolMask.astype(bool)

        # 2. Create a mapping from Old Index -> New Index
        # This creates an array where every index contains the count of "True" frames before it
        new_mapping = np.cumsum(VolMask) - 1 

        # 3. Filter and Map
        # We keep the spike IF the mask was True at that spot...
        # ...AND we convert it to the new index using our mapping.
        spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    if bsPyr[i].lower()=='motor':
        N =''
        AllPath.append(l)
        spk = sst_spike_correction_gui(TraceV, fs=500, save_dir=l,name = N,chunk_s=30)
    #     TracePathCal = os.path.join(l,'changepoint.csv')
    #     changeP = pd.read_csv(TracePathCal)
    #     changeP = np.array(changeP)
    #     changeP = np.array(changeP).flatten().tolist()
    #     #calMot,calRes = Split_cal(changeP,TraceV,TraceC,VolAX,CalAX,motor)
    #     MotorSinglePikeIDX = []
    #     RestSinglePikeIDX = []
    #     MotorSinglePikeCalIDX = []
    #     RestSinglePikeCalIDX = []
    #     MotorBurstSpikeIDX = []
    #     RestBurstSpikeIDX = []
    #     MotorComplexSpikeIDX = []
    #     RestComplexSpikeIDX = []
    #     RestFRpath =os.path.join(l,r'FiringRateRest.csv')
    #     MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        
    #     dfMot = pd.read_csv(MotorFRpath)
    #     MotorFR = dfMot['fr'].tolist()
    #     dfRest = pd.read_csv(RestFRpath)
    #     RestFR = dfRest['fr'].tolist()
    #     for k in range(len(MotorFR)):
            
    #         N = f'm{k}'
    #         TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
    #         TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
    #         VolTrace = pd.read_csv(TracePathVol)
    #         VolTrace = np.array(VolTrace)
    #         VolTrace = VolTrace.flatten()
    #         VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
    #         CalTrace = pd.read_csv(TracePathCal)
    #         CalTrace = np.array(CalTrace)
    #         CalTrace = CalTrace.flatten()
    #         CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
    #         CorrSPPath =  os.path.join(l,f'spikeTraceMot{k}.csv')
    #         spikeId = pd.read_csv(CorrSPPath)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #         txtPss = os.path.join(l,'SinglS_par.txt')
    #         spk = sst_spike_correction_gui(VolTrace, fs=500, save_dir=l,name = N)

    #     for j in range(len(RestFR)):
            
    #         N = f'r{j}'
    #         TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
    #         TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
    #         VolTrace = pd.read_csv(TracePathVol)
    #         VolTrace = np.array(VolTrace)
    #         VolTrace = VolTrace.flatten()
    #         VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
    #         CalTrace = pd.read_csv(TracePathCal)
    #         CalTrace = np.array(CalTrace)
    #         CalTrace = CalTrace.flatten()
    #         CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
    #         CorrSPPath =  os.path.join(l,f'spikeTraceMot{j}.csv')
    #         spikeId = pd.read_csv(CorrSPPath)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #         txtPss = os.path.join(l,'SinglS_par.txt')
    #         spk = sst_spike_correction_gui(VolTrace, fs=500, save_dir=l,name = N)
    # elif bsPyr[i].lower()=='awake':
    #     N = ''
    #     AllPath.append(l)
    #     fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    #     if os.path.exists(fpath):
    #         pathSpike = fpath
    #         spikeId = pd.read_csv(pathSpike)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #     else:
    #         pathSpike = os.path.join(l,r'SpikeIdx.csv')
    #         print('LOL')
    #         IspikeId = pd.read_csv(pathSpike)
    #         IspikeId = np.array(IspikeId)
    #         IspikeId = IspikeId.flatten()
    #         IspikeId = IspikeId.tolist()
    #         VolMask = VolMask.astype(bool)

    #         # 2. Create a mapping from Old Index -> New Index
    #         # This creates an array where every index contains the count of "True" frames before it
    #         new_mapping = np.cumsum(VolMask) - 1 

    #         # 3. Filter and Map
    #         # We keep the spike IF the mask was True at that spot...
    #         # ...AND we convert it to the new index using our mapping.
    #         spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    #     txtPss = os.path.join(l,'SinglS_par.txt')
    #     spk = sst_spike_correction_gui(TraceV, fs=500, save_dir=l,name = N,chunk_s=30)
    # elif bsPyr[i].lower()=='anst':
    #     N = ''
    #     AllPath.append(l)
    #     fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    #     if os.path.exists(fpath):
    #         pathSpike = fpath
    #         spikeId = pd.read_csv(pathSpike)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #     else:
    #         pathSpike = os.path.join(l,r'SpikeIdx.csv')
    #         print('LOL')
    #         IspikeId = pd.read_csv(pathSpike)
    #         IspikeId = np.array(IspikeId)
    #         IspikeId = IspikeId.flatten()
    #         IspikeId = IspikeId.tolist()
    #         VolMask = VolMask.astype(bool)

    #         # 2. Create a mapping from Old Index -> New Index
    #         # This creates an array where every index contains the count of "True" frames before it
    #         new_mapping = np.cumsum(VolMask) - 1 

    #         # 3. Filter and Map
    #         # We keep the spike IF the mask was True at that spot...
    #         # ...AND we convert it to the new index using our mapping.
    #         spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    #     spk = sst_spike_correction_gui(TraceV, fs=500, save_dir=l,name = N,chunk_s=50)



Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc17\Xb\17-06-2025\fov1\cell1
LOL


KeyboardInterrupt: 

In [7]:
# RAW_THRESHOLD_STAGE_V1
import os
import glob
import copy
import pickle
import numpy as np
import pandas as pd

def _remove_complex_regions_linear(trace, starts, ends):
    trace = np.asarray(trace, dtype=float).copy()
    n = len(trace)
    if starts is None or ends is None:
        return trace

    for s, e in zip(np.asarray(starts, dtype=int), np.asarray(ends, dtype=int)):
        if n == 0:
            break
        s = max(0, min(int(s), n - 1))
        e = max(0, min(int(e), n - 1))
        if e < s:
            s, e = e, s

        left = max(0, s - 1)
        right = min(n - 1, e + 1)
        if right <= left:
            trace[s:e + 1] = np.nan
            continue

        y0 = trace[left]
        y1 = trace[right]
        if not np.isfinite(y0) or not np.isfinite(y1):
            trace[s:e + 1] = np.nan
            continue

        interp_x = np.arange(s, e + 1)
        trace[interp_x] = y0 + (y1 - y0) * (interp_x - left) / (right - left)

    return trace

def _local_amp(trace, idx, pre_samples):
    idx = int(idx)
    lo = max(0, idx - pre_samples)
    baseline_win = trace[lo:idx]
    if baseline_win.size == 0:
        baseline = np.nan
    else:
        baseline = np.nanmedian(baseline_win)

    if not np.isfinite(baseline):
        baseline = 0.0
    return float(trace[idx] - baseline)

def _simple_bursts_from_spikes(simple_spikes, frame_rate, isi_ms=20):
    spikes = np.asarray(simple_spikes, dtype=int)
    if spikes.size == 0:
        return []

    spikes = np.sort(np.unique(spikes))
    isi_frames = max(1, int(round((isi_ms / 1000.0) * frame_rate)))

    bursts = []
    start = spikes[0]
    prev = spikes[0]
    count = 1

    for s in spikes[1:]:
        if (s - prev) <= isi_frames:
            prev = s
            count += 1
        else:
            bursts.append({'start': int(start), 'end': int(prev), 'n_spikes': int(count)})
            start = s
            prev = s
            count = 1
    bursts.append({'start': int(start), 'end': int(prev), 'n_spikes': int(count)})
    return bursts

def refine_simple_spikes_with_raw_threshold(
    final_dict,
    frame_rate=500,
    patch_sec=30,
    base_k=3.0,
    k_min=1.5,
    k_max=8.0,
    amp_quantile=0.25,
    baseline_ms=6.0,
    isi_ms_for_simple_burst=20,
):
    """
    Raw-threshold stage applied ONLY to final simple spikes/bursts.
    Complex spikes and complex burst dict stay unchanged.
    """
    out = copy.deepcopy(final_dict)

    trace = np.asarray(out.get('trace_vol', []), dtype=float)
    simple_spikes_prev = np.asarray(out.get('vm_simple_spikes', []), dtype=int)
    complex_spikes = np.asarray(out.get('vm_complex_spikes', []), dtype=int)

    burst_dict = out.get('vm_burst_dict', {}) if isinstance(out.get('vm_burst_dict', {}), dict) else {}
    cs_starts = burst_dict.get('starts', [])
    cs_ends = burst_dict.get('ends', [])

    trace_no_cs = _remove_complex_regions_linear(trace, cs_starts, cs_ends)

    patch_frames = max(1, int(round(patch_sec * frame_rate)))
    pre_samples = max(1, int(round((baseline_ms / 1000.0) * frame_rate)))

    refined_simple = []
    patch_rows = []

    n = len(trace_no_cs)
    for p0 in range(0, n, patch_frames):
        p1 = min(n, p0 + patch_frames)
        patch_trace = trace_no_cs[p0:p1]

        sigma = float(np.nanstd(patch_trace))
        if (not np.isfinite(sigma)) or sigma <= 0:
            sigma = 1e-9

        patch_spikes = simple_spikes_prev[(simple_spikes_prev >= p0) & (simple_spikes_prev < p1)]

        if patch_spikes.size > 0:
            amps = np.array([_local_amp(trace_no_cs, s, pre_samples) for s in patch_spikes], dtype=float)
            amps = amps[np.isfinite(amps)]
            if amps.size > 0:
                k_patch = float(np.clip(np.nanquantile(amps, amp_quantile) / sigma, k_min, k_max))
            else:
                k_patch = float(base_k)
        else:
            k_patch = float(base_k)

        thr = k_patch * sigma

        keep = []
        for s in patch_spikes:
            amp = _local_amp(trace_no_cs, s, pre_samples)
            if np.isfinite(amp) and amp >= thr:
                keep.append(int(s))
        refined_simple.extend(keep)

        patch_rows.append({
            'start_frame': int(p0),
            'end_frame': int(p1),
            'sigma': float(sigma),
            'k': float(k_patch),
            'threshold': float(thr),
            'n_simple_in': int(patch_spikes.size),
            'n_simple_kept': int(len(keep)),
        })

    refined_simple = np.asarray(sorted(set(refined_simple)), dtype=int)
    all_spikes_new = np.asarray(sorted(set(refined_simple.tolist() + complex_spikes.tolist())), dtype=int)

    out['vm_simple_spikes_before_raw_threshold'] = simple_spikes_prev
    out['vm_simple_spikes'] = refined_simple
    out['vm_all_spikes'] = all_spikes_new
    out['trace_no_complex_for_raw_threshold'] = trace_no_cs
    out['simple_bursts'] = _simple_bursts_from_spikes(refined_simple, frame_rate, isi_ms=isi_ms_for_simple_burst)
    # Preserve plateau classifications unchanged in this stage.
    if 'vm_plateaus_dict' in final_dict and 'vm_plateaus_dict' not in out:
        out['vm_plateaus_dict'] = copy.deepcopy(final_dict.get('vm_plateaus_dict', {}))

    out['raw_threshold_stage'] = {
        'enabled': True,
        'patch_sec': float(patch_sec),
        'base_k': float(base_k),
        'k_min': float(k_min),
        'k_max': float(k_max),
        'amp_quantile': float(amp_quantile),
        'baseline_ms': float(baseline_ms),
        'isi_ms_for_simple_burst': float(isi_ms_for_simple_burst),
    }

    patch_df = pd.DataFrame(patch_rows)
    return out, patch_df



In [27]:
# RAW_THRESHOLD_STAGE_V1 - loop over final dict files
# This updates ONLY final simple spikes/bursts; complex spikes are kept unchanged.
# Motor-state cells are supported (m0/m1/r0/r1): all matching PKLs are processed.

import os, glob, re, pickle, copy
import numpy as np

# Option A: set explicit files (or folders)
final_dict_paths =  pathPyr
# Examples:
# final_dict_paths = [
#     r"Z:\...\cell0\spike_detection_refined_new.pkl",
#     r"Z:\...\cell1",  # folder allowed
# ]

# Option B: auto-search from a root (uncomment)
# auto_search_root = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin"
# final_dict_paths = glob.glob(os.path.join(auto_search_root, "**", "spike_detection_refined_new*.pkl"), recursive=True)


def _natural_key_for_new_pkl(path):
    name = os.path.basename(path)
    m = re.search(r"spike_detection_refined_new(.*?)\.pkl$", name, flags=re.IGNORECASE)
    if not m:
        return (name.lower(), 0)
    suffix = m.group(1)
    if suffix == "":
        return ("", -1)
    try:
        return ("", int(suffix))
    except ValueError:
        # Handles m0/r1/etc. in stable lexical order
        return (suffix.lower(), 10**9)


def _resolve_final_pkl_paths(paths):
    resolved = []

    for item in paths:
        if item is None:
            continue
        item = str(item)

        # If folder was given, collect all new*.pkl files
        if os.path.isdir(item):
            resolved.extend(glob.glob(os.path.join(item, 'spike_detection_refined_new*.pkl')))
            continue

        # Existing explicit file path
        if os.path.isfile(item):
            resolved.append(item)
            continue

        # Missing canonical base: try motor variants in same folder
        folder = os.path.dirname(item)
        name = os.path.basename(item)
        if name.lower() == 'spike_detection_refined_new.pkl' and os.path.isdir(folder):
            hits = glob.glob(os.path.join(folder, 'spike_detection_refined_new*.pkl'))
            if len(hits) > 0:
                resolved.extend(hits)
                continue

        # Fallback: if item has wildcard-like intent, try direct glob
        hits = glob.glob(item)
        if len(hits) > 0:
            resolved.extend(hits)
            continue

        print(f"[RAW-TH][WARN] not found, skipping: {item}")

    resolved = [p for p in resolved if os.path.isfile(p)]
    resolved = list(dict.fromkeys(resolved))
    resolved = sorted(
        resolved,
        key=lambda pp: (_natural_key_for_new_pkl(pp), os.path.dirname(pp).lower(), os.path.basename(pp).lower())
    )
    return resolved


PATCH_SEC = 30
BASE_K = 3.0
K_MIN = 1.5
K_MAX = 8.0

resolved_paths = _resolve_final_pkl_paths(final_dict_paths)
if len(resolved_paths) == 0:
    raise FileNotFoundError('No spike_detection_refined_new*.pkl files resolved from final_dict_paths')

print(f"[RAW-TH] input items={len(final_dict_paths)} | resolved pkls={len(resolved_paths)}")

for pkl_path in resolved_paths:
    with open(pkl_path, 'rb') as f:
        final_dict = pickle.load(f)

    fr = final_dict.get('detection_params', {}).get('frame_rate', 500)
    complex_before = np.asarray(final_dict.get('vm_complex_spikes', []), dtype=int)
    simple_before = np.asarray(final_dict.get('vm_simple_spikes', []), dtype=int)
    plateaus_before = copy.deepcopy(final_dict.get('vm_plateaus_dict', {}))

    updated_dict, patch_df = refine_simple_spikes_with_raw_threshold(
        final_dict,
        frame_rate=fr,
        patch_sec=PATCH_SEC,
        base_k=BASE_K,
        k_min=K_MIN,
        k_max=K_MAX,
    )

    # Ensure complex spikes are unchanged
    complex_after = np.asarray(updated_dict.get('vm_complex_spikes', []), dtype=int)
    if not np.array_equal(complex_before, complex_after):
        raise RuntimeError(f'Complex spikes changed unexpectedly for: {pkl_path}')

    # Ensure plateau payload is unchanged
    plateaus_after = copy.deepcopy(updated_dict.get('vm_plateaus_dict', {}))
    if pickle.dumps(plateaus_before, protocol=pickle.HIGHEST_PROTOCOL) != pickle.dumps(plateaus_after, protocol=pickle.HIGHEST_PROTOCOL):
        raise RuntimeError(f'Plateau dict changed unexpectedly for: {pkl_path}')

    # Save back to same file name/path
    with open(pkl_path, 'wb') as f:
        pickle.dump(updated_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    stage_csv = os.path.splitext(pkl_path)[0] + '_raw_threshold_stage.csv'
    patch_df.to_csv(stage_csv, index=False)

    simple_after = np.asarray(updated_dict.get('vm_simple_spikes', []), dtype=int)
    print(
        f"[RAW-TH] {pkl_path} | FR={fr} | simple: {len(simple_before)} -> {len(simple_after)} | complex kept: {len(complex_after)}"
    )



[RAW-TH] input items=30 | resolved pkls=39
[RAW-TH] Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov6\cell0\spike_detection_refined_new.pkl | FR=500 | simple: 104 -> 77 | complex kept: 285
[RAW-TH] Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl | FR=500 | simple: 49 -> 36 | complex kept: 131
[RAW-TH] Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov2\cell1\spike_detection_refined_new.pkl | FR=500 | simple: 129 -> 96 | complex kept: 271
[RAW-TH] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\l\20-08-25-ans\cell0\spike_detection_refined_new.pkl | FR=500 | simple: 157 -> 117 | complex kept: 84
[RAW-TH] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\FOV1\cell4\spike_detection_refined_new.pkl | FR=500 | simple: 115 -> 85 | complex kept: 128
[RAW-TH] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\FOV1\cell5\spike_detection_refined_new.pkl | FR=500 | simple: 156 -> 116 | complex kept: 558
[RAW-TH] Z

In [ ]:
# Legacy scratch cell (kept as a safe template).
# Use this only if you want to manually inspect detect_bursts_from_vm outputs.
if False:
    ret = detect_bursts_from_vm(
        trace_idx,
        np.ones_like(trace_idx),
        complex_bursts_dict,
        all_spikes,
        fr,
        highpass=highpass,
        median_window=median_window,
        cb_amp_threshold=cb_amp_threshold,
        cb_duration_threshold=cb_duration_threshold,
        isi_threshold_ms=20,
        baseline_subtract=False,
        baseline_window_ms=20,
        baseline_percentile=None,
        merge_SS_ms=None,
        merge_CB_ms=None,
        plateau_amp_threshold=0.8,
        plateau_duration_threshold=100,
        plateau_kernel_ms=100,
        plateau_score_min_ms=80,
        plotflag=False,
    )
    # ret[7] is vm_plateaus_dict when available.


In [12]:
# RAW_THRESHOLD_GUI_V1 (chunk-wise)
import copy
import os
import glob
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button


def _sanitize_frame_rate(fr, default=500.0):
    try:
        fr = float(fr)
    except Exception:
        return float(default)
    # Guard against invalid/implausible values that create tiny/blank chunks
    if (not np.isfinite(fr)) or fr < 10 or fr > 5000:
        return float(default)
    return float(fr)
def _remove_complex_regions_linear_v2(trace, starts, ends):
    trace = np.asarray(trace, dtype=float).copy()
    n = len(trace)
    if starts is None or ends is None:
        return trace

    for s, e in zip(np.asarray(starts, dtype=int), np.asarray(ends, dtype=int)):
        if n == 0:
            break
        s = max(0, min(int(s), n - 1))
        e = max(0, min(int(e), n - 1))
        if e < s:
            s, e = e, s

        left = max(0, s - 1)
        right = min(n - 1, e + 1)
        if right <= left:
            trace[s:e + 1] = np.nan
            continue

        y0 = trace[left]
        y1 = trace[right]
        if not np.isfinite(y0) or not np.isfinite(y1):
            trace[s:e + 1] = np.nan
            continue

        interp_x = np.arange(s, e + 1)
        trace[interp_x] = y0 + (y1 - y0) * (interp_x - left) / (right - left)

    return trace
def _local_amp_v2(trace_no_cs, idx, pre_samples):
    idx = int(idx)
    lo = max(0, idx - pre_samples)
    baseline_win = trace_no_cs[lo:idx]
    baseline = np.nanmedian(baseline_win) if baseline_win.size > 0 else np.nan
    if not np.isfinite(baseline):
        baseline = 0.0
    return float(trace_no_cs[idx] - baseline)
def _simple_bursts_from_spikes_v2(simple_spikes, frame_rate, isi_ms=20):
    spikes = np.asarray(simple_spikes, dtype=int)
    if spikes.size == 0:
        return []
    spikes = np.sort(np.unique(spikes))
    isi_frames = max(1, int(round((isi_ms / 1000.0) * frame_rate)))

    bursts = []
    start = spikes[0]
    prev = spikes[0]
    count = 1
    for s in spikes[1:]:
        if (s - prev) <= isi_frames:
            prev = s
            count += 1
        else:
            bursts.append({'start': int(start), 'end': int(prev), 'n_spikes': int(count)})
            start = s
            prev = s
            count = 1
    bursts.append({'start': int(start), 'end': int(prev), 'n_spikes': int(count)})
    return bursts
def _extract_simple_burst_spikes_v2(final_dict):
    out = []
    for key in ('simple_bursts', 'vm_simple_bursts', 'regular_bursts'):
        obj = final_dict.get(key, None)
        if obj is None:
            continue

        if isinstance(obj, dict):
            for kk in ('spikes', 'indices', 'spike_indices', 'burst_spikes'):
                if kk in obj:
                    try:
                        out.extend(np.asarray(obj[kk], dtype=int).ravel().tolist())
                    except Exception:
                        pass
        elif isinstance(obj, (list, tuple)):
            for item in obj:
                if isinstance(item, dict):
                    for kk in ('spikes', 'indices', 'spike_indices', 'burst_spikes'):
                        if kk in item:
                            try:
                                out.extend(np.asarray(item[kk], dtype=int).ravel().tolist())
                            except Exception:
                                pass
                else:
                    try:
                        out.extend(np.asarray(item, dtype=int).ravel().tolist())
                    except Exception:
                        pass

    if len(out) == 0:
        return np.array([], dtype=int)
    return np.unique(np.asarray(out, dtype=int))


def _extract_simple_candidate_pool_v2(final_dict):
    """Broader simple-candidate pool for GUI threshold tuning.
    Includes refined simple, simple-burst spikes, vm_all_spikes, and input_spikes,
    then excludes complex spikes so complex remains locked.
    """
    simple_prev = np.asarray(final_dict.get('vm_simple_spikes', []), dtype=int).ravel()
    simple_prev_before = np.asarray(final_dict.get('vm_simple_spikes_before_raw_threshold', []), dtype=int).ravel()
    complex_spikes = np.asarray(final_dict.get('vm_complex_spikes', []), dtype=int).ravel()
    burst_simple = _extract_simple_burst_spikes_v2(final_dict)
    all_vm = np.asarray(final_dict.get('vm_all_spikes', []), dtype=int).ravel()
    input_spikes = np.asarray(final_dict.get('input_spikes', []), dtype=int).ravel()

    candidates = []
    for arr in (simple_prev, simple_prev_before, burst_simple, all_vm, input_spikes):
        if arr.size:
            candidates.append(arr)

    if len(candidates) == 0:
        return np.array([], dtype=int)

    pooled = np.unique(np.concatenate(candidates).astype(int))

    # Lock complex spikes: never editable in RAW-threshold GUI
    if complex_spikes.size:
        cset = set(complex_spikes.tolist())
        pooled = np.asarray([x for x in pooled if int(x) not in cset], dtype=int)

    return np.unique(pooled.astype(int))

def apply_raw_threshold_k_by_patch(
    final_dict,
    k_by_patch,
    frame_rate=500,
    patch_sec=30,
    baseline_ms=6.0,
    isi_ms_for_simple_burst=20,
):
    out = copy.deepcopy(final_dict)

    trace = np.asarray(out.get('trace_vol', []), dtype=float)
    simple_prev_only = np.asarray(out.get('vm_simple_spikes', []), dtype=int)
    complex_spikes = np.asarray(out.get('vm_complex_spikes', []), dtype=int)
    simple_prev = _extract_simple_candidate_pool_v2(out)

    burst_dict = out.get('vm_burst_dict', {}) if isinstance(out.get('vm_burst_dict', {}), dict) else {}
    cs_starts = burst_dict.get('starts', [])
    cs_ends = burst_dict.get('ends', [])

    trace_no_cs = _remove_complex_regions_linear_v2(trace, cs_starts, cs_ends)

    patch_frames = max(1, int(round(patch_sec * frame_rate)))
    pre_samples = max(1, int(round((baseline_ms / 1000.0) * frame_rate)))

    refined_simple = []
    rows = []

    n = len(trace_no_cs)
    patch_id = 0
    for p0 in range(0, n, patch_frames):
        p1 = min(n, p0 + patch_frames)
        k = float(k_by_patch[patch_id])

        patch_trace = trace_no_cs[p0:p1]
        sigma = float(np.nanstd(patch_trace))
        if (not np.isfinite(sigma)) or sigma <= 0:
            sigma = 1e-9
        mu = float(np.nanmedian(patch_trace)) if patch_trace.size > 0 else 0.0
        if not np.isfinite(mu):
            mu = 0.0

        patch_spikes = simple_prev[(simple_prev >= p0) & (simple_prev < p1)]
        thr_amp = k * sigma

        keep = []
        threshold_line = mu + thr_amp
        for s in patch_spikes:
            y = trace[s]
            if np.isfinite(y) and y >= threshold_line:
                keep.append(int(s))
        refined_simple.extend(keep)

        rows.append({
            'patch_id': int(patch_id),
            'start_frame': int(p0),
            'end_frame': int(p1),
            'start_sec': float(p0 / frame_rate),
            'end_sec': float(p1 / frame_rate),
            'mu': float(mu),
            'sigma': float(sigma),
            'k': float(k),
            'threshold_amp': float(thr_amp),
            'threshold_line': float(mu + thr_amp),
            'n_simple_in': int(patch_spikes.size),
            'n_simple_kept': int(len(keep)),
        })
        patch_id += 1

    refined_simple = np.asarray(sorted(set(refined_simple)), dtype=int)
    all_spikes_new = np.asarray(sorted(set(refined_simple.tolist() + complex_spikes.tolist())), dtype=int)

    out['vm_simple_spikes_before_raw_threshold'] = simple_prev_only
    out['vm_simple_candidate_pool_before_raw_threshold'] = simple_prev
    out['vm_simple_spikes'] = refined_simple
    out['vm_all_spikes'] = all_spikes_new
    out['trace_no_complex_for_raw_threshold'] = trace_no_cs
    out['simple_bursts'] = _simple_bursts_from_spikes_v2(refined_simple, frame_rate, isi_ms=isi_ms_for_simple_burst)
    out['raw_threshold_stage'] = {
        'enabled': True,
        'mode': 'chunkwise_k',
        'patch_sec': float(patch_sec),
        'baseline_ms': float(baseline_ms),
        'isi_ms_for_simple_burst': float(isi_ms_for_simple_burst),
    }

    return out, pd.DataFrame(rows)
def _select_k_for_one_patch_gui(
    trace,
    trace_no_cs,
    simple_prev,
    complex_spikes,
    all_final_spikes,
    p0,
    p1,
    frame_rate,
    init_k=3.0,
    k_min=0.5,
    k_max=6.0,
    baseline_ms=6.0,
):
    patch_simple = simple_prev[(simple_prev >= p0) & (simple_prev < p1)]
    patch_complex = complex_spikes[(complex_spikes >= p0) & (complex_spikes < p1)]
    patch_all = all_final_spikes[(all_final_spikes >= p0) & (all_final_spikes < p1)]

    if (p1 <= p0) or (p0 >= len(trace)):
        return float(init_k)

    patch_trace = trace_no_cs[p0:p1]
    sigma = float(np.nanstd(patch_trace))
    if (not np.isfinite(sigma)) or sigma <= 0:
        sigma = 1e-9
    mu = float(np.nanmedian(patch_trace)) if patch_trace.size > 0 else 0.0
    if not np.isfinite(mu):
        mu = 0.0

    pre_samples = max(1, int(round((baseline_ms / 1000.0) * frame_rate)))
    t = np.arange(p0, p1) / frame_rate

    fig, ax = plt.subplots(figsize=(13, 4.8))
    plt.subplots_adjust(bottom=0.24)

    ax.plot(t, trace[p0:p1], color='gray', lw=0.7, alpha=0.85, label='Raw trace')

    if patch_complex.size > 0:
        cx = patch_complex / frame_rate
        cy = trace[patch_complex]
        ax.scatter(cx, cy, s=16, c='deeppink', alpha=0.85, label='Complex spikes')

    if patch_all.size > 0:
        ax.scatter(
            patch_all / frame_rate,
            trace[patch_all],
            s=12,
            facecolors='none',
            edgecolors='goldenrod',
            linewidths=0.7,
            alpha=0.75,
            label='All final spikes (context)'
        )

    if patch_simple.size > 0:
        sx = patch_simple / frame_rate
        sy = trace[patch_simple]
        ax.scatter(sx, sy, s=14, c='lightskyblue', alpha=0.6, label='Simple candidate (editable)')

    kept_sc = ax.scatter([], [], s=20, c='dodgerblue', alpha=0.95, label='Simple kept')
    thr_line = ax.axhline(mu + init_k * sigma, ls='--', lw=1.2, c='black', label='Threshold')

    info = ax.text(
        0.01, 0.98, '', transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )

    ax.text(
        0.5, 1.02, 'Adjust k (0.5-6), then click Accept or press Enter (or N/Space)',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.75)
    )

    ax.set_title(f'Chunk {p0/frame_rate:.1f}s - {p1/frame_rate:.1f}s')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Signal')
    ax.legend(loc='upper right', fontsize=8)

    ax_k = plt.axes([0.12, 0.10, 0.60, 0.05])
    s_k = Slider(ax_k, 'k', k_min, k_max, valinit=float(init_k), valstep=0.05)

    ax_ok = plt.axes([0.76, 0.09, 0.16, 0.07])
    b_ok = Button(ax_ok, 'Accept', color='lightgray', hovercolor='0.85')

    state = {'k': float(init_k)}

    def redraw(_):
        k = float(s_k.val)
        state['k'] = k
        thr_amp = k * sigma
        thr_line.set_ydata([mu + thr_amp])

        keep = []
        threshold_line = mu + thr_amp
        for s in patch_simple:
            y = trace[s]
            if np.isfinite(y) and y >= threshold_line:
                keep.append(int(s))

        if len(keep) > 0:
            kept_xy = np.c_[np.asarray(keep, dtype=float) / frame_rate, trace[np.asarray(keep, dtype=int)]]
        else:
            kept_xy = np.empty((0, 2))
        kept_sc.set_offsets(kept_xy)

        info.set_text(
            f'k={k:.2f} | sigma={sigma:.4g} | thr_line={mu + thr_amp:.4g} | editable_in={len(patch_simple)} | kept={len(keep)} | all_final_in_chunk={len(patch_all)} | complex_locked={len(patch_complex)}'
        )
        fig.canvas.draw_idle()

    def accept_and_close(_=None):
        plt.close(fig)

    def on_key(event):
        key = (event.key or '').lower()
        if key in ('enter', 'return', 'kp_enter', 'n', ' '):
            accept_and_close()

    s_k.on_changed(redraw)
    b_ok.on_clicked(accept_and_close)
    fig.canvas.mpl_connect('key_press_event', on_key)

    redraw(None)
    plt.show(block=False)
    # Robust wait loop across notebook backends: continue only after this figure is closed.
    while plt.fignum_exists(fig.number):
        plt.pause(0.05)

    return float(state['k'])
def select_k_raw_threshold_interactive_chunkwise(
    final_dict,
    frame_rate=500,
    patch_sec=30,
    init_k=3.0,
    k_min=0.5,
    k_max=6.0,
    baseline_ms=6.0,
):
    frame_rate = _sanitize_frame_rate(frame_rate)
    trace = np.asarray(final_dict.get('trace_vol', []), dtype=float)
    simple_prev = _extract_simple_candidate_pool_v2(final_dict)
    complex_spikes = np.asarray(final_dict.get('vm_complex_spikes', []), dtype=int)
    all_final_spikes = np.asarray(final_dict.get('vm_all_spikes', []), dtype=int)
    if all_final_spikes.size == 0:
        all_final_spikes = np.unique(np.concatenate([simple_prev, complex_spikes])) if (simple_prev.size or complex_spikes.size) else np.array([], dtype=int)

    burst_dict = final_dict.get('vm_burst_dict', {}) if isinstance(final_dict.get('vm_burst_dict', {}), dict) else {}
    cs_starts = burst_dict.get('starts', [])
    cs_ends = burst_dict.get('ends', [])
    trace_no_cs = _remove_complex_regions_linear_v2(trace, cs_starts, cs_ends)

    patch_frames = max(1, int(round(patch_sec * frame_rate)))
    patch_bounds = []
    for p0 in range(0, len(trace), patch_frames):
        p1 = min(len(trace), p0 + patch_frames)
        patch_bounds.append((p0, p1))

    k_by_patch = np.full(len(patch_bounds), float(init_k), dtype=float)

    for i, (p0, p1) in enumerate(patch_bounds):
        patch_simple = simple_prev[(simple_prev >= p0) & (simple_prev < p1)]

        print(f'Tune chunk {i+1}/{len(patch_bounds)}: {p0/frame_rate:.1f}s-{p1/frame_rate:.1f}s (n_simple_candidates={len(patch_simple)}) | complex locked')
        k_sel = _select_k_for_one_patch_gui(
            trace=trace,
            trace_no_cs=trace_no_cs,
            simple_prev=simple_prev,
            complex_spikes=complex_spikes,
            all_final_spikes=all_final_spikes,
            p0=p0,
            p1=p1,
            frame_rate=frame_rate,
            init_k=float(k_by_patch[i]),
            k_min=k_min,
            k_max=k_max,
            baseline_ms=baseline_ms,
        )
        k_by_patch[i] = k_sel

    updated_dict, patch_df = apply_raw_threshold_k_by_patch(
        final_dict,
        k_by_patch=k_by_patch,
        frame_rate=frame_rate,
        patch_sec=patch_sec,
        baseline_ms=baseline_ms,
    )

    updated_dict['raw_threshold_stage']['k_by_patch'] = [float(v) for v in k_by_patch.tolist()]

    return k_by_patch, updated_dict, patch_df



In [13]:
# RAW_THRESHOLD_GUI_V1 - interactive save loop (chunk-wise k per 30s patch)
# Supports motor variants: spike_detection_refined_new{N}.pkl

final_dict_paths = [r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl'
   #r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\26-11-2025-ANST\FOV4\cell4\spike_detection_refined_new.pkl",
    # r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\...\spike_detection_refined_new0.pkl",
    # r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\...\spike_detection_refined_new1.pkl",
]

#input_folders = pathPyr[-1]
PATCH_SEC = 30
INIT_K = 3.0

def _write_latest_spike_deleted_html_from_pkl(pkl_path):
    """Regenerate a post-RAW-threshold HTML that is guaranteed to use the just-saved PKL arrays."""
    try:
        import plotly.graph_objects as go

        with open(pkl_path, 'rb') as f:
            d = pickle.load(f)

        trace = np.asarray(d.get('trace_vol', []), dtype=float).ravel()
        if trace.size == 0:
            return None

        det = d.get('detection_params', {}) if isinstance(d.get('detection_params', {}), dict) else {}
        fr = float(det.get('frame_rate', 500.0))
        if (not np.isfinite(fr)) or fr <= 0:
            fr = 500.0

        heights = np.asarray(d.get('spike_heights', []), dtype=float).ravel()
        scale = float(np.nanmedian(heights)) if heights.size > 0 else 1.0
        if (not np.isfinite(scale)) or scale <= 0:
            scale = 1.0
        trace_norm = trace / scale

        vm = np.asarray(d.get('vm', []), dtype=float).ravel()
        vm_norm = np.full(trace_norm.shape[0], np.nan, dtype=float)
        if vm.size > 0:
            n_copy = min(vm.size, vm_norm.size)
            vm_norm[:n_copy] = vm[:n_copy]

        t = np.arange(trace_norm.size) / fr

        def _as_idx(x):
            arr = np.asarray(x, dtype=int).ravel()
            if arr.size == 0:
                return arr
            return arr[(arr >= 0) & (arr < trace_norm.size)]

        idx_in = _as_idx(d.get('input_spikes', []))
        idx_simple = _as_idx(d.get('vm_simple_spikes', []))
        idx_complex = _as_idx(d.get('vm_complex_spikes', []))

        fig = go.Figure()
        fig.add_trace(go.Scattergl(x=t, y=trace_norm, mode='lines', name='Trace (norm)', line=dict(color='black', width=1), opacity=0.6))
        if np.isfinite(vm_norm).any():
            fig.add_trace(go.Scattergl(x=t, y=vm_norm, mode='lines', name='Vm (norm)', line=dict(color='green', width=1), opacity=0.9))

        def _add_points(name, idx, color, symbol, size):
            if idx.size == 0:
                return
            fig.add_trace(go.Scattergl(x=t[idx], y=trace_norm[idx], mode='markers', name=name, marker=dict(color=color, symbol=symbol, size=size)))

        _add_points('Input spike_idx', idx_in, 'rgba(80,80,80,0.8)', 'circle', 6)
        # Keep legacy legend names but force synchronization to post-deletion final labels
        _add_points('Single Spikes (Qixin)', idx_simple, 'blue', 'circle', 8)
        _add_points('Complex Spikes (Qixin)', idx_complex, 'red', 'x', 10)
        _add_points('Simple Spikes (Vm)', idx_simple, 'cyan', 'circle-open', 8)
        _add_points('Complex Spikes (Vm)', idx_complex, 'magenta', 'x', 10)

        bd = d.get('vm_burst_dict', {}) if isinstance(d.get('vm_burst_dict', {}), dict) else {}
        starts = np.asarray(bd.get('starts', []), dtype=float).ravel()
        ends = np.asarray(bd.get('ends', []), dtype=float).ravel()
        for s, e in zip(starts, ends):
            if not (np.isfinite(s) and np.isfinite(e)):
                continue
            fig.add_vrect(x0=float(s) / fr, x1=float(e) / fr, fillcolor='yellow', opacity=0.2, line_width=0, layer='below')

        latest_candidates = []
        for label, arr in [('Complex Spikes (Vm)', idx_complex), ('Complex Spikes (Qixin)', idx_complex), ('Simple Spikes (Vm)', idx_simple), ('Single Spikes (Qixin)', idx_simple)]:
            if arr.size:
                latest_candidates.append((int(arr.max()), label))
        if latest_candidates:
            latest_idx, latest_label = max(latest_candidates, key=lambda x: x[0])
            fig.add_trace(go.Scattergl(x=[t[latest_idx]], y=[trace_norm[latest_idx]], mode='markers+text', name='Latest spike', marker=dict(color='gold', size=14, symbol='star'), text=[f'Latest: {latest_label}'], textposition='top center'))

        title = (
            'Latest spike deleted & classified (interactive)'
            + f" | CS={det.get('pnorm_CS', np.nan):.2f}"
            + f" SS={det.get('pnorm_SS', np.nan):.2f}"
            + f" T={det.get('threshold', np.nan):.2f}"
            + f" HP={det.get('f_hp', np.nan):.1f}Hz"
        )
        fig.update_layout(title=title, xaxis_title='Time (s)', yaxis_title='Trace (norm)', template='plotly_white', height=650, legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0))

        suf_match = re.search(r"spike_detection_refined_new(.*?)\.pkl$", os.path.basename(pkl_path), flags=re.IGNORECASE)
        suf = (suf_match.group(1).strip() if suf_match else '')
        out_html = os.path.join(os.path.dirname(pkl_path), f'latest_spike_deleted_and_classified{suf}.html')
        fig.write_html(out_html, include_plotlyjs=True, full_html=True)
        return out_html
    except Exception as e:
        print(f"[WARN] failed to regenerate deleted/classified HTML for {pkl_path}: {e}")
        return None


def _natural_key_for_new_pkl(path):
    name = os.path.basename(path)
    m = re.search(r"spike_detection_refined_new(.*?)\.pkl$", name, flags=re.IGNORECASE)
    if not m:
        return (name.lower(), 0)
    suffix = m.group(1)
    if suffix == "":
        return ("", -1)
    try:
        return ("", int(suffix))
    except ValueError:
        return ("", 10**9)


if len(final_dict_paths) == 0:
    if len(input_folders) == 0 and 'pathPyr' in globals():
        input_folders = list(pathPyr)
    for folder in input_folders:
        final_dict_paths.extend(glob.glob(os.path.join(folder, 'spike_detection_refined_new*.pkl')))

# Deterministic de-duplication and ordering
final_dict_paths = list(dict.fromkeys(final_dict_paths))
final_dict_paths = sorted(final_dict_paths, key=lambda p: (_natural_key_for_new_pkl(p), os.path.dirname(p).lower(), os.path.basename(p).lower()))

if len(final_dict_paths) == 0:
    raise FileNotFoundError('No spike_detection_refined_new*.pkl files found. Fill final_dict_paths or input_folders first.')

for pkl_path in final_dict_paths:
    print(f'Opening chunk-wise GUI for: {pkl_path}')

    with open(pkl_path, 'rb') as f:
        final_dict = pickle.load(f)

    fr = final_dict.get('detection_params', {}).get('frame_rate', 500)
    fr = _sanitize_frame_rate(fr)
    complex_before = np.asarray(final_dict.get('vm_complex_spikes', []), dtype=int)

    k_by_patch, updated_dict, patch_df = select_k_raw_threshold_interactive_chunkwise(
        final_dict,
        frame_rate=fr,
        patch_sec=PATCH_SEC,
        init_k=INIT_K,
        k_min=0.5,
        k_max=6.0,
    )

    complex_after = np.asarray(updated_dict.get('vm_complex_spikes', []), dtype=int)

    # Persist chosen GUI k values (all chunks) into the saved PKL
    if 'raw_threshold_stage' not in updated_dict or not isinstance(updated_dict.get('raw_threshold_stage'), dict):
        updated_dict['raw_threshold_stage'] = {}

    gui_k_values = [float(v) for v in np.asarray(k_by_patch, dtype=float).ravel().tolist()]

    k_records = []
    if isinstance(patch_df, pd.DataFrame) and ('patch_id' in patch_df.columns):
        for _, r in patch_df.iterrows():
            k_records.append({
                'patch_id': int(r.get('patch_id', -1)),
                'start_frame': int(r.get('start_frame', -1)),
                'end_frame': int(r.get('end_frame', -1)),
                'start_sec': float(r.get('start_sec', np.nan)),
                'end_sec': float(r.get('end_sec', np.nan)),
                'k': float(r.get('k', np.nan)),
            })

    updated_dict['raw_threshold_stage'].update({
        'gui_k_by_patch': gui_k_values,
        'gui_k_records': k_records,
        'gui_patch_sec': float(PATCH_SEC),
        'gui_init_k': float(INIT_K),
    })

    if 'detection_params' not in updated_dict or not isinstance(updated_dict.get('detection_params'), dict):
        updated_dict['detection_params'] = {}
    updated_dict['detection_params']['raw_threshold_gui_k_by_patch'] = gui_k_values
    updated_dict['detection_params']['raw_threshold_gui_patch_sec'] = float(PATCH_SEC)

    if not np.array_equal(complex_before, complex_after):
        raise RuntimeError(f'Complex spikes changed unexpectedly for: {pkl_path}')

    with open(pkl_path, 'wb') as f:
        pickle.dump(updated_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    stage_csv = os.path.splitext(pkl_path)[0] + '_raw_threshold_stage.csv'
    patch_df.to_csv(stage_csv, index=False)

    print(f"[RAW-TH GUI] saved {pkl_path} | chunks={len(k_by_patch)} | k_range=({np.min(k_by_patch):.2f},{np.max(k_by_patch):.2f}) | simple now={len(updated_dict.get('vm_simple_spikes', []))} | complex kept={len(complex_after)}")

    try:
        html_synced = _write_latest_spike_deleted_html_from_pkl(pkl_path)
        if html_synced:
            print(f"[RAW-TH GUI] html synced -> {html_synced}")
    except Exception as _e_html:
        print(f"[RAW-TH GUI] html sync skipped: {_e_html}")




Opening chunk-wise GUI for: Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl
Tune chunk 1/4: 0.0s-30.0s (n_simple_candidates=608) | complex locked
Tune chunk 2/4: 30.0s-60.0s (n_simple_candidates=536) | complex locked
Tune chunk 3/4: 60.0s-90.0s (n_simple_candidates=625) | complex locked
Tune chunk 4/4: 90.0s-120.0s (n_simple_candidates=703) | complex locked
[RAW-TH GUI] saved Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl | chunks=4 | k_range=(3.75,4.10) | simple now=144 | complex kept=104
[RAW-TH GUI] html synced -> Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\latest_spike_deleted_and_classified.html


In [14]:
# PLATEAU_BATCH_POST_SPIKE_V2
# Run plateau detection only (no GUI) after spike detection.
# Does NOT change spike detection labels.
# Writes a new copy PKL: final refinement + plateau.

import os
import glob
import re
import copy
import pickle
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from qixin_spike_detection_4 import platue_detection

# Optional explicit list. If empty, resolve from final_dict_paths / input_folders / pathPyr.
PLATEAU_TARGET_PKLS = [r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl']

# Keep original final PKL unchanged by default.
PLATEAU_OVERWRITE_ORIGINAL = False

# Default plateau params (used if missing in detection_params)
PLATEAU_AMP_THRESHOLD = 0.8
PLATEAU_DURATION_THRESHOLD = 100.0
PLATEAU_KERNEL_MS = 100.0
PLATEAU_SCORE_MIN_MS = 80.0
ISI_THRESHOLD_MS = 20.0  # kept for metadata consistency; not used by platue_detection()
VM_CROSSING_THRESHOLD = 0.4
PLATEAU_DROP_ABS_THRESHOLD = 0.35
PLATEAU_DROP_FRAC_THRESHOLD = 0.45
PLATEAU_DROP_MIN_AFTER_PEAK_MS = 8.0
PLATEAU_DROP_REJECT_ENABLED = False
PLATEAU_HIGHPASS_HZ = 2.0

# Force these constants for plateau stage (recommended for consistency across cells).
# If False, values are read from each PKL detection_params when available.
PLATEAU_FORCE_DEFAULT_PARAMS = True


def _natural_key_for_new_pkl(path):
    name = os.path.basename(path)
    m = re.search(r"spike_detection_refined_new(.*?)\.pkl$", name, flags=re.IGNORECASE)
    if not m:
        return (name.lower(), 0)
    suffix = m.group(1)
    if suffix == "":
        return ("", -1)
    try:
        return ("", int(suffix))
    except ValueError:
        return ("", 10**9)


def _is_base_new_pkl(path):
    name = os.path.basename(str(path))
    return re.match(r"^spike_detection_refined_new(?:[mr]\d+)?\.pkl$", name, flags=re.IGNORECASE) is not None


def _resolve_target_pkls():
    def _normalize_paths(items):
        out = []
        if items is None:
            return out
        for x in items:
            if x is None:
                continue
            if isinstance(x, float):
                if np.isnan(x):
                    continue
            try:
                sx = os.fspath(x).strip()
            except Exception:
                continue
            if not sx:
                continue
            out.append(sx)
        return out

    def _expand_to_pkls(items):
        out = []
        for it in _normalize_paths(items):
            if os.path.isdir(it):
                out.extend([p for p in glob.glob(os.path.join(it, 'spike_detection_refined_new*.pkl')) if _is_base_new_pkl(p)])
            elif os.path.isfile(it) and it.lower().endswith('.pkl') and _is_base_new_pkl(it):
                out.append(it)
            else:
                # tolerate unresolved paths
                pass
        return _normalize_paths(out)

    if isinstance(PLATEAU_TARGET_PKLS, (list, tuple)) and len(PLATEAU_TARGET_PKLS) > 0:
        clean = _expand_to_pkls(PLATEAU_TARGET_PKLS)
        return sorted(list(dict.fromkeys(clean)), key=lambda p: (_natural_key_for_new_pkl(p), p.lower()))

    paths = []
    if 'final_dict_paths' in globals() and isinstance(final_dict_paths, (list, tuple)) and len(final_dict_paths) > 0:
        paths.extend(_expand_to_pkls(final_dict_paths))

    if len(paths) == 0:
        folders = []
        if 'input_folders' in globals() and isinstance(input_folders, (list, tuple)):
            folders.extend(_normalize_paths(input_folders))
        elif 'pathPyr' in globals():
            folders.extend(_normalize_paths(pathPyr))
        for folder in folders:
            if os.path.isdir(folder):
                paths.extend([p for p in glob.glob(os.path.join(folder, 'spike_detection_refined_new*.pkl')) if _is_base_new_pkl(p)])

    paths = _normalize_paths(paths)
    paths = list(dict.fromkeys(paths))
    return sorted(paths, key=lambda p: (_natural_key_for_new_pkl(p), os.path.dirname(p).lower(), os.path.basename(p).lower()))

def _as_sorted_unique_int(arr):
    if arr is None:
        return np.array([], dtype=int)
    a = np.asarray(arr, dtype=int).ravel()
    if a.size == 0:
        return np.array([], dtype=int)
    return np.unique(a)


def _get_trace_for_plateau(d):
    for k in ('trace_vol', 'raw_trace', 'trace'):
        tr = d.get(k, None)
        if tr is not None:
            tr = np.asarray(tr, dtype=float).ravel()
            if tr.size:
                return tr
    return np.array([], dtype=float)


def _get_all_spikes_for_plateau(d):
    for k in ('vm_all_spikes', 'all_spikes'):
        sp = d.get(k, None)
        if sp is not None:
            a = _as_sorted_unique_int(sp)
            if a.size:
                return a
    ss = _as_sorted_unique_int(d.get('vm_simple_spikes', []))
    cs = _as_sorted_unique_int(d.get('vm_complex_spikes', []))
    if ss.size or cs.size:
        return _as_sorted_unique_int(np.concatenate([ss, cs]))
    return np.array([], dtype=int)


def _build_trace_idx_for_plateau(d, trace):
    heights = np.asarray(d.get('spike_heights', []), dtype=float).ravel()
    hs = float(np.nanmedian(heights)) if heights.size and np.any(np.isfinite(heights)) else 1.0
    hs = max(hs, 1e-12)
    return trace / hs


def _get_frame_rate(d):
    fr = d.get('detection_params', {}).get('frame_rate', 500)
    try:
        fr = float(fr)
    except Exception:
        fr = 500.0
    if not np.isfinite(fr) or fr <= 0:
        fr = 500.0
    return fr


def _get_complex_burst_dict(d):
    cbd = d.get('vm_burst_dict', None)
    if isinstance(cbd, dict):
        return cbd
    return {'starts': np.array([], dtype=int), 'ends': np.array([], dtype=int)}


def _plateau_output_path(input_pkl_path):
    if PLATEAU_OVERWRITE_ORIGINAL:
        return input_pkl_path
    folder = os.path.dirname(input_pkl_path)
    out_name = "spike_detection_refined_new_plus_plateau.pkl"
    return os.path.join(folder, out_name)


def _pkl_suffix_from_name(pkl_path):
    name = os.path.basename(pkl_path)
    m = re.search(r"spike_detection_refined_new(.*?)\.pkl$", name, flags=re.IGNORECASE)
    return (m.group(1) if m else "").strip()


def _extract_vm_for_plot(d):
    bd = d.get('vm_burst_dict', {}) if isinstance(d.get('vm_burst_dict', {}), dict) else {}
    for key in ('trace_mf', 'trace_filt_wmf', 'trace_filt'):
        arr = np.asarray(bd.get(key, []), dtype=float).ravel()
        if arr.size:
            return arr
    vm = np.asarray(d.get('vm', []), dtype=float).ravel()
    if vm.size:
        return vm
    return np.asarray(d.get('trace_vol', []), dtype=float).ravel()


def _write_latest_with_plateaus_html(d, source_pkl_path, out_dir):
    trace_cal = np.asarray(d.get('trace_cal', []), dtype=float).ravel()
    trace_raw = np.asarray(d.get('trace_vol', []), dtype=float).ravel()
    trace_vm = _extract_vm_for_plot(d)

    fr = _get_frame_rate(d)
    cal_sr = 30.0

    t_raw = np.arange(trace_raw.size) / fr if trace_raw.size else np.asarray([], dtype=float)
    t_vm = np.arange(trace_vm.size) / fr if trace_vm.size else np.asarray([], dtype=float)
    t_cal = np.arange(trace_cal.size) / cal_sr if trace_cal.size else np.asarray([], dtype=float)

    det = d.get('detection_params', {}) if isinstance(d.get('detection_params', {}), dict) else {}
    vm_cross = float(det.get('vm_crossing_threshold', VM_CROSSING_THRESHOLD))
    plateau_amp_thr = float(det.get('plateau_amp_threshold', PLATEAU_AMP_THRESHOLD))

    trace_idx_plot = np.asarray([], dtype=float)
    vm_plateau = np.asarray([], dtype=float)
    plateau_score_plot = np.asarray([], dtype=float)
    try:
        _pld, _dbg, _par = _run_plateau_detection_only(d)
        _tr = _get_trace_for_plateau(d)
        if _tr.size:
            trace_idx_plot = _build_trace_idx_for_plateau(d, _tr)
        vm_plateau = np.asarray(_dbg.get('vm', []), dtype=float).ravel()
        plateau_score_plot = np.asarray(_dbg.get('plateau_score', []), dtype=float).ravel()
    except Exception:
        pass

    t_idx = np.arange(trace_idx_plot.size) / fr if trace_idx_plot.size else np.asarray([], dtype=float)
    t_plvm = np.arange(vm_plateau.size) / fr if vm_plateau.size else np.asarray([], dtype=float)
    t_plscore = np.arange(plateau_score_plot.size) / fr if plateau_score_plot.size else np.asarray([], dtype=float)

    simple_spikes = _as_sorted_unique_int(d.get('vm_simple_spikes', []))
    complex_spikes = _as_sorted_unique_int(d.get('vm_complex_spikes', []))
    all_spikes = _as_sorted_unique_int(d.get('vm_all_spikes', []))
    simple_spikes = simple_spikes[(simple_spikes >= 0) & (simple_spikes < trace_raw.size)]
    complex_spikes = complex_spikes[(complex_spikes >= 0) & (complex_spikes < trace_raw.size)]
    all_spikes = all_spikes[(all_spikes >= 0) & (all_spikes < trace_raw.size)]

    plateaus = d.get('vm_plateaus_dict', {}) if isinstance(d.get('vm_plateaus_dict', {}), dict) else {}
    pl_starts = np.asarray(plateaus.get('starts', []), dtype=int).ravel()
    pl_ends = np.asarray(plateaus.get('ends', []), dtype=int).ravel()
    pl_locs = np.asarray(plateaus.get('locs', []), dtype=int).ravel()
    rej_windows = plateaus.get('rejected_drop_windows', [])
    if not isinstance(rej_windows, list):
        rej_windows = []

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    if trace_cal.size:
        fig.add_trace(
            go.Scatter(x=t_cal, y=trace_cal, mode='lines', line=dict(color='#7f7f7f', width=1.1), name='Calcium'),
            secondary_y=False
        )
    if trace_raw.size:
        fig.add_trace(
            go.Scatter(x=t_raw, y=trace_raw, mode='lines', line=dict(color='#f28e2b', width=0.8), name='Raw voltage'),
            secondary_y=True
        )
    if trace_vm.size:
        fig.add_trace(
            go.Scatter(x=t_vm, y=trace_vm, mode='lines', line=dict(color='green', width=1.0), name='VM (CS detection)'),
            secondary_y=True
        )
    if trace_idx_plot.size:
        fig.add_trace(
            go.Scatter(x=t_idx, y=trace_idx_plot, mode='lines', line=dict(color='#17becf', width=1.0), name='Plateau input trace_idx'),
            secondary_y=True
        )
    if vm_plateau.size:
        fig.add_trace(
            go.Scatter(x=t_plvm, y=vm_plateau, mode='lines', line=dict(color='#2ca02c', width=1.2, dash='dot'), name='Plateau Vm'),
            secondary_y=True
        )
    if plateau_score_plot.size:
        fig.add_trace(
            go.Scatter(x=t_plscore, y=plateau_score_plot, mode='lines', line=dict(color='#1f77b4', width=1.1), name='Plateau score'),
            secondary_y=True
        )

    for s, e in zip(pl_starts, pl_ends):
        if trace_raw.size == 0:
            break
        ss = int(min(s, e))
        ee = int(max(s, e))
        if ss < trace_raw.size and ee < trace_raw.size:
            fig.add_vrect(x0=t_raw[ss], x1=t_raw[ee], fillcolor='rgba(190, 60, 255, 0.20)', line_width=0, layer='below')
            # Explicit boundaries to make plateau windows visible even when fill is subtle.
            fig.add_vline(x=t_raw[ss], line=dict(color='purple', width=1, dash='dot'))
            fig.add_vline(x=t_raw[ee], line=dict(color='purple', width=1, dash='dot'))

    # Optional: visualize rejected plateau windows (drop criterion)
    for rw in rej_windows:
        try:
            ss = int(min(rw.get('start', 0), rw.get('end', 0)))
            ee = int(max(rw.get('start', 0), rw.get('end', 0)))
            if trace_raw.size and ss < trace_raw.size and ee < trace_raw.size and ss <= ee:
                fig.add_vrect(
                    x0=t_raw[ss], x1=t_raw[ee],
                    fillcolor='rgba(255,140,0,0.18)', line_width=0, layer='below'
                )
        except Exception:
            pass

    if all_spikes.size:
        fig.add_trace(
            go.Scatter(x=t_raw[all_spikes], y=trace_raw[all_spikes], mode='markers',
                       marker=dict(color='rgba(0,0,255,0.25)', size=4), name='All spikes'),
            secondary_y=True
        )
    if simple_spikes.size:
        fig.add_trace(
            go.Scatter(x=t_raw[simple_spikes], y=trace_raw[simple_spikes], mode='markers',
                       marker=dict(color='rgb(31,119,180)', size=6), name='Simple spikes'),
            secondary_y=True
        )
    if complex_spikes.size:
        fig.add_trace(
            go.Scatter(x=t_raw[complex_spikes], y=trace_raw[complex_spikes], mode='markers',
                       marker=dict(color='black', size=7), name='Complex spikes'),
            secondary_y=True
        )
    if pl_locs.size and trace_raw.size:
        pl_locs = pl_locs[(pl_locs >= 0) & (pl_locs < trace_raw.size)]
        if pl_locs.size:
            fig.add_trace(
                go.Scatter(x=t_raw[pl_locs], y=trace_raw[pl_locs], mode='markers',
                           marker=dict(color='purple', size=9, symbol='diamond-open'),
                           name='Plateau loci'),
                secondary_y=True
            )

    # Legend handle for shaded plateau windows.
    if len(pl_starts) > 0:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode='markers',
                marker=dict(size=10, color='rgba(190, 60, 255, 0.35)', symbol='square'),
                name='Plateau window'
            ),
            secondary_y=True
        )
    if len(rej_windows) > 0:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode='markers',
                marker=dict(size=10, color='rgba(255,140,0,0.35)', symbol='square'),
                name='Rejected (drop)'
            ),
            secondary_y=True
        )


    # Add horizontal thresholds used in plateau detection
    _t_ends = [arr[-1] for arr in (t_raw, t_vm, t_idx, t_plvm, t_plscore) if arr.size]
    _x1 = float(max(_t_ends)) if _t_ends else 1.0
    fig.add_shape(
        type='line', x0=0.0, x1=_x1, y0=float(vm_cross), y1=float(vm_cross),
        xref='x', yref='y2', line=dict(color='#d62728', width=1.5, dash='dash')
    )
    fig.add_shape(
        type='line', x0=0.0, x1=_x1, y0=float(plateau_amp_thr), y1=float(plateau_amp_thr),
        xref='x', yref='y2', line=dict(color='#1f77b4', width=1.5, dash='dashdot')
    )
    fig.add_trace(
        go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='#d62728', width=1.5, dash='dash'), name=f'Vm crossing thr={vm_cross:g}'),
        secondary_y=True
    )
    fig.add_trace(
        go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='#1f77b4', width=1.5, dash='dashdot'), name=f'Plateau amp thr={plateau_amp_thr:g}'),
        secondary_y=True
    )

    out_html = os.path.join(out_dir, "latest_spike_detected_and_classified_platue.html")
    n_plateaus = int(min(len(pl_starts), len(pl_ends)))
    n_rej = int(len(rej_windows))
    fig.update_layout(
        template='plotly_white',
        title=f"{os.path.basename(os.path.dirname(source_pkl_path))} | latest_spike_detected_and_classified_platue | kept={n_plateaus} rejected={n_rej}",
        width=1400,
        height=520,
        legend=dict(x=1.02, y=1, xanchor='left', yanchor='top'),
        margin=dict(l=60, r=240, t=80, b=60),
    )
    fig.update_xaxes(title_text='Time (s)')
    fig.update_yaxes(title_text='Calcium (dF/F)', secondary_y=False)
    fig.update_yaxes(title_text='Voltage / VM (a.u.)', secondary_y=True)
    fig.write_html(out_html, include_plotlyjs='cdn')
    return out_html


def _run_plateau_detection_only(d):
    trace = _get_trace_for_plateau(d)
    if trace.size == 0:
        raise ValueError('no trace found in pkl')

    fr = _get_frame_rate(d)
    all_spikes = _get_all_spikes_for_plateau(d)
    if all_spikes.size == 0:
        raise ValueError('no spikes found in pkl')

    trace_idx = _build_trace_idx_for_plateau(d, trace)
    all_spikes = all_spikes[(all_spikes >= 0) & (all_spikes < len(trace_idx))]
    cb_dict = _get_complex_burst_dict(d)

    det = d.get('detection_params', {}) if isinstance(d.get('detection_params', {}), dict) else {}
    if PLATEAU_FORCE_DEFAULT_PARAMS:
        amp = float(PLATEAU_AMP_THRESHOLD)
        dur = float(PLATEAU_DURATION_THRESHOLD)
        ker = float(PLATEAU_KERNEL_MS)
        score_min = float(PLATEAU_SCORE_MIN_MS)
        drop_abs = float(PLATEAU_DROP_ABS_THRESHOLD)
        drop_frac = float(PLATEAU_DROP_FRAC_THRESHOLD)
        drop_guard = float(PLATEAU_DROP_MIN_AFTER_PEAK_MS)
    else:
        amp = float(det.get('plateau_amp_threshold', PLATEAU_AMP_THRESHOLD))
        dur = float(det.get('plateau_duration_threshold', PLATEAU_DURATION_THRESHOLD))
        ker = float(det.get('plateau_kernel_ms', PLATEAU_KERNEL_MS))
        score_min = float(det.get('plateau_score_min_ms', PLATEAU_SCORE_MIN_MS))
        drop_abs = float(det.get('plateau_drop_abs_threshold', PLATEAU_DROP_ABS_THRESHOLD))
        drop_frac = float(det.get('plateau_drop_frac_threshold', PLATEAU_DROP_FRAC_THRESHOLD))
        drop_guard = float(det.get('plateau_drop_min_after_peak_ms', PLATEAU_DROP_MIN_AFTER_PEAK_MS))

    # Use one global normalized Vm threshold for all traces (logic-fixed, not per-trace value).
    cross = float(VM_CROSSING_THRESHOLD)

    plateaus_dict, dbg = platue_detection(
        trace_idx=trace_idx,
        spike_heights_interpolated=np.ones_like(trace_idx, dtype=float),
        all_spikes=all_spikes,
        fr=fr,
        complex_bursts_dict=cb_dict,
        highpass=float(PLATEAU_HIGHPASS_HZ),
        median_window=11,
        baseline_subtract=True,
        baseline_window_ms=20,
        baseline_percentile=10,
        vm_crossing_threshold=cross,
        plateau_duration_threshold=dur,
        plateau_amp_threshold=amp,
        plateau_kernel_ms=ker,
        plateau_score_min_ms=score_min,
        plateau_min_spikes=1,
        drop_reject_enabled=bool(PLATEAU_DROP_REJECT_ENABLED),
        drop_abs_threshold=drop_abs,
        drop_frac_threshold=drop_frac,
        drop_min_after_peak_ms=drop_guard,
    )

    params_used = {
        'plateau_amp_threshold': amp,
        'plateau_duration_threshold': dur,
        'plateau_kernel_ms': ker,
        'plateau_score_min_ms': score_min,
        'isi_threshold_ms': float(ISI_THRESHOLD_MS),
        'vm_crossing_threshold': cross,
        'plateau_drop_abs_threshold': drop_abs,
        'plateau_drop_frac_threshold': drop_frac,
        'plateau_drop_min_after_peak_ms': drop_guard,
        'plateau_drop_reject_enabled': bool(PLATEAU_DROP_REJECT_ENABLED),
        'plateau_highpass_hz': float(PLATEAU_HIGHPASS_HZ),
        'plateau_force_default_params': bool(PLATEAU_FORCE_DEFAULT_PARAMS),
    }
    return plateaus_dict, dbg, params_used


paths = _resolve_target_pkls()
# Safety: expand any folder path to PKLs and drop non-PKL items
paths_expanded = []
for _it in paths:
    try:
        _s = os.fspath(_it).strip()
    except Exception:
        continue
    if not _s:
        continue
    if os.path.isdir(_s):
        paths_expanded.extend([p for p in glob.glob(os.path.join(_s, 'spike_detection_refined_new*.pkl')) if _is_base_new_pkl(p)])
    elif os.path.isfile(_s) and _s.lower().endswith('.pkl') and _is_base_new_pkl(_s):
        paths_expanded.append(_s)
paths = sorted(list(dict.fromkeys(paths_expanded)), key=lambda p: (_natural_key_for_new_pkl(p), os.path.dirname(p).lower(), os.path.basename(p).lower()))
if len(paths) == 0:
    raise FileNotFoundError('No spike_detection_refined_new*.pkl found for plateau batch stage.')

saved = []
saved_html = []
for pkl_path in paths:
    print('\n' + '=' * 90)
    print(f'PLATEAU batch stage: {pkl_path}')

    with open(pkl_path, 'rb') as f:
        d = pickle.load(f)

    # Keep spike labels exactly as-is
    simple_before = _as_sorted_unique_int(d.get('vm_simple_spikes', []))
    complex_before = _as_sorted_unique_int(d.get('vm_complex_spikes', []))
    all_before = _as_sorted_unique_int(d.get('vm_all_spikes', []))

    try:
        plateaus_dict, _dbg, params_used = _run_plateau_detection_only(d)
    except Exception as e:
        print(f'  skipped: {e}')
        continue

    d_new = copy.deepcopy(d)
    d_new['vm_plateaus_dict'] = plateaus_dict

    if 'detection_params' not in d_new or not isinstance(d_new['detection_params'], dict):
        d_new['detection_params'] = {}
    d_new['detection_params'].update(params_used)

    # Hard safety check: spike classification must remain unchanged
    simple_after = _as_sorted_unique_int(d_new.get('vm_simple_spikes', []))
    complex_after = _as_sorted_unique_int(d_new.get('vm_complex_spikes', []))
    all_after = _as_sorted_unique_int(d_new.get('vm_all_spikes', []))
    if (not np.array_equal(simple_before, simple_after)) or (not np.array_equal(complex_before, complex_after)) or (not np.array_equal(all_before, all_after)):
        raise RuntimeError(f'spike arrays changed unexpectedly for {pkl_path}')

    out_pkl_path = _plateau_output_path(pkl_path)
    d_new['plateau_source_pkl_path'] = str(pkl_path)
    d_new['plateau_output_pkl_path'] = str(out_pkl_path)

    with open(out_pkl_path, 'wb') as f:
        pickle.dump(d_new, f, protocol=pickle.HIGHEST_PROTOCOL)

    try:
        html_path = _write_latest_with_plateaus_html(d_new, pkl_path, os.path.dirname(out_pkl_path))
        saved_html.append(html_path)
        print(f"[PLATEAU BATCH] html  {html_path}")
    except Exception as e:
        print(f"[PLATEAU BATCH] html failed for {pkl_path}: {e}")

    n_pl = len(np.asarray(plateaus_dict.get('starts', []), dtype=int))
    n_rej = len(plateaus_dict.get('rejected_drop_windows', [])) if isinstance(plateaus_dict.get('rejected_drop_windows', []), list) else 0
    print(f"[PLATEAU BATCH] saved {out_pkl_path} | source={pkl_path} | n_plateaus={n_pl} | rejected_drop={n_rej}")
    saved.append(out_pkl_path)

print(f"\nDone. input={len(paths)} | saved_pkls={len(saved)} | saved_html={len(saved_html)}")



PLATEAU batch stage: Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl
[PLATEAU BATCH] html  Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\latest_spike_detected_and_classified_platue.html
[PLATEAU BATCH] saved Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new_plus_plateau.pkl | source=Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detection_refined_new.pkl | n_plateaus=2 | rejected_drop=0

Done. input=1 | saved_pkls=1 | saved_html=1


In [ ]:
# -------------------------
# Export spike classification overlays (with VM in green)
# Export spike classification overlays (raw + VM used for CS detection)
# Uses notebook DB variables: pathPyr / bsPyr / AllCalSR (or DB fallback)
# -------------------------

import os
import glob
import re
import pickle
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
def _safe_float(x, default=30.0):
    try:
        v = float(x)
        if np.isfinite(v) and v > 0:
            return v
    except Exception:
        pass
    return float(default)
def _pkl_suffix_from_name(pkl_path):
    name = os.path.basename(pkl_path)
    m = re.search(r"spike_detection_refined_new(.*?)\.pkl$", name, flags=re.IGNORECASE)
    suf = m.group(1) if m else ''
    return suf.strip()
def _pick_pkls_for_cell(cell_dir, brain_state=''):
    pkl_paths = glob.glob(os.path.join(cell_dir, 'spike_detection_refined_new*.pkl'))
    if not pkl_paths:
        return []

    pkl_paths = sorted(pkl_paths, key=lambda p: os.path.getmtime(p))
    is_motor = str(brain_state).strip().lower() == 'motor'

    if is_motor:
        # For motor cells keep all relevant segments (e.g., m0/m1/r0/r1), dedup by suffix.
        by_suffix = {}
        for p in pkl_paths:
            by_suffix[_pkl_suffix_from_name(p)] = p
        return [by_suffix[k] for k in sorted(by_suffix.keys())]

    # For non-motor keep latest only.
    return [pkl_paths[-1]]
def _in_bounds(idx, n):
    idx = np.asarray(idx, dtype=int).ravel()
    if n <= 0 or idx.size == 0:
        return np.asarray([], dtype=int)
    m = (idx >= 0) & (idx < n)
    if not np.any(m):
        return np.asarray([], dtype=int)
    return np.unique(idx[m])
def _extract_vm_trace_for_cs(data, raw_len):
    # Preferred order: explicit saved VM fields, then VM from burst dict (used in CS detection pipeline).
    burst_dict = data.get('vm_burst_dict', {}) if isinstance(data.get('vm_burst_dict', {}), dict) else {}

    candidates = [
        # This is the VM/subthreshold trace used in complex-spike detection.
        burst_dict.get('trace_mf', None),
        burst_dict.get('trace_filt_wmf', None),
        burst_dict.get('trace_filt', None),
        data.get('vm_no_spike', None),
        data.get('vm_trace', None),
        data.get('vm', None),
        data.get('trace_no_complex_for_raw_threshold', None),
    ]

    for c in candidates:
        if c is None:
            continue
        arr = np.asarray(c, dtype=float).ravel()
        if arr.size > 0:
            return arr

    # fallback: empty
    return np.asarray([], dtype=float)
def build_latest_like_figure_with_vm(data, cal_sr=30.0, title=''):
    trace_cal = np.asarray(data.get('trace_cal', []), dtype=float).ravel()
    trace_raw = np.asarray(data.get('trace_vol', []), dtype=float).ravel()
    trace_vm = _extract_vm_trace_for_cs(data, raw_len=trace_raw.size)

    det = data.get('detection_params', {}) or {}
    vm_sr = _safe_float(det.get('frame_rate', 500.0), default=500.0)
    cal_sr = _safe_float(cal_sr, default=30.0)

    t_cal = np.arange(trace_cal.size) / cal_sr if trace_cal.size else np.asarray([], dtype=float)
    t_raw = np.arange(trace_raw.size) / vm_sr if trace_raw.size else np.asarray([], dtype=float)
    t_vm = np.arange(trace_vm.size) / vm_sr if trace_vm.size else np.asarray([], dtype=float)

    all_spikes = _in_bounds(data.get('vm_all_spikes', []), trace_raw.size)
    simple_spikes = _in_bounds(data.get('vm_simple_spikes', []), trace_raw.size)
    complex_spikes = _in_bounds(data.get('vm_complex_spikes', []), trace_raw.size)

    burst_dict = data.get('vm_burst_dict', {}) if isinstance(data.get('vm_burst_dict', {}), dict) else {}
    cs_starts = _in_bounds(burst_dict.get('starts', []), trace_raw.size)
    cs_ends = _in_bounds(burst_dict.get('ends', []), trace_raw.size)

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    if trace_cal.size:
        fig.add_trace(
            go.Scatter(
                x=t_cal,
                y=trace_cal,
                mode='lines',
                name='Calcium trace',
                line=dict(color='#7f7f7f', width=1.2),
            ),
            secondary_y=False,
        )

    # Raw voltage trace
    if trace_raw.size:
        fig.add_trace(
            go.Scatter(
                x=t_raw,
                y=trace_raw,
                mode='lines',
                name='Raw voltage trace',
                line=dict(color='#f28e2b', width=0.9),
                opacity=0.85,
            ),
            secondary_y=True,
        )

    # VM trace used for complex detection (no-spike/subthreshold style)
    if trace_vm.size:
        fig.add_trace(
            go.Scatter(
                x=t_vm,
                y=trace_vm,
                mode='lines',
                name='VM trace (CS detection, no-spike)',
                line=dict(color='green', width=1.1),
                opacity=0.95,
            ),
            secondary_y=True,
        )

    # Yellow windows for complex-spike event spans (start to end from PKL).
    if cs_starts.size and cs_ends.size:
        n_pair = min(cs_starts.size, cs_ends.size)
        for i in range(n_pair):
            st = int(cs_starts[i])
            en = int(cs_ends[i])
            if en < st:
                st, en = en, st
            if st < trace_raw.size and en < trace_raw.size:
                x0 = t_raw[st]
                x1 = t_raw[en]
                fig.add_vrect(
                    x0=x0,
                    x1=x1,
                    fillcolor='rgba(235, 228, 120, 0.35)',
                    line_width=0,
                    layer='below',
                )

    # Spike markers remain on raw voltage for direct visual alignment with original spikes
    if all_spikes.size:
        fig.add_trace(
            go.Scatter(
                x=t_raw[all_spikes],
                y=trace_raw[all_spikes],
                mode='markers',
                name='All VM spikes',
                marker=dict(color='rgba(0,0,255,0.22)', size=4),
            ),
            secondary_y=True,
        )

    if simple_spikes.size:
        fig.add_trace(
            go.Scatter(
                x=t_raw[simple_spikes],
                y=trace_raw[simple_spikes],
                mode='markers',
                name='Simple spikes (VM)',
                marker=dict(color='rgb(31,119,180)', size=6),
            ),
            secondary_y=True,
        )

    if complex_spikes.size:
        fig.add_trace(
            go.Scatter(
                x=t_raw[complex_spikes],
                y=trace_raw[complex_spikes],
                mode='markers',
                name='Complex spikes (VM)',
                marker=dict(color='black', size=7, symbol='circle'),
            ),
            secondary_y=True,
        )

    fig.update_layout(
        template='plotly_white',
        title=title,
        width=1400,
        height=520,
        legend=dict(x=1.02, y=1, xanchor='left', yanchor='top'),
        margin=dict(l=60, r=240, t=80, b=60),
    )
    fig.update_xaxes(title_text='Time (s)')
    fig.update_yaxes(title_text='Calcium (dF/F)', secondary_y=False)
    fig.update_yaxes(title_text='Voltage / VM (a.u.)', secondary_y=True)

    return fig
def _build_source_table_from_existing_db():
    # Preferred: notebook vectors built by existing DB-reading cells.
    if 'pathPyr' in globals() and pathPyr is not None and len(pathPyr) > 0:
        folders = [str(x) for x in list(pathPyr)]
        n = len(folders)

        states = [''] * n
        if 'bsPyr' in globals() and bsPyr is not None and len(bsPyr) == n:
            states = [str(x) for x in list(bsPyr)]

        cals = [30.0] * n
        if 'AllCalSR' in globals() and AllCalSR is not None:
            try:
                vals = list(AllCalSR)
                if len(vals) == n:
                    cals = [_safe_float(v, 30.0) for v in vals]
            except Exception:
                pass

        return pd.DataFrame({'folder': folders, 'brainState': states, 'CALsr': cals})

    # Fallback: use DB dataframe if present in notebook.
    if 'DB' in globals() and isinstance(DB, pd.DataFrame) and len(DB) > 0:
        src = DB.copy()
        if 'folder' not in src.columns and 'Link' in src.columns:
            src['folder'] = src['Link']
        if 'brainState' not in src.columns:
            src['brainState'] = ''
        if 'CALsr' not in src.columns:
            src['CALsr'] = 30.0
        return src[['folder', 'brainState', 'CALsr']]

    raise RuntimeError(
        'No in-notebook DB source found. Run the DB-reading cell first (defines pathPyr/bsPyr/AllCalSR or DB).'
    )
src_df = _build_source_table_from_existing_db()
src_df = src_df.dropna(subset=['folder']).drop_duplicates(subset=['folder'], keep='last').reset_index(drop=True)
saved_html = []
missing_pkls = []
failed = []
for _, row in src_df.iterrows():
    cell_dir = str(row['folder'])
    state = str(row.get('brainState', ''))
    cal_sr = _safe_float(row.get('CALsr', 30.0), default=30.0)

    if not os.path.isdir(cell_dir):
        missing_pkls.append((cell_dir, 'folder_missing'))
        continue

    pkl_list = _pick_pkls_for_cell(cell_dir, brain_state=state)
    if len(pkl_list) == 0:
        missing_pkls.append((cell_dir, 'no_spike_detection_refined_new*.pkl'))
        continue

    for pkl_path in pkl_list:
        suffix = _pkl_suffix_from_name(pkl_path)
        N = suffix if suffix else 'main'
        out_html = os.path.join(cell_dir, f'spike_detected_and_classified_vm{N}.html')

        try:
            with open(pkl_path, 'rb') as f:
                d = pickle.load(f)

            title = (
                f"{os.path.basename(cell_dir)} | {os.path.basename(pkl_path)}"
                f" | state={state} | CALsr={cal_sr:.1f}"
            )
            fig = build_latest_like_figure_with_vm(d, cal_sr=cal_sr, title=title)
            fig.write_html(out_html, include_plotlyjs='cdn')
            saved_html.append(out_html)
            print(f"[OK] {out_html}")
        except Exception as e:
            failed.append((pkl_path, str(e)))
            print(f"[FAIL] {pkl_path}: {e}")

print(f"\nSaved HTML files: {len(saved_html)}")
if missing_pkls:
    print(f"Missing cells/files: {len(missing_pkls)}")
if failed:
    print(f"Failed PKLs: {len(failed)}")



[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\spike_detected_and_classified_vmmain.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov2\cell1\spike_detected_and_classified_vmmain.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov6\cell0\spike_detected_and_classified_vmmain.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\l\20-08-25-ans\cell0\spike_detected_and_classified_vmmain.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\17-09-2025-rugc42-wh-s1-ans\fov1\2\cell0\spike_detected_and_classified_vmmain.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\cell1\spike_detected_and_classified_vmm0.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\cell1\spike_detected_and_classified_vmm1.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\cell1\spike_detected_and_classified_vmr0.html
[OK] Z:\Adam-Lab-Shared\Data\Michal_

In [10]:
# PLATEAU_BACKFILL_STAGE_V1
# Add/refresh plateau detection into existing spike_detection_refined_new*.pkl
# WITHOUT changing any already-classified final spikes.
# Also recreates latest_spike_detected_and_classified*.html with plateaus marked.

import os
import re
import glob
import copy
import pickle
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _pkl_suffix_from_name(pkl_path):
    m = re.search(r"spike_detection_refined_new(.*?)\.pkl$", os.path.basename(pkl_path), flags=re.IGNORECASE)
    suf = m.group(1) if m else ''
    return suf.strip()

def _resolve_final_pkl_paths(items):
    resolved = []
    for item in (items or []):
        if item is None:
            continue
        p = os.path.normpath(str(item).strip())
        if not p:
            continue

        if os.path.isfile(p) and p.lower().endswith('.pkl'):
            resolved.append(p)
            continue

        if os.path.isdir(p):
            hits = sorted(glob.glob(os.path.join(p, 'spike_detection_refined_new*.pkl')))
            resolved.extend(hits)
            continue

        if any(ch in p for ch in ['*', '?', '[']):
            hits = sorted([h for h in glob.glob(p) if os.path.isfile(h) and h.lower().endswith('.pkl')])
            resolved.extend(hits)
            continue

        if os.path.isfile(p + '.pkl'):
            resolved.append(p + '.pkl')
            continue

        print(f"[WARN] skipping unresolved path: {p}")

    out = []
    seen = set()
    for p in resolved:
        k = os.path.normcase(os.path.normpath(p))
        if k in seen:
            continue
        seen.add(k)
        out.append(p)
    return out
def _as_int_unique(x):
    a = np.asarray(x, dtype=int).ravel()
    if a.size == 0:
        return np.asarray([], dtype=int)
    return np.unique(a)
def _normalize_plateaus_dict(d):
    if not isinstance(d, dict):
        return {
            'starts': np.asarray([], dtype=np.int64),
            'ends': np.asarray([], dtype=np.int64),
            'durations_ms': np.asarray([], dtype=float),
            'amplitudes': np.asarray([], dtype=float),
            'baselines': np.asarray([], dtype=float),
            'locs': np.asarray([], dtype=np.int64),
            'peaks': np.asarray([], dtype=float),
            'n_spikes': np.asarray([], dtype=np.int64),
            'spike_indices': [],
        }
    out = {}
    out['starts'] = np.asarray(d.get('starts', []), dtype=np.int64)
    out['ends'] = np.asarray(d.get('ends', []), dtype=np.int64)
    out['durations_ms'] = np.asarray(d.get('durations_ms', []), dtype=float)
    out['amplitudes'] = np.asarray(d.get('amplitudes', []), dtype=float)
    out['baselines'] = np.asarray(d.get('baselines', []), dtype=float)
    out['locs'] = np.asarray(d.get('locs', []), dtype=np.int64)
    out['peaks'] = np.asarray(d.get('peaks', []), dtype=float)
    out['n_spikes'] = np.asarray(d.get('n_spikes', []), dtype=np.int64)
    raw_si = d.get('spike_indices', [])
    if isinstance(raw_si, (list, tuple)):
        out['spike_indices'] = [np.asarray(si, dtype=np.int64) for si in raw_si]
    else:
        out['spike_indices'] = []
    return out
def _extract_vm_for_detection(final_dict):
    vm = np.asarray(final_dict.get('vm', []), dtype=float).ravel()
    if vm.size > 0:
        return vm
    return np.asarray(final_dict.get('trace_vol', []), dtype=float).ravel()
def _extract_vm_for_plot(final_dict):
    bd = final_dict.get('vm_burst_dict', {}) if isinstance(final_dict.get('vm_burst_dict', {}), dict) else {}
    for key in ('trace_mf', 'trace_filt_wmf', 'trace_filt'):
        arr = np.asarray(bd.get(key, []), dtype=float).ravel()
        if arr.size > 0:
            return arr
    vm = np.asarray(final_dict.get('vm', []), dtype=float).ravel()
    if vm.size > 0:
        return vm
    return np.asarray(final_dict.get('trace_vol', []), dtype=float).ravel()
def _recreate_latest_with_plateaus(final_dict, pkl_path, cal_sr=30.0):
    trace_cal = np.asarray(final_dict.get('trace_cal', []), dtype=float).ravel()
    trace_raw = np.asarray(final_dict.get('trace_vol', []), dtype=float).ravel()
    trace_vm = _extract_vm_for_plot(final_dict)

    fr = float(final_dict.get('detection_params', {}).get('frame_rate', 500.0))
    if (not np.isfinite(fr)) or fr <= 0:
        fr = 500.0
    if (not np.isfinite(cal_sr)) or cal_sr <= 0:
        cal_sr = 30.0

    t_raw = np.arange(trace_raw.size) / fr if trace_raw.size else np.asarray([], dtype=float)
    t_vm = np.arange(trace_vm.size) / fr if trace_vm.size else np.asarray([], dtype=float)
    t_cal = np.arange(trace_cal.size) / cal_sr if trace_cal.size else np.asarray([], dtype=float)

    simple_spikes = _as_int_unique(final_dict.get('vm_simple_spikes', []))
    complex_spikes = _as_int_unique(final_dict.get('vm_complex_spikes', []))
    all_spikes = _as_int_unique(final_dict.get('vm_all_spikes', []))

    simple_spikes = simple_spikes[(simple_spikes >= 0) & (simple_spikes < trace_raw.size)]
    complex_spikes = complex_spikes[(complex_spikes >= 0) & (complex_spikes < trace_raw.size)]
    all_spikes = all_spikes[(all_spikes >= 0) & (all_spikes < trace_raw.size)]

    plateaus = _normalize_plateaus_dict(final_dict.get('vm_plateaus_dict', {}))
    pl_starts = plateaus.get('starts', np.asarray([], dtype=np.int64))
    pl_ends = plateaus.get('ends', np.asarray([], dtype=np.int64))
    pl_locs = plateaus.get('locs', np.asarray([], dtype=np.int64))

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    if trace_cal.size:
        fig.add_trace(go.Scatter(x=t_cal, y=trace_cal, mode='lines', line=dict(color='#7f7f7f', width=1.1), name='Calcium'), secondary_y=False)
    if trace_raw.size:
        fig.add_trace(go.Scatter(x=t_raw, y=trace_raw, mode='lines', line=dict(color='#f28e2b', width=0.8), name='Raw voltage'), secondary_y=True)
    if trace_vm.size:
        fig.add_trace(go.Scatter(x=t_vm, y=trace_vm, mode='lines', line=dict(color='green', width=1.0), name='VM (CS detection)'), secondary_y=True)

    n_pair = min(len(pl_starts), len(pl_ends))
    for i in range(n_pair):
        s = int(pl_starts[i]); e = int(pl_ends[i])
        if e < s:
            s, e = e, s
        if s < trace_raw.size and e < trace_raw.size:
            fig.add_vrect(x0=t_raw[s], x1=t_raw[e], fillcolor='rgba(190, 60, 255, 0.20)', line_width=0, layer='below')

    if all_spikes.size:
        fig.add_trace(go.Scatter(x=t_raw[all_spikes], y=trace_raw[all_spikes], mode='markers', marker=dict(color='rgba(0,0,255,0.25)', size=4), name='All spikes'), secondary_y=True)
    if simple_spikes.size:
        fig.add_trace(go.Scatter(x=t_raw[simple_spikes], y=trace_raw[simple_spikes], mode='markers', marker=dict(color='rgb(31,119,180)', size=6), name='Simple spikes'), secondary_y=True)
    if complex_spikes.size:
        fig.add_trace(go.Scatter(x=t_raw[complex_spikes], y=trace_raw[complex_spikes], mode='markers', marker=dict(color='black', size=7), name='Complex spikes'), secondary_y=True)

    if pl_locs.size:
        pl_locs = pl_locs[(pl_locs >= 0) & (pl_locs < trace_raw.size)]
        if pl_locs.size:
            fig.add_trace(
                go.Scatter(
                    x=t_raw[pl_locs],
                    y=trace_raw[pl_locs],
                    mode='markers',
                    marker=dict(color='purple', size=9, symbol='diamond-open'),
                    name='Plateau loci',
                ),
                secondary_y=True,
            )

    title = f"{os.path.basename(os.path.dirname(pkl_path))} | {os.path.basename(pkl_path)} | latest_spike_detected_and_classified (with plateaus)"
    fig.update_layout(template='plotly_white', title=title, width=1400, height=520, legend=dict(x=1.02, y=1, xanchor='left', yanchor='top'), margin=dict(l=60, r=240, t=80, b=60))
    fig.update_xaxes(title_text='Time (s)')
    fig.update_yaxes(title_text='Calcium (dF/F)', secondary_y=False)
    fig.update_yaxes(title_text='Voltage / VM (a.u.)', secondary_y=True)

    suffix = _pkl_suffix_from_name(pkl_path)
    save_dir = os.path.dirname(pkl_path)
    out_html = os.path.join(save_dir, f"latest_spike_detected_and_classified{suffix}.html")
    fig.write_html(out_html, include_plotlyjs='cdn')
    return out_html
# ---------- USER CONFIG ----------
final_dict_paths = pathPyr

# Optional auto-search if list above is empty
# auto_search_root = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin"
# if len(final_dict_paths) == 0:
#     final_dict_paths = glob.glob(os.path.join(auto_search_root, "**", "spike_detection_refined_new*.pkl"), recursive=True)

# Plateau params (same meaning as detect_bursts_from_vm)
plateau_amp_threshold = 0.8
plateau_duration_threshold = 100
plateau_kernel_ms = 100
plateau_score_min_ms = 80

# Keep false to guarantee existing spike labels remain untouched
ALLOW_SPIKE_LABEL_CHANGES = False

saved_pkls = []
saved_figs = []

resolved_paths = _resolve_final_pkl_paths(final_dict_paths)
if len(resolved_paths) == 0:
    print('[INFO] No matching spike_detection_refined_new*.pkl files found from final_dict_paths')

for pkl_path in resolved_paths:
    if (not os.path.isfile(pkl_path)) or (not str(pkl_path).lower().endswith('.pkl')):
        print(f"[WARN] skipping non-pkl path: {pkl_path}")
        continue

    with open(pkl_path, 'rb') as f:
        d = pickle.load(f)

    simple_before = _as_int_unique(d.get('vm_simple_spikes', []))
    complex_before = _as_int_unique(d.get('vm_complex_spikes', []))
    all_before = _as_int_unique(d.get('vm_all_spikes', []))

    trace_idx = _extract_vm_for_detection(d)
    fr = float(d.get('detection_params', {}).get('frame_rate', 500.0))
    if (not np.isfinite(fr)) or fr <= 0:
        fr = 500.0

    all_spikes = _as_int_unique(d.get('vm_all_spikes', []))
    all_spikes = all_spikes[(all_spikes >= 0) & (all_spikes < trace_idx.size)]

    spike_heights = np.asarray(d.get('spike_heights', []), dtype=float).ravel()
    if spike_heights.size != trace_idx.size:
        spike_heights = np.ones_like(trace_idx, dtype=float)

    burst_dict = d.get('vm_burst_dict', {}) if isinstance(d.get('vm_burst_dict', {}), dict) else {}

    det = d.get('detection_params', {}) if isinstance(d.get('detection_params', {}), dict) else {}
    cb_amp = float(det.get('cb_amp_threshold', 0.3))
    cb_dur = float(det.get('cb_duration_threshold', 20))

    ret = detect_bursts_from_vm(
        trace_idx,
        spike_heights,
        burst_dict,
        all_spikes,
        fr,
        highpass=1.0,
        median_window=11,
        cb_amp_threshold=cb_amp,
        cb_duration_threshold=cb_dur,
        isi_threshold_ms=25,
        baseline_subtract=True,
        baseline_window_ms=20,
        baseline_percentile=10,
        merge_SS_ms=20,
        merge_CB_ms=5,
        plateau_duration_threshold=plateau_duration_threshold,
        plateau_amp_threshold=plateau_amp_threshold,
        plateau_kernel_ms=plateau_kernel_ms,
        plateau_score_min_ms=plateau_score_min_ms,
        plotflag=False,
    )

    plateaus_new = {}
    if isinstance(ret, tuple):
        if len(ret) >= 8:
            plateaus_new = ret[7]
        elif len(ret) >= 7 and isinstance(ret[6], dict) and 'plateaus_dict' in ret[6]:
            plateaus_new = ret[6].get('plateaus_dict', {})

    d_new = copy.deepcopy(d)
    d_new['vm_plateaus_dict'] = _normalize_plateaus_dict(plateaus_new)
    d_new['detection_params'] = copy.deepcopy(det)
    d_new['detection_params']['plateau_amp_threshold'] = float(plateau_amp_threshold)
    d_new['detection_params']['plateau_duration_threshold'] = float(plateau_duration_threshold)
    d_new['detection_params']['plateau_kernel_ms'] = float(plateau_kernel_ms)
    d_new['detection_params']['plateau_score_min_ms'] = float(plateau_score_min_ms)

    # Safety: keep existing final spike labels unchanged
    simple_after = _as_int_unique(d_new.get('vm_simple_spikes', []))
    complex_after = _as_int_unique(d_new.get('vm_complex_spikes', []))
    all_after = _as_int_unique(d_new.get('vm_all_spikes', []))

    if (not ALLOW_SPIKE_LABEL_CHANGES) and (
        (not np.array_equal(simple_before, simple_after))
        or (not np.array_equal(complex_before, complex_after))
        or (not np.array_equal(all_before, all_after))
    ):
        raise RuntimeError(f"Spike labels changed unexpectedly in {pkl_path}. Aborting.")

    with open(pkl_path, 'wb') as f:
        pickle.dump(d_new, f, protocol=pickle.HIGHEST_PROTOCOL)
    saved_pkls.append(pkl_path)

    try:
        html_path = _recreate_latest_with_plateaus(d_new, pkl_path, cal_sr=30.0)
        saved_figs.append(html_path)
        print(f"[OK] backfilled+figure: {pkl_path} -> {html_path} | n_plateaus={len(d_new['vm_plateaus_dict'].get('starts', []))}")
    except Exception as e:
        print(f"[WARN] backfilled pkl but figure failed for {pkl_path}: {e}")

print(f"\nDone. Input items: {len(final_dict_paths)} | Resolved PKLs: {len(resolved_paths)} | PKLs updated: {len(saved_pkls)} | Figures saved: {len(saved_figs)}")




TypeError: detect_bursts_from_vm() got an unexpected keyword argument 'plateau_duration_threshold'